# ARC-AGI-2 solver v2 (symbolic + LLM harness, offline, GPU)
Hybrid engine: deterministic symbolic propose (dynamics + composer
program enumeration, LOO-certified, 8.9% local baseline) AND a staged
Qwen2.5-3B-Instruct (pre-trained, dataset-mounted, no internet) whose
grid serializations are CERTIFIED before competing: V1 self-consistency
(majority of K samples) + V2 leave-one-out understanding (the model
must reproduce held-out train outputs). Certified symbolic programs
outrank fluent guesses; the LLM only leads when the symbolic side is
unverified and its own LOO-understanding passes. Resumable partial +
LLM deadline guard; schema-validated submission.json, 2 attempts per
test input. Apache-2.0 weights, evaluation offline (rules-legal).


In [ ]:
import glob, json, os, sys, time

# Coverage over depth: at ~82s/puzzle with K=4+LOO2 the full eval set
# overruns the 12h kernel; K=2+LOO1 (~25-40s) covers every task inside
# the 9.5h LLM budget (certify stays: majority of 2 + 1 LOO check).
os.environ.setdefault("LLM_K_SAMPLES", "2")
os.environ.setdefault("LLM_LOO_MAX", "1")

MODULES = {"recognizer.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"\nARC-AGI-2 Recognizer \u2014 RSI orient step: WHAT is this puzzle about?\n\nInstead of trying strategies blindly (fixed order), first CLASSIFY the\npuzzle dynamic from its train pairs, then route to ONE strategy.\n\n  classify(trains) -> (dynamic, evidence)\n  solve_routed(trains, test) -> apply predicted strategy, fallback to chain\n\nDetectors are honest: a dynamic is claimed only if its strategy\nreproduces EVERY train output from its train input (no test leakage).\n\nMetrics (all deterministic, rule-based \u2014 R10):\n  - precision@1: predicted strategy alone solves the test\n  - oracle: ANY known strategy solves (upper bound of current toolbox)\n  - efficiency: strategies tried per puzzle (routed ~1-2 vs blind ~15)\n  - per-dynamic precision table (feeds the learning loop)\n\nNew strategy from this module: try_object_shift (uniform translation\nof all objects). If it solves puzzles nothing else solves \u2192 RSI gain.\n\"\"\"\nimport json\nimport os\nimport sys\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Callable, Dict, List, Optional, Tuple\n\nsys.path.insert(0, str(Path(__file__).parent))\nimport solver_smart_v2 as v2\nimport solver_v3 as v3\nfrom rsi_module import try_adaptive_tiling\n\nGrid = List[List[int]]\nTrains = List[Tuple[Grid, Grid]]\n\n# \u2500\u2500\u2500 Grid helpers \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef shape(g: Grid) -> Tuple[int, int]:\n    return (len(g), len(g[0]) if g else 0)\n\n\ndef bg_color(g: Grid) -> int:\n    c = Counter(c for row in g for c in row)\n    return c.most_common(1)[0][0]\n\n\ndef components(g: Grid, bg: Optional[int] = None):\n    \"\"\"Connected components (4-dir) of non-background cells, per color.\"\"\"\n    if bg is None:\n        bg = bg_color(g)\n    R, C = shape(g)\n    seen = set()\n    comps = []\n    for r in range(R):\n        for c in range(C):\n            if (r, c) in seen or g[r][c] == bg:\n                continue\n            color = g[r][c]\n            cells, stack = [], [(r, c)]\n            while stack:\n                cr, cc = stack.pop()\n                if ((cr, cc) in seen or cr < 0 or cr >= R or cc < 0 or cc >= C\n                        or g[cr][cc] != color):\n                    continue\n                seen.add((cr, cc))\n                cells.append((cr, cc))\n                stack.extend([(cr - 1, cc), (cr + 1, cc),\n                              (cr, cc - 1), (cr, cc + 1)])\n            if cells:\n                rs = [x[0] for x in cells]\n                cs = [x[1] for x in cells]\n                sig = frozenset((rr - min(rs), cc - min(cs))\n                                for rr, cc in cells)\n                comps.append({\"color\": color, \"cells\": cells,\n                              \"size\": len(cells), \"sig\": sig,\n                              \"anchor\": (min(rs), min(cs))})\n    return comps\n\n\ndef contains_subgrid(big: Grid, small: Grid) -> bool:\n    R, C = shape(big)\n    r, c = shape(small)\n    if r > R or c > C or r == 0:\n        return False\n    for i in range(R - r + 1):\n        for j in range(C - c + 1):\n            if all(big[i + di][j + dj] == small[di][dj]\n                   for di in range(r) for dj in range(c)):\n                return True\n    return False\n\n\ndef verify(fn: Callable, trains: Trains) -> bool:\n    \"\"\"Strategy reproduces every train output from its train input.\"\"\"\n    try:\n        for inp, out in trains:\n            pred = fn(trains, inp)\n            if pred is None or not v3.grids_equal(pred, out):\n                return False\n        return True\n    except Exception:\n        return False\n\n\ndef verify_loo(fn: Callable, trains: Trains) -> bool:\n    \"\"\"Leave-one-out: predict each train output from the OTHER pairs.\n\n    Kills memorizers (e.g. nearest_neighbor returns the matching train\n    output verbatim \u2192 plain verify passes trivially, LOO fails).\n    Detectors must prove generalization, not recall.\n    \"\"\"\n    if len(trains) < 2:\n        return verify(fn, trains)\n    try:\n        for i, (inp, out) in enumerate(trains):\n            others = trains[:i] + trains[i + 1:]\n            pred = fn(others, inp)\n            if pred is None or not v3.grids_equal(pred, out):\n                return False\n        return True\n    except Exception:\n        return False\n\n\ndef verify_loo_fraction(fn: Callable, trains: Trains) -> float:\n    \"\"\"Fraction of train pairs reproduced leave-one-out (0.0-1.0).\n\n    Fractional sibling of verify_loo for critic scoring \u2014 the A+B audit\n    (2026-10-16) found agent.propose called this name, got AttributeError,\n    and silently scored every producer's 0.5-weight LOO term as 0.0.\n    Deterministic (R10): same pairs, same fraction, always.\n    \"\"\"\n    if not trains:\n        return 0.0\n    if len(trains) < 2:\n        try:\n            pred = fn(trains, trains[0][0])\n            return (1.0 if pred is not None and\n                    v3.grids_equal(pred, trains[0][1]) else 0.0)\n        except Exception:\n            return 0.0\n    ok = 0\n    for i, (_inp, out) in enumerate(trains):\n        others = trains[:i] + trains[i + 1:]\n        try:\n            pred = fn(others, _inp)\n            if pred is not None and v3.grids_equal(pred, out):\n                ok += 1\n        except Exception:\n            pass\n    return ok / len(trains)\n\n\ndef safe_apply(fn: Callable, trains: Trains, test: Grid):\n    try:\n        return fn(trains, test)\n    except Exception:\n        return None\n\n# \u2500\u2500\u2500 New strategy: uniform object shift \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef detect_shift(trains: Trains) -> Optional[Tuple[int, int]]:\n    \"\"\"All objects translated by one consistent (dr, dc)?\"\"\"\n    shifts = []\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        ic = components(inp)\n        oc = components(out)\n        if not ic or len(ic) != len(out and oc):\n            return None\n        # match by (color, size, shape)\n        omap = defaultdict(list)\n        for o in oc:\n            omap[(o[\"color\"], o[\"size\"], o[\"sig\"])].append(o[\"anchor\"])\n        pair_shifts = set()\n        for o in ic:\n            key = (o[\"color\"], o[\"size\"], o[\"sig\"])\n            if len(omap.get(key, [])) != 1:\n                return None\n            (or_, oc_) = omap[key][0]\n            (ir, ic_) = o[\"anchor\"]\n            pair_shifts.add((or_ - ir, oc_ - ic_))\n        if len(pair_shifts) != 1:\n            return None\n        shifts.append(next(iter(pair_shifts)))\n    if not shifts or len(set(shifts)) != 1:\n        return None\n    return shifts[0]\n\n\ndef try_object_shift(trains: Trains, test: Grid) -> Optional[Grid]:\n    s = detect_shift(trains)\n    if s is None:\n        return None\n    dr, dc = s\n    R, C = shape(test)\n    bg = bg_color(trains[0][0])\n    out = [[bg] * C for _ in range(R)]\n    for comp in components(test, bg):\n        for (r, c) in comp[\"cells\"]:\n            nr, nc = r + dr, c + dc\n            if 0 <= nr < R and 0 <= nc < C:\n                out[nr][nc] = comp[\"color\"]\n            else:\n                return None  # shift leaves the grid \u2192 not this dynamic\n    return out\n\n# \u2500\u2500\u2500 Band match (rearrange family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 18447a8d. Static comps (same cells in/out, learned per subset)\n# form BANDS; movers pair with bands REVERSE by size (bands size-asc /\n# movers size-desc, ties: row-asc / color-desc) \u2014 corroborated by exact\n# complementarity (band8 + moversize = 15 in all 3 trains). Each mover\n# rigid-translates to its band (TOP = band min-row, LEFT = leftmost >= 0\n# non-overlapping vs static). Movers vacate (bg shows through).\n\ndef _static_colors(trains, bg):\n    colors = None\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        # static OBJECTS (identical cell sets in/out), grouped by color\n        imap, omap = defaultdict(set), defaultdict(set)\n        for o in components(inp, bg):\n            imap[o[\"color\"]] |= set(o[\"cells\"])\n        for o in components(out, bg):\n            omap[o[\"color\"]] |= set(o[\"cells\"])\n        stat = {c for c in set(imap) | set(omap)\n                if imap.get(c, set()) == omap.get(c, set()) and imap.get(c)}\n        colors = stat if colors is None else (colors & stat)\n    return colors\n\n\ndef try_band_match(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    bg = bg_color(trains[0][0])\n    stat = _static_colors(trains, bg)\n    if not stat:\n        return None\n    R, C = shape(test)\n    tcomps = components(test, bg)\n    bands = [o for o in tcomps if o[\"color\"] in stat]\n    movers = [o for o in tcomps if o[\"color\"] not in stat]\n    if not movers or len(bands) != len(movers):\n        return None\n    bands.sort(key=lambda o: (o[\"size\"], o[\"anchor\"][0], o[\"anchor\"][1]))\n    movers.sort(key=lambda o: (-o[\"size\"], -o[\"color\"],\n                               o[\"anchor\"][0], o[\"anchor\"][1]))\n    out = [row[:] for row in test]\n    for m in movers:\n        for (r, c) in m[\"cells\"]:\n            out[r][c] = bg\n    static_cells = set()\n    for b in bands:\n        static_cells |= set(b[\"cells\"])\n    for band, mover in zip(bands, movers):\n        top = min(r for (r, c) in band[\"cells\"])\n        (ar, ac) = mover[\"anchor\"]\n        placed = None\n        for left in range(0, C):\n            cells = [(r - ar + top, c - ac + left)\n                     for (r, c) in mover[\"cells\"]]\n            if not all(0 <= r < R and 0 <= c < C for (r, c) in cells):\n                continue\n            if any((r, c) in static_cells for (r, c) in cells):\n                continue\n            placed = cells\n            break\n        if placed is None:\n            return None\n        for (r, c) in placed:\n            out[r][c] = mover[\"color\"]\n            static_cells.add((r, c))\n    return out\n\n\ndef d_bandmatch(trains):\n    return verify_loo(try_band_match, trains)\n\n# \u2500\u2500\u2500 Legend swap (rearrange family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 0becf7df: the top-left 2x2 corner is an immutable LEGEND whose\n# ROWS define color-swap pairs (row0 [a,b] -> a<->b, row1 [c,d] -> c<->d);\n# every other cell swaps accordingly. Learned per input (pairs differ\n# per train!), legend cells never change. Colors outside the pairs and\n# degenerate pairs (a==b) pass through.\n\ndef _legend_pairs(inp):\n    if len(inp) < 2 or len(inp[0]) < 2:\n        return None\n    a, b = inp[0][0], inp[0][1]\n    c, d = inp[1][0], inp[1][1]\n    return {a: b, b: a, c: d, d: c}\n\n\ndef try_legend_swap(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        m = _legend_pairs(inp)\n        if m is None:\n            return None\n        R, C = shape(inp)\n        for r in range(R):\n            for c in range(C):\n                if r < 2 and c < 2:\n                    if out[r][c] != inp[r][c]:\n                        return None  # legend must be immutable\n                elif out[r][c] != m.get(inp[r][c], inp[r][c]):\n                    return None\n    m = _legend_pairs(test)\n    if m is None:\n        return None\n    R, C = shape(test)\n    return [[test[r][c] if (r < 2 and c < 2) else m.get(test[r][c],\n            test[r][c]) for c in range(C)] for r in range(R)]\n\n\ndef d_legendswap(trains):\n    return verify_loo(try_legend_swap, trains)\n\n# \u2500\u2500\u2500 Lattice interpolation (rearrange family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 06df4c85/train0: same-shape same-color blocks on lattice lines\n# get intermediate lattice points filled with shifted copies (step 3 =\n# gcd of same-line gaps; 0-blocks replaced). Multi-texture puzzles (whose\n# trains need different rules, like 06df4c85 itself) HONESTLY fail LOO.\n# Placeholders: fills may overwrite bg or the single most-common block\n# class (learned per subset); anything else -> None (fail safe).\n\ndef _grid_step(inp, bg):\n    \"\"\"Per-grid lattice step: min spacing of full-bg lines, else gcd.\"\"\"\n    R, C = shape(inp)\n    rl = [r for r in range(R) if all(v == bg for v in inp[r])]\n    cl = [c for c in range(C) if all(inp[r][c] == bg for r in range(R))]\n    sp = []\n    if len(rl) >= 2:\n        sp.append(min(b - a for a, b in zip(rl, rl[1:])))\n    if len(cl) >= 2:\n        sp.append(min(b - a for a, b in zip(cl, cl[1:])))\n    if sp:\n        return min(sp) if min(sp) >= 2 else None\n    gaps = []\n    comps = components(inp, bg)\n    byrc = defaultdict(list)\n    for o in comps:\n        byrc[(o[\"color\"], \"r\", o[\"anchor\"][0])].append(o[\"anchor\"][1])\n        byrc[(o[\"color\"], \"c\", o[\"anchor\"][1])].append(o[\"anchor\"][0])\n    for _k, v in byrc.items():\n        v = sorted(v)\n        for a, b in zip(v, v[1:]):\n            if b > a:\n                gaps.append(b - a)\n    if not gaps:\n        return None\n    from math import gcd\n    from functools import reduce\n    s = reduce(gcd, gaps)\n    return s if s >= 2 else None\n\n\ndef _placeholder(inp, bg):\n    cnt = Counter()\n    for o in components(inp, bg):\n        cnt[(o[\"color\"], o[\"sig\"])] += 1\n    return cnt.most_common(1)[0][0] if cnt else None\n\n\ndef _lattice_apply(inp, s, ph_color, bg):\n    R, C = shape(inp)\n    out = [row[:] for row in inp]\n    comps = components(inp, bg)\n    groups = defaultdict(list)\n    for o in comps:\n        groups[(o[\"color\"], o[\"sig\"])].append(o)\n    for (_color, _sig), objs in groups.items():\n        if len(objs) < 2:\n            continue\n        byrow, bycol = defaultdict(list), defaultdict(list)\n        for o in objs:\n            byrow[o[\"anchor\"][0]].append(o)\n            bycol[o[\"anchor\"][1]].append(o)\n        for _k, run in list(byrow.items()) + list(bycol.items()):\n            run = sorted(run, key=lambda o: o[\"anchor\"])\n            for a, b in zip(run, run[1:]):\n                ar, ac = a[\"anchor\"]\n                br, bc = b[\"anchor\"]\n                span = br - ar + bc - ac\n                if span % s != 0:\n                    return None\n                steps = span // s\n                for t in range(1, steps):\n                    f = t / steps\n                    or_ = round(ar + (br - ar) * f)\n                    oc = round(ac + (bc - ac) * f)\n                    cells = [(cr - ar + or_, cc - ac + oc)\n                             for (cr, cc) in a[\"cells\"]]\n                    if not all(0 <= nr < R and 0 <= nc < C\n                               for (nr, nc) in cells):\n                        return None\n                    # occupied by pattern endpoints (e.g. a colored block\n                    # where the 0-lattice wants to paint): skip the copy,\n                    # never half-paint. Final verify catches real misses.\n                    if all(out[nr][nc] in (bg, ph_color)\n                           for (nr, nc) in cells):\n                        for (nr, nc) in cells:\n                            out[nr][nc] = a[\"color\"]\n    return out\n\n\ndef try_lattice_interp(trains: Trains, test: Grid) -> Optional[Grid]:\n    # Per-grid analysis (bg, step, placeholder all native to each grid):\n    # 06df4c85 trains have DIFFERENT bgs (1/8/4) \u2014 a shared-bg rule can\n    # never cover them. The RULE (interpolate on bg lattice) is uniform;\n    # parameters are per-grid. Vacuity guard: the rule must visibly do\n    # something in at least one train (else identity puzzles false-fire).\n    if not trains:\n        return None\n    worked = 0\n    for inp, out in trains:\n        bg = bg_color(inp)\n        s = _grid_step(inp, bg)\n        ph = _placeholder(inp, bg)\n        if s is None or ph is None:\n            return None\n        pred = _lattice_apply(inp, s, ph[0], bg)\n        if pred is None or not v3.grids_equal(pred, out):\n            return None\n        worked += sum(1 for r in range(len(inp))\n                      for c in range(len(inp[0]))\n                      if pred[r][c] != inp[r][c])\n    if worked == 0:\n        return None\n    bg = bg_color(test)\n    s = _grid_step(test, bg)\n    ph = _placeholder(test, bg)\n    if s is None or ph is None:\n        return None\n    return _lattice_apply(test, s, ph[0], bg)\n\n\ndef d_lattice(trains):\n    return verify_loo(try_lattice_interp, trains)\n\n# \u2500\u2500\u2500 Connect dots (draw family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 992798f6: an `x` king-path joins `o` to `#`. Rule, fit on all\n# 4 trains: the ELBOW endpoint emits [1 diagonal toward target +\n# straight run of (max-min) along the LONG axis]; the DIAG endpoint emits\n# pure diagonal king-steps until adjacent to drawn content. Endpoint\n# roles, dot colors and ray color are LEARNED per puzzle (both role\n# orders tried; first reproducing all trains wins). bg-only painting;\n# rays stop before non-bg (content preserved). No overwrite evidence.\n\ndef _king_step(fr, fc, tr, tc):\n    dr = (tr > fr) - (tr < fr)\n    dc = (tc > fc) - (tc < fc)\n    return (fr + dr, fc + dc)\n\n\ndef _singles(g, bg):\n    out = {}\n    for r, row in enumerate(g):\n        for c, v in enumerate(row):\n            if v != bg:\n                out.setdefault(v, []).append((r, c))\n    return {k: v[0] for k, v in out.items() if len(v) == 1}\n\n\ndef _connect_build(inp, pe, pd, X, bg):\n    R, C = shape(inp)\n    out = [row[:] for row in inp]\n    drawn = {pe, pd}\n\n    def paintable(r, c):\n        return 0 <= r < R and 0 <= c < C and out[r][c] == bg\n\n    # elbow ray: 1 diagonal + straight (max-min) along long axis\n    dr = pd[0] - pe[0]\n    dc = pd[1] - pe[1]\n    if abs(dr) + abs(dc) <= 1:\n        return None\n    r, c = _king_step(pe[0], pe[1], pd[0], pd[1])\n    if (r, c) == pd or not paintable(r, c):\n        return None\n    out[r][c] = X\n    drawn.add((r, c))\n    if abs(dr) >= abs(dc):\n        step, n = (1 if dr > 0 else -1, 0), abs(dr) - abs(dc)\n    else:\n        step, n = (0, 1 if dc > 0 else -1), abs(dc) - abs(dr)\n    for _ in range(n):\n        r, c = r + step[0], c + step[1]\n        if not (0 <= r < R and 0 <= c < C):\n            return None\n        if out[r][c] != bg:\n            break  # truncate before content; trains never hit this\n        out[r][c] = X\n        drawn.add((r, c))\n    # diag ray: king-steps until adjacent to drawn content\n    r, c = pd\n    for _ in range(64):\n        nr, nc = _king_step(r, c, pe[0], pe[1])\n        if not (0 <= nr < R and 0 <= nc < C):\n            return None\n        if out[nr][nc] != bg:\n            break\n        out[nr][nc] = X\n        drawn.add((nr, nc))\n        r, c = nr, nc\n    # must end adjacent to the elbow construction (else unlinked)\n    if not any(abs(r - dr_) + abs(c - dc_) == 1 or\n               (abs(r - dr_) == 1 and abs(c - dc_) == 1)\n               for (dr_, dc_) in drawn if (dr_, dc_) != (r, c)):\n        pass  # linked via paint trail by construction; no extra check\n    return out\n\n\ndef try_connect_dots(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    bg = bg_color(trains[0][0])\n    # learn: exactly-2 single-cell colors consistent + single ray color\n    cols = None\n    for inp, _out in trains:\n        s = set(_singles(inp, bg).keys())\n        cols = s if cols is None else (cols & s)\n        if len(cols or ()) > 2:\n            return None\n    if not cols or len(cols) != 2:\n        return None\n    c1, c2 = sorted(cols)\n    # strictness: outputs differ only by bg->X additions, single X\n    added = set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        for r in range(len(inp)):\n            for c in range(len(inp[0])):\n                a, b = inp[r][c], out[r][c]\n                if a == bg and b != bg:\n                    added.add(b)\n                elif a != b:\n                    return None\n    if len(added) != 1:\n        return None\n    X = next(iter(added))\n    if X in cols:\n        return None\n    for (e_col, d_col) in ((c1, c2), (c2, c1)):\n        ok = True\n        for inp, out in trains:\n            se = _singles(inp, bg)\n            if e_col not in se or d_col not in se:\n                ok = False\n                break\n            pred = _connect_build(inp, se[e_col], se[d_col], X, bg)\n            if pred is None or not v3.grids_equal(pred, out):\n                ok = False\n                break\n        if ok:\n            se = _singles(test, bg)\n            if e_col not in se or d_col not in se:\n                return None\n            return _connect_build(test, se[e_col], se[d_col], X, bg)\n    return None\n\n\ndef d_connect(trains):\n    return verify_loo(try_connect_dots, trains)\n\n# \u2500\u2500\u2500 Grow ring (draw family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 0962bcdd: a clean DIAMOND (center C, 4 orthogonal arms color A\n# length exactly 1, diagonal corners bg) grows one ring: arms extend to\n# length 2 (same arm color per direction), all diagonal cells within\n# radius 2 take the CENTER color, center kept. Star centered at input\n# center, shifted minimally to fit grid (up-first order); (10,8)->(9,8)\n# in train0 is grid-fit, not a rule change. Multi-motif grids handled\n# per motif; strict (any failure -> None). Conservative bg-only paint.\n\ndef _find_diamonds(g, bg):\n    R, C = shape(g)\n    found = []\n    for r in range(R):\n        for c in range(C):\n            if g[r][c] == bg:\n                continue\n            nbrs = [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]\n            if not all(0 <= nr < R and 0 <= nc < C and\n                       g[nr][nc] != bg for (nr, nc) in nbrs):\n                continue\n            acol = {g[nr][nc] for (nr, nc) in nbrs}\n            if len(acol) != 1:\n                continue\n            diags = [(r - 1, c - 1), (r - 1, c + 1),\n                     (r + 1, c - 1), (r + 1, c + 1)]\n            if not all(0 <= nr < R and 0 <= nc < C and\n                       g[nr][nc] == bg for (nr, nc) in diags):\n                continue\n            # arms exactly length 1 (cells at distance 2 along axes bg/out)\n            ok = True\n            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                nr, nc = r + 2 * dr, c + 2 * dc\n                if 0 <= nr < R and 0 <= nc < C and g[nr][nc] != bg:\n                    ok = False\n                    break\n            if not ok:\n                continue\n            found.append({\"center\": (r, c), \"ccolor\": g[r][c],\n                          \"arm\": next(iter(acol))})\n    return found\n\n\ndef _grow_star(g, bg, motif):\n    R, C = shape(g)\n    (cr, cc) = motif[\"center\"]\n    arm, cen = motif[\"arm\"], motif[\"ccolor\"]\n    for dor, doc in ((0, 0), (-1, 0), (1, 0), (0, -1), (0, 1),\n                     (-1, -1), (-1, 1), (1, -1), (1, 1)):\n        nr, nc = cr + dor, cc + doc\n        if not (0 <= nr < R and 0 <= nc < C):\n            return None\n    out = [row[:] for row in g]\n    # arms: extend each axis using that direction's arm color\n    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n        ar, ac = cr + dr, cc + dc  # existing arm cell (len 1)\n        # walk outward while same arm color (robust to len>1 inputs)\n        cells = [(ar, ac)]\n        nr, nc = ar + dr, ac + dc\n        while 0 <= nr < R and 0 <= nc < C and out[nr][nc] == arm:\n            cells.append((nr, nc))\n            nr, nc = nr + dr, nc + dc\n        # one new cell beyond, if bg\n        if 0 <= nr < R and 0 <= nc < C:\n            if out[nr][nc] != bg:\n                return None\n            out[nr][nc] = arm\n        # corners along the way take center color\n    for dr in range(-2, 3):\n        for dc in range(-2, 3):\n            if abs(dr) != abs(dc) or (dr == 0 and dc == 0):\n                continue\n            nr, nc = cr + dr, cc + dc\n            if out[nr][nc] == bg:\n                out[nr][nc] = cen\n            elif out[nr][nc] != cen:\n                return None\n    return out\n\n\ndef try_grow_ring(trains: Trains, test: Grid) -> Optional[Grid]:\n    # In-place only: observed stars never shift (an earlier (10,8)->(9,8)\n    # \"shift\" was a misread; the diamond was already centered (9,8)).\n    # Out-of-bounds growth -> None (honest; revisit with evidence if seen).\n    if not trains:\n        return None\n    for inp, out in trains:\n        bg = bg_color(inp)\n        motifs = _find_diamonds(inp, bg)\n        if not motifs:\n            return None\n        pred = [row[:] for row in inp]\n        for m in motifs:\n            pred = _grow_star(pred, bg, m)\n            if pred is None:\n                return None\n        if not v3.grids_equal(pred, out):\n            return None\n    bg = bg_color(test)\n    motifs = _find_diamonds(test, bg)\n    if not motifs:\n        return None\n    pred = [row[:] for row in test]\n    for m in motifs:\n        pred = _grow_star(pred, bg, m)\n        if pred is None:\n            return None\n    return pred\n\n\ndef d_growring(trains):\n    return verify_loo(try_grow_ring, trains)\n\n# \u2500\u2500\u2500 Symmetry complete (draw family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 11852cab (C4-closure reproduces all 3 trains): paint the orbit\n# of every non-bg cell under a symmetry group about the content-bbox\n# center (additive only: bg cells fill, content never erased/recolored).\n# Groups tried in order (specific first): C4 rotations, H flip, V flip,\n# D4 (fixpoint of rot90+flipH). First group reproducing all trains wins.\n\ndef _sym_orbit_fill(g, bg, center, ops):\n    R, C = shape(g)\n    CR, CC = center\n    # doubled integer arithmetic (supports .5 centers)\n    ICR, ICC = int(round(2 * CR)), int(round(2 * CC))\n    out = [row[:] for row in g]\n\n    def img(op, r, c):\n        R2, C2 = 2 * r, 2 * c\n        if op == \"rot90\":\n            R2, C2 = ICR - (C2 - ICC), ICC + (R2 - ICR)\n        elif op == \"flipH\":\n            C2 = 2 * ICC - C2\n        elif op == \"flipV\":\n            R2 = 2 * ICR - R2\n        else:\n            return None\n        if R2 % 2 or C2 % 2:\n            return None\n        return (R2 // 2, C2 // 2)\n\n    for _ in range(8):  # fixpoint: closure under group\n        changed = False\n        for r in range(R):\n            for c in range(C):\n                if out[r][c] == bg:\n                    continue\n                for op in ops:\n                    p = img(op, r, c)\n                    if p is None:\n                        continue\n                    or_, oc = p\n                    if 0 <= or_ < R and 0 <= oc < C and out[or_][oc] == bg:\n                        out[or_][oc] = out[r][c]\n                        changed = True\n        if not changed:\n            break\n    return out\n\n\nSYM_GROUPS = [\n    (\"C4\", [\"rot90\"]),\n    (\"H\", [\"flipH\"]),\n    (\"V\", [\"flipV\"]),\n    (\"D4\", [\"rot90\", \"flipH\"]),\n]\n\n\ndef try_symmetry_complete(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    for group, ops in SYM_GROUPS:\n        ok = True\n        for inp, out in trains:\n            bg = bg_color(inp)\n            cells = [(r, c) for r, row in enumerate(inp)\n                     for c, v in enumerate(row) if v != bg]\n            if not cells:\n                ok = False\n                break\n            cr = (min(r for r, c in cells) + max(r for r, c in cells)) / 2\n            cc = (min(c for r, c in cells) + max(c for r, c in cells)) / 2\n            pred = _sym_orbit_fill(inp, bg, (cr, cc), ops)\n            if not v3.grids_equal(pred, out):\n                ok = False\n                break\n        if ok:\n            bg = bg_color(test)\n            cells = [(r, c) for r, row in enumerate(test)\n                     for c, v in enumerate(row) if v != bg]\n            if not cells:\n                return None\n            cr = (min(r for r, c in cells) + max(r for r, c in cells)) / 2\n            cc = (min(c for r, c in cells) + max(c for r, c in cells)) / 2\n            return _sym_orbit_fill(test, bg, (cr, cc), ops)\n    return None\n\n\ndef d_symcomplete(trains):\n    return verify_loo(try_symmetry_complete, trains)\n\n# \u2500\u2500\u2500 Invert + recolor (EV-hunt find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 6ea4a07e (EV top 0.29 via pattern_fill): binary occupancy\n# INVERTED (6/6 trains: out-cells = 9 - in-cells) plus a learned\n# fg->newfg map (5->4, 8->2, 3->1, consistent across trains). Strict:\n# single fg color per train input; bg stays bg; map must agree.\n\ndef try_invert_recolor(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    mapping = {}\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        bg = bg_color(inp)\n        # NOTE: never bg_color(out): frequency flips when fg dominates\n        # (6ea4a07e/train0 out is 5 O vs 4 dots). Input bg rules.\n        fg = {v for row in inp for v in row if v != bg}\n        if len(fg) != 1:\n            return None\n        old = next(iter(fg))\n        newf = {out[r][c] for r in range(len(out))\n                for c in range(len(out[0]))\n                if inp[r][c] == bg}\n        if len(newf) != 1:\n            return None\n        new = next(iter(newf))\n        if new == bg:\n            return None\n        if old in mapping and mapping[old] != new:\n            return None\n        mapping[old] = new\n        for r in range(len(inp)):\n            for c in range(len(inp[0])):\n                want = new if inp[r][c] == bg else bg\n                if out[r][c] != want:\n                    return None\n    bg = bg_color(test)\n    fg = {v for row in test for v in row if v != bg}\n    if len(fg) != 1 or next(iter(fg)) not in mapping:\n        return None\n    new = mapping[next(iter(fg))]\n    return [[new if v == bg else bg for v in row] for row in test]\n\n\ndef d_invertrecolor(trains):\n    # Same coverage-vs-redundancy lesson as sizefill: strict LOO demands\n    # each pair's key appear in OTHER pairs; 6ea4a07e/train3 is the only\n    # carrier of key 3 (LOO-3 unlearnable, yet the rule is true). Use\n    # full-train consistency + evidence bar (distinct keys >= 2, or a key\n    # repeated). Memorizers still excluded: output is fully determined by\n    # the small map + inversion structure, not by copying a train.\n    if not verify(try_invert_recolor, trains):\n        return False\n    keys = []\n    for inp, _out in trains:\n        bg = bg_color(inp)\n        fg = {v for row in inp for v in row if v != bg}\n        if len(fg) != 1:\n            return False\n        keys.append(next(iter(fg)))\n    return len(set(keys)) >= 2 or len(keys) > len(set(keys))\n\n# \u2500\u2500\u2500 Mirror double (EV-hunt find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 3af2c5a8 (EV 0.26 via adaptive_tiling: right neighborhood!).\n# Output = symmetric mirror-pad to exactly 2x (NO edge repeat):\n# rows [0..R-1, R-1..0], cols [0..C-1, C-1..0], i.e. four quadrants\n# [orig | fliplr ; flipud | both]. Found by eliminating tiling modes\n# (col-flip fails exactly the cells mirror-pad explains).\n\ndef _mirror_double(g):\n    R = len(g)\n    C = len(g[0]) if R else 0\n    if R == 0 or C == 0:\n        return None\n    top = [row[:] + row[::-1] for row in g]\n    return top + top[::-1]\n\n\ndef try_mirror_double(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    for inp, out in trains:\n        ir, ic = shape(inp)\n        or_, oc = shape(out)\n        if (or_, oc) != (2 * ir, 2 * ic):\n            return None\n        pred = _mirror_double(inp)\n        if pred is None or not v3.grids_equal(pred, out):\n            return None\n    ir, ic = shape(test)\n    return _mirror_double(test)\n\n\ndef d_mirrordouble(trains):\n    return verify_loo(try_mirror_double, trains)\n\n# \u2500\u2500\u2500 Block census (cola-20 find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 6773b310 (4 trains, all hand-verified cell-exact): partition by\n# uniform dividers (reuse _oddblock_partition); blocks contain only one\n# non-bg color C; output one cell per block: mark-color M iff C-count==2.\n# Threshold 2 is literal (no 0/3+ block observed; documented scope).\n# Strict LOO detector.\n\ndef _blockcensus_params(trains):\n    Cset, Mset = None, None\n    for inp, out in trains:\n        blocks = _oddblock_partition(inp)\n        if blocks is None:\n            return None\n        bg = bg_color(inp)\n        or_, oc = shape(out)\n        # recover block grid dims\n        R, C = shape(inp)\n        hdiv = [r for r in range(R) if len(set(inp[r])) == 1]\n        vdiv = [c for c in range(C) if len({inp[r][c] for r in range(R)}) == 1]\n        if (len(hdiv) + 1, len(vdiv) + 1) != (or_, oc):\n            return None\n        ncols = len(vdiv) + 1\n        # block colors: single non-bg color across all blocks?\n        palette = {v for b in blocks for row in b for v in row} - {bg}\n        if len(palette) != 1:\n            return None\n        ccol = next(iter(palette))\n        Cset = {ccol} if Cset is None else (Cset & {ccol})\n        # output: monochrome mark color + bg?\n        obg = bg_color(out)\n        opal = {v for row in out for v in row} - {obg}\n        if len(opal) != 1:\n            return None\n        mcol = next(iter(opal))\n        if mcol == bg:\n            return None\n        Mset = {mcol} if Mset is None else (Mset & {mcol})\n        # reproduce: count==2 -> mark\n        for i, b in enumerate(blocks):\n            k = sum(1 for row in b for v in row if v == ccol)\n            want = mcol if k == 2 else obg\n            if out[i // ncols][i % ncols] != want:\n                return None\n    if not Cset or len(Cset) != 1 or not Mset or len(Mset) != 1:\n        return None\n    return (next(iter(Cset)), next(iter(Mset)))\n\n\ndef try_blockcensus(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    learned = _blockcensus_params(trains)\n    if learned is None:\n        return None\n    ccol, mcol = learned\n    bg = bg_color(test)\n    blocks = _oddblock_partition(test)\n    if blocks is None:\n        return None\n    R, C = shape(test)\n    hdiv = [r for r in range(R) if len(set(test[r])) == 1]\n    vdiv = [c for c in range(C) if len({test[r][c] for r in range(R)}) == 1]\n    nr, nc = len(hdiv) + 1, len(vdiv) + 1\n    if len(blocks) != nr * nc:\n        return None\n    out = []\n    for i in range(nr):\n        row = []\n        for j in range(nc):\n            b = blocks[i * nc + j]\n            # scope: blocks hold only bg + learned color\n            if {v for r in b for v in r} - {bg, ccol}:\n                return None\n            k = sum(1 for r in b for v in r if v == ccol)\n            row.append(mcol if k == 2 else bg)\n        out.append(row)\n    return out\n\n\ndef d_blockcensus(trains):\n    return verify_loo(try_blockcensus, trains)\n\n# \u2500\u2500\u2500 Halves family (overlay boolops) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# norhalves (fafffa47: even-row halves; 6430c8c4: odd rows split\n# around a middle uniform divider, divider excluded; 66f2d22f +\n# e345f17b: vertical even halves as last resort), xorhalves\n# (99b1bc43: EXACTLY ONE half figure), orhalves (ce4f8723: AT\n# LEAST ONE half figure). Shared machinery: split kinds tried in\n# precedence order (hdiv, heven, veven for NOR; H-only for XOR/OR \u2014\n# vertical XOR/OR unobserved, scope held); ground = palette\n# INTERSECTION, never majority; mark M learned across trains\n# (LOO enforces stability). Strict LOO detectors.\n\ndef _halves_cand(inp, kind):\n    R = len(inp)\n    C = len(inp[0]) if inp else 0\n    if kind == \"hdiv\":\n        if R % 2 == 0:\n            return None\n        dh = R // 2\n        # the middle row must be a real divider (uniform NON-bg: an\n        # all-bg middle row is empty space, not structure)\n        if len(set(inp[dh])) != 1 or inp[dh][0] == bg_color(inp):\n            return None\n        return inp[:dh], inp[dh + 1:]\n    if kind == \"heven\":\n        if R % 2 or R < 2:\n            return None\n        h = R // 2\n        return inp[:h], inp[h:]\n    if kind == \"veven\":\n        if C % 2 or C < 2:\n            return None\n        w = C // 2\n        return [row[:w] for row in inp], [row[w:] for row in inp]\n    if kind == \"sdiv\":\n        strips = _vstrips_split(inp)\n        if strips is None or len(strips) != 2:\n            return None\n        return strips[0], strips[1]\n    return None\n\n\ndef _halves_ground(a, b):\n    g = ({v for row in a for v in row} &\n         {v for row in b for v in row})\n    return next(iter(g)) if len(g) == 1 else None\n\n\ndef _halves_fit(trains, kinds, mask):\n    for kind in kinds:\n        marks = set()\n        ok = True\n        for inp, out in trains:\n            cand = _halves_cand(inp, kind)\n            if cand is None:\n                ok = False\n                break\n            a, b = cand\n            g = _halves_ground(a, b)\n            if g is None:\n                ok = False\n                break\n            h, w = len(a), len(a[0])\n            if (h, w) != (len(out), len(out[0])):\n                ok = False\n                break\n            fig = {v for row in out for v in row} - {g}\n            if len(fig) != 1:\n                ok = False\n                break\n            m = next(iter(fig))\n            good = True\n            for r in range(h):\n                for c in range(w):\n                    want = m if mask(a[r][c] != g, b[r][c] != g) else g\n                    if out[r][c] != want:\n                        good = False\n                        break\n                if not good:\n                    break\n            if not good:\n                ok = False\n                break\n            marks.add(m)\n        if ok and len(marks) == 1:\n            return (kind, next(iter(marks)))\n    return None\n\n\ndef _halves_try(trains, test, kinds, mask):\n    if not trains:\n        return None\n    fit = _halves_fit(trains, kinds, mask)\n    if fit is None:\n        return None\n    kind, m = fit\n    cand = _halves_cand(test, kind)\n    if cand is None:\n        return None\n    a, b = cand\n    g = _halves_ground(a, b)\n    if g is None:\n        return None\n    h, w = len(a), len(a[0])\n    return [[m if mask(a[r][c] != g, b[r][c] != g) else g\n             for c in range(w)] for r in range(h)]\n\n\ndef _mask_nor(fa, fb):\n    return not fa and not fb\n\n\ndef _mask_xor(fa, fb):\n    return fa != fb\n\n\ndef _mask_or(fa, fb):\n    return fa or fb\n\n\n_NOR_KINDS = (\"hdiv\", \"heven\", \"veven\", \"sdiv\")\n_H_KINDS = (\"hdiv\", \"heven\")\n\n\ndef try_norhalves(trains: Trains, test: Grid) -> Optional[Grid]:\n    return _halves_try(trains, test, _NOR_KINDS, _mask_nor)\n\n\ndef d_norhalves(trains):\n    return verify_loo(try_norhalves, trains)\n\n# \u2500\u2500\u2500 Xor halves (overlay-XOR) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 99b1bc43: same halves machinery (H-only kinds), mask marks\n# cells where EXACTLY ONE half is figure (hand-verified: both-figure\n# and both-bg stay 0 \u2014 a cell-wise (top,bot) function is refuted:\n# (1,2) and (1,3) share the pair yet differ). Mark color M learned\n# across trains; LOO enforces its stability. Strict detector.\n\ndef try_xorhalves(trains: Trains, test: Grid) -> Optional[Grid]:\n    return _halves_try(trains, test, _H_KINDS, _mask_xor)\n\n\ndef d_xorhalves(trains):\n    return verify_loo(try_xorhalves, trains)\n\n# \u2500\u2500\u2500 Or halves (overlay-OR) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in ce4f8723: same halves machinery (H-only kinds), mask marks\n# cells where AT LEAST ONE half is figure (union). Completes the\n# boolean family: NOR, XOR, OR. Mark M learned across trains; LOO\n# enforces stability. Strict detector.\n\ndef try_orhalves(trains: Trains, test: Grid) -> Optional[Grid]:\n    return _halves_try(trains, test, _H_KINDS, _mask_or)\n\n\ndef d_orhalves(trains):\n    return verify_loo(try_orhalves, trains)\n\n# \u2500\u2500\u2500 Pack (fit pieces into 3x3) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 137eaa0f: multi-cell straight bars go to fixed slots\n# (horizontals: first-in-reading \u2192 top row, last \u2192 bottom row;\n# verticals: side column opposite the bottom row); the modal single\n# color dots the center, every other modal single is dropped, all\n# remaining singles fill free cells slot by slot in canvas reading\n# order \u2014 each slot takes its nearest input single (slot home =\n# cell center scaled to input dims; ties: leftmost column, then\n# reading), skipping singles that would 4-merge \u2014 rest 0.\n# Alignment notes (scope, all evidenced): top row mirrors the input\n# side; bottom row is left iff a top bar exists; vertical rows are top\n# iff the top bar is left-aligned. Only the modal color is learned\n# across trains (LOO enforces its stability) \u2014 everything else is a\n# deterministic function of the input. Strict LOO detector.\n\ndef _pack_parts(inp):\n    comps = components(inp)\n    H = [c for c in comps\n         if c[\"size\"] > 1 and len({r for r, _ in c[\"cells\"]}) == 1]\n    V = [c for c in comps\n         if c[\"size\"] > 1 and len({cc for _, cc in c[\"cells\"]}) == 1]\n    S = [c for c in comps if c[\"size\"] == 1]\n    if any(c not in H and c not in V and c not in S for c in comps):\n        return None\n    if not H:\n        return None\n    return H, V, S\n\n\ndef _pack_modal(ts):\n    modals = set()\n    for inp, _out in ts:\n        parts = _pack_parts(inp)\n        if parts is None:\n            return None\n        _H, _V, S = parts\n        cnt = Counter(c[\"color\"] for c in S)\n        if not cnt or list(cnt.values()).count(max(cnt.values())) != 1:\n            return None\n        modals.add(cnt.most_common(1)[0][0])\n    if len(modals) != 1:\n        return None\n    return next(iter(modals))\n\n\ndef _pack_place(inp, modal):\n    parts = _pack_parts(inp)\n    if parts is None:\n        return None\n    H, V, S = parts\n    if len(H) > 2 or len(V) > 1:\n        return None\n    if any(c[\"color\"] == modal for c in H + V):\n        return None\n    W = len(inp[0])\n    mid = (W - 1) / 2.0\n    top = H[0] if len(H) == 2 else None\n    bot = H[-1]\n    occ = {}\n\n    def _put(cells, color):\n        for cell in cells:\n            r, c = cell\n            if not (0 <= r < 3 and 0 <= c < 3) or cell in occ:\n                return False\n            occ[cell] = color\n        return True\n\n    if top is not None:\n        w = len(top[\"cells\"])\n        align = \"L\" if sum(cc for _, cc in top[\"cells\"]) / w > mid else \"R\"\n        cells = ([(0, c) for c in range(3)] if w == 3 else\n                 [(0, c) for c in ([0, 1] if align == \"L\" else [1, 2])])\n        if not _put(cells, top[\"color\"]):\n            return None\n        top_align = align if w == 2 else \"full\"\n    else:\n        top_align = None\n    w = len(bot[\"cells\"])\n    if w == 3:\n        verildi = [(2, c) for c in range(3)]\n    else:\n        b_align = \"L\" if top is not None else \"R\"\n        verildi = [(2, c) for c in ([0, 1] if b_align == \"L\" else [1, 2])]\n    if not _put(verildi, bot[\"color\"]):\n        return None\n    bot_align = \"full\" if w == 3 else b_align\n    if V:\n        if w == 3:\n            return None  # unobserved: vertical + full-width bottom\n        v = V[0]\n        col = 2 if bot_align == \"L\" else 0\n        if top_align == \"L\":\n            rows = [0, 1]\n        else:\n            rows = [1, 2]\n        if not _put([(r, col) for r in rows], v[\"color\"]):\n            # retry the other row band (deterministic fallback)\n            occ = {k: vv for k, vv in occ.items()\n                   if vv != v[\"color\"]}\n            rows = [1, 2] if rows == [0, 1] else [0, 1]\n            if not _put([(r, col) for r in rows], v[\"color\"]):\n                return None\n    if (1, 1) in occ:\n        return None\n    occ[(1, 1)] = modal\n    Hh, Ww = len(inp), len(inp[0])\n    rest = [c for c in S if c[\"color\"] != modal]\n    for r in range(3):\n        for c in range(3):\n            if (r, c) in occ:\n                continue\n            hr, hc = (r + 0.5) * Hh / 3.0, (c + 0.5) * Ww / 3.0\n            cands = sorted(\n                rest,\n                key=lambda s: ((s[\"anchor\"][0] - hr) ** 2 +\n                               (s[\"anchor\"][1] - hc) ** 2,\n                               s[\"anchor\"][1], s[\"anchor\"][0]))\n            for s in cands:\n                if any(occ.get((r + dr, c + dc)) == s[\"color\"]\n                       for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1))):\n                    continue\n                occ[(r, c)] = s[\"color\"]\n                rest.remove(s)\n                break\n    if rest:\n        return None  # every non-modal single must land somewhere\n    return [[occ.get((r, c), 0) for c in range(3)] for r in range(3)]\n\n\ndef _pack_params(trains):\n    modal = _pack_modal(trains)\n    if modal is None:\n        return None\n    for inp, out in trains:\n        if _pack_place(inp, modal) != out:\n            return None\n    return modal\n\n\ndef try_pack(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    modal = _pack_modal(trains)\n    if modal is None:\n        return None\n    return _pack_place(test, modal)\n\n\ndef d_pack(trains):\n    return verify_loo(try_pack, trains)\n\n# \u2500\u2500\u2500 Anomaly crop (odd-count block) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in a87f7484: input is a strip of 3x3 monochrome blocks (one row\n# or one column of blocks); output is a verbatim copy of the block\n# whose non-bg cell count is UNIQUE across the strip (exactly one\n# singleton count, else None). Shape and symmetry are decoys (both\n# refuted: t3/TEST have fully-symmetric decoys); color varies per\n# train by design. Nothing is learned across trains \u2014 the rule is\n# structural; LOO still guards it per puzzle. Strict LOO detector.\n\ndef _anomaly_blocks(inp):\n    R, C = shape(inp)\n    if R == 3 and C % 3 == 0 and C >= 6:\n        return [[row[c:c + 3] for row in inp] for c in range(0, C, 3)]\n    if C == 3 and R % 3 == 0 and R >= 6:\n        return [[row[:] for row in inp[r:r + 3]] for r in range(0, R, 3)]\n    return None\n\n\ndef _anomaly_pick(blocks, bg):\n    counts = [sum(1 for row in b for v in row if v != bg)\n              for b in blocks]\n    freq = Counter(counts)\n    singles = [c for c, n in freq.items() if n == 1]\n    if len(singles) != 1:\n        return None\n    return blocks[counts.index(singles[0])]\n\n\ndef try_anomaly(trains: Trains, test: Grid) -> Optional[Grid]:\n    blocks = _anomaly_blocks(test)\n    if blocks is None or not trains:\n        return None\n    pick = _anomaly_pick(blocks, bg_color(test))\n    if pick is None:\n        return None\n    return [row[:] for row in pick]\n\n\ndef d_anomaly(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        blocks = _anomaly_blocks(inp)\n        if blocks is None:\n            return False\n        pick = _anomaly_pick(blocks, bg_color(inp))\n        if pick is None or pick != out:\n            return False\n    return verify_loo(try_anomaly, trains)\n\n# \u2500\u2500\u2500 Quadrant collage (bbox per quadrant) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 0bb8deee: exactly one full-length uniform horizontal\n# divider + one vertical divider split the input into 4 quadrants;\n# output is a 6x6 collage where each quadrant is replaced by the\n# tight bbox of its content (must fit 3x3, else None). Padding (when\n# the bbox is smaller) hugs content AWAY from the dividers: right\n# quadrants pad left, left quadrants pad right, top quadrants pad\n# bottom, bottom quadrants pad top (single evidenced case t2-TR;\n# LOO + sweep judge). Strict LOO detector.\n\ndef _qcollage_split(inp):\n    R, C = shape(inp)\n    bg = bg_color(inp)\n    hdiv = [r for r in range(R)\n            if len(set(inp[r])) == 1 and inp[r][0] != bg]\n    vdiv = [c for c in range(C)\n            if len({inp[r][c] for r in range(R)}) == 1\n            and inp[0][c] != bg]\n    if len(hdiv) != 1 or len(vdiv) != 1:\n        return None\n    dh, dv = hdiv[0], vdiv[0]\n    if dh == 0 or dh == R - 1 or dv == 0 or dv == C - 1:\n        return None\n    quads = {\n        \"TL\": ([row[:dv] for row in inp[:dh]], True, True),\n        \"TR\": ([row[dv + 1:] for row in inp[:dh]], True, False),\n        \"BL\": ([row[:dv] for row in inp[dh + 1:]], False, True),\n        \"BR\": ([row[dv + 1:] for row in inp[dh + 1:]], False, False),\n    }\n    return quads\n\n\ndef _qcollage_quad(q, bg, is_top, is_left):\n    content = [(r, c) for r, row in enumerate(q) for c, v in enumerate(row)\n               if v != bg]\n    if not content:\n        return None\n    rs = [r for r, _ in content]\n    cs = [c for _, c in content]\n    r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)\n    h, w = r1 - r0 + 1, c1 - c0 + 1\n    if h > 3 or w > 3:\n        return None\n    box = [[q[r][c] if q[r][c] != bg else 0\n            for c in range(c0, c1 + 1)] for r in range(r0, r1 + 1)]\n    out = [[0] * 3 for _ in range(3)]\n    r_off = 0 if is_top else 3 - h\n    c_off = 0 if is_left else 3 - w\n    for r in range(h):\n        for c in range(w):\n            out[r_off + r][c_off + c] = box[r][c]\n    return out\n\n\ndef try_qcollage(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    quads = _qcollage_split(test)\n    if quads is None:\n        return None\n    bg = bg_color(test)\n    cells = {}\n    for name, (q, is_top, is_left) in quads.items():\n        got = _qcollage_quad(q, bg, is_top, is_left)\n        if got is None:\n            return None\n        cells[name] = got\n    return [cells[\"TL\"][r] + cells[\"TR\"][r] for r in range(3)] + \\\n           [cells[\"BL\"][r] + cells[\"BR\"][r] for r in range(3)]\n\n\ndef d_qcollage(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        quads = _qcollage_split(inp)\n        if quads is None:\n            return False\n        bg = bg_color(inp)\n        cells = {}\n        for name, (q, is_top, is_left) in quads.items():\n            got = _qcollage_quad(q, bg, is_top, is_left)\n            if got is None:\n                return False\n            cells[name] = got\n        want = [cells[\"TL\"][r] + cells[\"TR\"][r] for r in range(3)] + \\\n               [cells[\"BL\"][r] + cells[\"BR\"][r] for r in range(3)]\n        if want != out:\n            return False\n    return verify_loo(try_qcollage, trains)\n\n# \u2500\u2500\u2500 Quadrant stack (z-order overlay) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in a68b268e: one full horizontal + one vertical divider split\n# the input into 4 SAME-SIZE quadrants, each with its own figure\n# color on bg; output stacks them with fixed z-priority TL > TR >\n# BL > BR (first non-bg wins, else 0). Hand-verified: every shared\n# cell follows the priority (e.g. {TR,BL,BR} \u2192 TR, {BL,BR} \u2192 BL).\n# Strict LOO detector.\n\ndef _qstack_place(quads, bg):\n    names = [\"TL\", \"TR\", \"BL\", \"BR\"]\n    grids = [quads[n][0] for n in names]\n    R = len(grids[0])\n    if any(len(g) != R or any(len(row) != len(grids[0][0]) for row in g)\n           for g in grids):\n        return None\n    C = len(grids[0][0])\n    out = []\n    for r in range(R):\n        row = []\n        for c in range(C):\n            v = 0\n            for g in grids:\n                if g[r][c] != bg:\n                    v = g[r][c]\n                    break\n            row.append(v)\n        out.append(row)\n    return out\n\n\ndef try_qstack(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    quads = _qcollage_split(test)\n    if quads is None:\n        return None\n    return _qstack_place(quads, bg_color(test))\n\n\ndef d_qstack(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        quads = _qcollage_split(inp)\n        if quads is None:\n            return False\n        if _qstack_place(quads, bg_color(inp)) != out:\n            return False\n    return verify_loo(try_qstack, trains)\n\n# \u2500\u2500\u2500 Quadrant stack 2 (TR-first overlay) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 75b8110e: NO dividers \u2014 plain even halves split into 4\n# quadrants; output stacks with fixed z-priority TR > BL > BR > TL\n# (first non-bg wins, else 0). Assert-verified on all trains+TEST\n# (an earlier eyeball park of this white was a misread \u2014 the code\n# verdicts rule). Strict LOO detector.\n\ndef _quads_split(inp):\n    R, C = shape(inp)\n    if R % 2 or C % 2 or R < 2 or C < 2:\n        return None\n    h, w = R // 2, C // 2\n    return {\n        \"TL\": [[v for v in row[:w]] for row in inp[:h]],\n        \"TR\": [[v for v in row[w:]] for row in inp[:h]],\n        \"BL\": [[v for v in row[:w]] for row in inp[h:]],\n        \"BR\": [[v for v in row[w:]] for row in inp[h:]],\n    }\n\n\ndef _qstack2_place(quads, bg):\n    grids = [quads[n] for n in (\"TR\", \"BL\", \"BR\", \"TL\")]\n    R = len(grids[0])\n    if any(len(g) != R or any(len(row) != len(grids[0][0]) for row in g)\n           for g in grids):\n        return None\n    C = len(grids[0][0])\n    out = []\n    for r in range(R):\n        row = []\n        for c in range(C):\n            v = 0\n            for g in grids:\n                if g[r][c] != bg:\n                    v = g[r][c]\n                    break\n            row.append(v)\n        out.append(row)\n    return out\n\n\ndef try_qstack2(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    quads = _quads_split(test)\n    if quads is None:\n        return None\n    return _qstack2_place(quads, bg_color(test))\n\n\ndef d_qstack2(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        quads = _quads_split(inp)\n        if quads is None:\n            return False\n        if _qstack2_place(quads, bg_color(inp)) != out:\n            return False\n    return verify_loo(try_qstack2, trains)\n\n# \u2500\u2500\u2500 Quadrant stack learned (consensus order) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in e99362f0: divider quads like qstack, but the priority order\n# is LEARNED per puzzle \u2014 the unique order consistent with ALL trains\n# (here [3,0,1,2]; computed, not assumed). Ambiguous (several\n# consistent orders) \u2192 None: an underdetermined mechanism is an\n# honest miss, not a guess. Strict LOO detector.\n\ndef _qstackL_order(trains):\n    import itertools as _it\n    ok = []\n    for order in _it.permutations(range(4)):\n        good = True\n        for inp, out in trains:\n            quads = _qcollage_split(inp)\n            if quads is None:\n                good = False\n                break\n            Q = [quads[\"TL\"][0], quads[\"TR\"][0],\n                 quads[\"BL\"][0], quads[\"BR\"][0]]\n            bg = bg_color(inp)\n            for r in range(len(Q[0])):\n                for c in range(len(Q[0][0])):\n                    v = 0\n                    for i in order:\n                        if Q[i][r][c] != bg:\n                            v = Q[i][r][c]\n                            break\n                    if out[r][c] != v:\n                        good = False\n                        break\n                if not good:\n                    break\n            if not good:\n                break\n        if good:\n            ok.append(order)\n    return ok[0] if len(ok) == 1 else None\n\n\ndef try_qstackL(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    order = _qstackL_order(trains)\n    if order is None:\n        return None\n    quads = _qcollage_split(test)\n    if quads is None:\n        return None\n    Q = [quads[\"TL\"][0], quads[\"TR\"][0],\n         quads[\"BL\"][0], quads[\"BR\"][0]]\n    bg = bg_color(test)\n    return [[next((Q[i][r][c] for i in order if Q[i][r][c] != bg), 0)\n             for c in range(len(Q[0][0]))] for r in range(len(Q[0]))]\n\n\ndef d_qstackL(trains):\n    if not trains:\n        return False\n    if _qstackL_order(trains) is None:\n        return False\n    return verify_loo(try_qstackL, trains)\n\n# \u2500\u2500\u2500 Vertical stack (strip z-order) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 281123b4: full-length uniform divider columns (all one\n# color) split the input into equal vertical strips; output stacks\n# them with halves priority \u2014 right-half strips left-to-right,\n# then left-half strips left-to-right (for 4 strips: B2>B3>B0>B1).\n# First non-bg wins, else 0. Hand-verified on all t0/t1 cells.\n# Strict LOO detector.\n\ndef _vstrips_split(inp):\n    R, C = shape(inp)\n    bycolor = {}\n    for c in range(C):\n        col = [inp[r][c] for r in range(R)]\n        if len(set(col)) == 1:\n            bycolor.setdefault(col[0], []).append(c)\n    cands = []\n    for _color, divs in bycolor.items():\n        bounds, start = [], 0\n        for c in range(C + 1):\n            if c in divs or c == C:\n                if start < c:\n                    bounds.append((start, c))\n                start = c + 1\n        if len(bounds) < 2:\n            continue\n        w0 = bounds[0][1] - bounds[0][0]\n        if any(b - a != w0 for a, b in bounds):\n            continue\n        cands.append([[row[a:b] for row in inp] for a, b in bounds])\n    if len(cands) != 1:\n        return None\n    return cands[0]\n\n\ndef _vstack_split(inp):\n    strips = _vstrips_split(inp)\n    if strips is None or len(strips) % 2:\n        return None\n    return strips\n\n\ndef try_vstack(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    strips = _vstack_split(test)\n    if strips is None:\n        return None\n    bg = bg_color(test)\n    half = len(strips) // 2\n    order = strips[half:] + strips[:half]\n    R = len(strips[0])\n    if any(len(s) != R or any(len(row) != len(strips[0][0]) for row in s)\n           for s in strips):\n        return None\n    C = len(strips[0][0])\n    return [[next((s[r][c] for s in order if s[r][c] != bg), 0)\n             for c in range(C)] for r in range(R)]\n\n\ndef d_vstack(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        strips = _vstack_split(inp)\n        if strips is None:\n            return False\n        if try_vstack(trains, inp) != out:\n            return False\n    return verify_loo(try_vstack, trains)\n\n# \u2500\u2500\u2500 Vertical union (conditional overlay) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in bbb1b8b6: one full uniform divider column splits the input\n# into same-size left+right strips; output = their union IFF they are\n# disjoint, else the left strip alone (all-or-nothing: t1's right 2x2\n# overlaps left, so even its non-overlapping cell (1,1)=2 stays 0 \u2014\n# a priority rule is refuted). Colors copied verbatim, nothing\n# learned. Extended 2026-09-12 (e98196ab): same with a horizontal\n# divider (top/bottom strips, else-first-strip); precedence vdiv\n# then hdiv (status quo first, deterministic). Strict LOO detector.\n\ndef _vunion_cand(inp, kind):\n    R, C = shape(inp)\n    bg = bg_color(inp)\n    cands = []\n    if kind == \"vdiv\":\n        for dv in range(C):\n            if len({inp[r][dv] for r in range(R)}) != 1:\n                continue\n            if inp[0][dv] == bg:\n                continue\n            left = [row[:dv] for row in inp]\n            right = [row[dv + 1:] for row in inp]\n            if not left[0] or not right[0]:\n                continue\n            if len(left[0]) != len(right[0]):\n                continue\n            cands.append((left, right))\n    else:\n        for dh in range(R):\n            if len(set(inp[dh])) != 1:\n                continue\n            if inp[dh][0] == bg:\n                continue\n            top, bot = inp[:dh], inp[dh + 1:]\n            if not top or not bot or len(top) != len(bot):\n                continue\n            if len(top[0]) != len(bot[0]):\n                continue\n            cands.append((top, bot))\n    if len(cands) != 1:\n        return None\n    return cands[0]\n\n\ndef _vunion_place(inp):\n    for kind in (\"vdiv\", \"hdiv\"):\n        sp = _vunion_cand(inp, kind)\n        if sp is None:\n            continue\n        a, b = sp\n        R = len(a)\n        bg = bg_color(inp)\n        if any(a[r][c] != bg and b[r][c] != bg\n               for r in range(R) for c in range(len(a[0]))):\n            return [row[:] for row in a]\n        return [[b[r][c] if a[r][c] == bg else a[r][c]\n                 for c in range(len(a[0]))] for r in range(R)]\n    return None\n\n\ndef try_vunion(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    return _vunion_place(test)\n\n\ndef d_vunion(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        if _vunion_place(inp) != out:\n            return False\n    return verify_loo(try_vunion, trains)\n\n# \u2500\u2500\u2500 Vertical first (strip reading priority) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in cf98881b: divider columns split into 3 equal strips;\n# output takes the first non-bg scanning strips left-to-right\n# (B0>B1>B2 \u2014 hand-verified on all trains+TEST; the halves priority\n# of vstack is refuted here and vice versa: the two detectors are\n# structurally exclusive). Strict LOO detector.\n\ndef try_vfirst(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    strips = _vstrips_split(test)\n    if strips is None or len(strips) < 2:\n        return None\n    bg = bg_color(test)\n    R = len(strips[0])\n    if any(len(s) != R or any(len(row) != len(strips[0][0]) for row in s)\n           for s in strips):\n        return None\n    C = len(strips[0][0])\n    return [[next((s[r][c] for s in strips if s[r][c] != bg), 0)\n             for c in range(C)] for r in range(R)]\n\n\ndef d_vfirst(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        strips = _vstrips_split(inp)\n        if strips is None or len(strips) < 2:\n            return False\n        if try_vfirst(trains, inp) != out:\n            return False\n    return verify_loo(try_vfirst, trains)\n\n# \u2500\u2500\u2500 Vertical rows (collect non-empty rows) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 25c199f5: divider columns split into equal strips sharing\n# one bg; output collects every row containing non-bg, blocks\n# right-to-left, each block's rows top-down, bottom-justified (pad\n# top with bg rows if short; more rows than block height \u2192 None).\n# Hand-verified: L-to-R + top-justify refuted (t0-r3 is B0r1, not\n# position 1). Strict LOO detector.\n\ndef try_vrows(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    strips = _vstrips_split(test)\n    if strips is None or len(strips) < 2:\n        return None\n    bg = bg_color(test)\n    H = len(strips[0])\n    W = len(strips[0][0])\n    if any(len(s) != H or any(len(row) != W for row in s)\n           for s in strips):\n        return None\n    rows = []\n    for s in reversed(strips):\n        for row in s:\n            if any(v != bg for v in row):\n                rows.append(row[:])\n    if len(rows) > H:\n        return None\n    return [[bg] * W for _ in range(H - len(rows))] + rows\n\n\ndef d_vrows(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        strips = _vstrips_split(inp)\n        if strips is None or len(strips) < 2:\n            return False\n        if try_vrows(trains, inp) != out:\n            return False\n    return verify_loo(try_vrows, trains)\n\n# \u2500\u2500\u2500 Vertical columns (positional column slices) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 5ecac7f7: three 5-wide strips share one bg; output takes\n# fixed positional slices \u2014 left strip cols 0-1, middle strip col 2\n# (center), right strip cols 3-4 (2+1+2). Hand-verified on all\n# trains+TEST (t1/t2/TEST exact cell-by-cell). Narrow scope (3\n# strips x width 5); LOO + sweep judge any generalization. Strict.\n\ndef try_vcols(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    strips = _vstrips_split(test)\n    if strips is None or len(strips) != 3:\n        return None\n    if any(len(s) != 5 or any(len(row) != 5 for row in s)\n           for s in strips):\n        return None\n    b0, b1, b2 = strips\n    return [[b0[r][0], b0[r][1], b1[r][2], b2[r][3], b2[r][4]]\n            for r in range(5)]\n\n\ndef d_vcols(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        strips = _vstrips_split(inp)\n        if strips is None or len(strips) != 3:\n            return False\n        if try_vcols(trains, inp) != out:\n            return False\n    return verify_loo(try_vcols, trains)\n\n# \u2500\u2500\u2500 Frames overlay (first figure wins) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 25e02866: two or more same-color SQUARE borders (h==w>=3,\n# border complete; interior may hold ring-colored cells \u2014 only the\n# border must be present, so solid blocks register harmlessly as\n# figure-less frames; bg-colored borders excluded) share one\n# (color,size); output paints the ring and overlays interiors in\n# frame reading order \u2014 first non-ring figure wins (in-sample\n# figures never overlap, so order is unloaded but deterministic).\n# Nothing learned across trains; LOO still guards per puzzle.\n\ndef _frames_find(inp):\n    bg = bg_color(inp)\n    found = []\n    for c in components(inp, bg=bg):\n        cells = set(c[\"cells\"])\n        rs = [r for r, _ in cells]\n        cs = [cc for _, cc in cells]\n        h, w = max(rs) - min(rs) + 1, max(cs) - min(cs) + 1\n        if h != w or h < 3:\n            continue\n        t, l = min(rs), min(cs)\n        border = ({(t, cc) for cc in range(l, l + w)} |\n                  {(t + h - 1, cc) for cc in range(l, l + w)} |\n                  {(r, l) for r in range(t, t + h)} |\n                  {(r, l + w - 1) for r in range(t, t + h)})\n        if border <= cells:\n            found.append((c[\"color\"], h, t, l))\n    return found\n\n\ndef try_frames(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    found = _frames_find(test)\n    if len(found) < 2:\n        return None\n    if len({(c, h) for c, h, _, _ in found}) != 1:\n        return None\n    color, N, _, _ = found[0]\n    out = [[color] * N for _ in range(N)]\n    for _, _, t, l in sorted(found, key=lambda f: (f[2], f[3])):\n        for r in range(1, N - 1):\n            for c in range(1, N - 1):\n                v = test[t + r][l + c]\n                if v != color and out[r][c] == color:\n                    out[r][c] = v\n    return out\n\n\ndef d_frames(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        if try_frames(trains, inp) != out:\n            return False\n    return verify_loo(try_frames, trains)\n\n# \u2500\u2500\u2500 Dot frame (dye in dots' frame) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 3de23699: dot-colored cells frame a canvas (their bbox\n# shrunk by 1); output shows that canvas with blob-colored cells\n# dyed to dot-color and everything else 0 (dots inside erase).\n# Blob = color with the uniquely-largest component; dot = the only\n# other non-bg color whose shrunk bbox is valid and contains all\n# blob cells. Colors vary per train by design (no cross-train\n# learning \u2014 structural rule, LOO guards). Strict LOO detector.\n\ndef _dotframe_place(grid):\n    R, C = shape(grid)\n    bg = bg_color(grid)\n    comps = components(grid, bg=bg)\n    if not comps:\n        return None\n    bycolor = {}\n    for c in comps:\n        bycolor.setdefault(c[\"color\"], []).append(c)\n    biggest = {col: max(len(c[\"cells\"]) for c in cs)\n               for col, cs in bycolor.items()}\n    top = max(biggest.values())\n    if sum(1 for v in biggest.values() if v == top) != 1:\n        return None\n    blob = next(col for col, v in biggest.items() if v == top)\n    bcells = [(r, c) for cs in [bycolor[blob]] for c in cs\n              for (r, c) in c[\"cells\"]]\n    br0, br1 = min(r for r, _ in bcells), max(r for r, _ in bcells)\n    bc0, bc1 = min(c for _, c in bcells), max(c for _, c in bcells)\n    cands = []\n    for col, cs in bycolor.items():\n        if col == blob:\n            continue\n        dcells = [(r, c) for c in cs for (r, c) in c[\"cells\"]]\n        dr0 = min(r for r, _ in dcells)\n        dr1 = max(r for r, _ in dcells)\n        dc0 = min(c for _, c in dcells)\n        dc1 = max(c for _, c in dcells)\n        cr, cc = (dr0 + 1, dr1 - 1), (dc0 + 1, dc1 - 1)\n        if cr[0] > cr[1] or cc[0] > cc[1]:\n            continue\n        if not (cr[0] <= br0 and br1 <= cr[1] and\n                cc[0] <= bc0 and bc1 <= cc[1]):\n            continue\n        cands.append((col, cr, cc))\n    if len(cands) != 1:\n        return None\n    dot, (r0, r1), (c0, c1) = cands[0]\n    return [[dot if grid[r][c] == blob else 0\n             for c in range(c0, c1 + 1)] for r in range(r0, r1 + 1)]\n\n\ndef try_dotframe(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    return _dotframe_place(test)\n\n\ndef d_dotframe(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        if _dotframe_place(inp) != out:\n            return False\n    return verify_loo(try_dotframe, trains)\n\n# \u2500\u2500\u2500 Below marker (fixed window) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 48d8fb45: the unique singleton-colored cell (exactly one\n# cell of its color grid-wide, bg excluded) is the marker; output is\n# the verbatim 3x3 window directly below it, horizontally centered\n# (rows mr+1..mr+3, cols mc-1..mc+1; out of bounds \u2192 None).\n# Hand-verified cell-exact on all trains+TEST. Nothing learned\n# across trains (marker found per grid); LOO still guards it.\n\ndef _below_window(grid):\n    R, C = shape(grid)\n    bg = bg_color(grid)\n    cnt = Counter(v for row in grid for v in row if v != bg)\n    ones = [c for c, n in cnt.items() if n == 1]\n    if len(ones) != 1:\n        return None\n    m = ones[0]\n    mr = next(r for r in range(R) for c in range(C) if grid[r][c] == m)\n    mc = next(c for r in range(R) for c in range(C) if grid[r][c] == m)\n    if mr + 3 >= R or mc < 1 or mc + 1 >= C:\n        return None\n    return [[grid[r][c] for c in range(mc - 1, mc + 2)]\n            for r in range(mr + 1, mr + 4)]\n\n\ndef try_below(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    return _below_window(test)\n\n\ndef d_below(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        if _below_window(inp) != out:\n            return False\n    return verify_loo(try_below, trains)\n\n# \u2500\u2500\u2500 Midline (line through object) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in da2b0fe3: the line runs along the object's MINOR principal\n# axis (variance comparison \u2014 robust to outliers where bbox extent\n# lies: t1's stray cell stretches bbox width but not spread, and the\n# line is horizontal). Position is the ceiling of the bbox midpoint\n# along the line axis; the line overwrites everything with the\n# learned color (singleton across trains). Equal variances (a dot \u2014\n# no orientable axis) \u2192 None. Hand-verified + asserts. Strict.\n\ndef _midline_orient(fig):\n    n = len(fig)\n    mr = sum(r for r, _ in fig) / n\n    mc = sum(c for _, c in fig) / n\n    vr = sum((r - mr) ** 2 for r, _ in fig) / n\n    vc = sum((c - mc) ** 2 for _, c in fig) / n\n    if vr == vc:\n        return None\n    return \"h\" if vr > vc else \"v\"\n\n\ndef _midline_params(trains):\n    colors = set()\n    for inp, out in trains:\n        bg = bg_color(inp)\n        R, C = shape(inp)\n        if shape(out) != (R, C):\n            return None\n        fig = [(r, c) for r in range(R) for c in range(C)\n               if inp[r][c] != bg]\n        if not fig:\n            return None\n        orient = _midline_orient(fig)\n        if orient is None:\n            return None\n        r0, r1 = min(r for r, _ in fig), max(r for r, _ in fig)\n        c0, c1 = min(c for _, c in fig), max(c for _, c in fig)\n        want = [row[:] for row in inp]\n        if orient == \"v\":\n            pos = (c0 + c1 + 1) // 2\n            for r in range(R):\n                want[r][pos] = None\n        else:\n            pos = (r0 + r1 + 1) // 2\n            for c in range(C):\n                want[pos][c] = None\n        vals = {out[r][c] for (r, c) in\n                [(r, c) for r in range(R) for c in range(C)\n                 if want[r][c] is None]}\n        if len(vals) != 1:\n            return None\n        m = next(iter(vals))\n        chk = [row[:] for row in inp]\n        if orient == \"v\":\n            for r in range(R):\n                chk[r][pos] = m\n        else:\n            for c in range(C):\n                chk[pos][c] = m\n        if chk != out:\n            return None\n        colors.add(m)\n    if len(colors) != 1:\n        return None\n    return next(iter(colors))\n\n\ndef try_midline(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    m = _midline_params(trains)\n    if m is None:\n        return None\n    bg = bg_color(test)\n    R, C = shape(test)\n    fig = [(r, c) for r in range(R) for c in range(C)\n           if test[r][c] != bg]\n    if not fig:\n        return None\n    orient = _midline_orient(fig)\n    if orient is None:\n        return None\n    r0, r1 = min(r for r, _ in fig), max(r for r, _ in fig)\n    c0, c1 = min(c for _, c in fig), max(c for _, c in fig)\n    got = [row[:] for row in test]\n    if orient == \"v\":\n        pos = (c0 + c1 + 1) // 2\n        for r in range(R):\n            got[r][pos] = m\n    else:\n        pos = (r0 + r1 + 1) // 2\n        for c in range(C):\n            got[pos][c] = m\n    return got\n\n\ndef d_midline(trains):\n    if not trains:\n        return False\n    if _midline_params(trains) is None:\n        return False\n    return verify_loo(try_midline, trains)\n\n# \u2500\u2500\u2500 Asymmetric block (odd one out) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 662c240a: a vertical strip of 3x3 blocks (width 3, rows\n# multiple of 3); output is a verbatim copy of the UNIQUE block\n# that is NOT transpose-symmetric (exactly one, else None).\n# Hand-verified + asserts on all trains+TEST. Strict LOO detector.\n\ndef _transpose_sym(g):\n    n = len(g)\n    if any(len(row) != n for row in g):\n        return False\n    return all(g[r][c] == g[c][r] for r in range(n) for c in range(n))\n\n\ndef _asym_blocks(inp):\n    R, C = shape(inp)\n    if C != 3 or R % 3 or R < 6:\n        return None\n    return [[row[:] for row in inp[r:r + 3]] for r in range(0, R, 3)]\n\n\ndef try_asym(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    blocks = _asym_blocks(test)\n    if blocks is None:\n        return None\n    odd = [b for b in blocks if not _transpose_sym(b)]\n    if len(odd) != 1:\n        return None\n    return [row[:] for row in odd[0]]\n\n\ndef d_asym(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        blocks = _asym_blocks(inp)\n        if blocks is None:\n            return False\n        odd = [b for b in blocks if not _transpose_sym(b)]\n        if len(odd) != 1 or odd[0] != out:\n            return False\n    return verify_loo(try_asym, trains)\n\n# \u2500\u2500\u2500 Divider-emitter with push (05a7bcf2) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Full-length 8-divider splits emitter-side (4s) from mover-side (2s).\n# Per grid (orientation varies: vertical col or horizontal row):\n#   A: every 4 -> 3, plus 4-fill from block to divider (gap must be bg).\n#   D: rays of 8 from divider to far edge on B-side, on ray lines\n#      (rows/cols containing 4s). Rest of ray line = 8.\n#   B: 2s overwritten by rays reappear far-edge-justified on same\n#      line, order preserved (counts conserved). Non-ray lines untouched.\n# Strict: exactly one full-8 divider, no 3s in input, 4s/2s on opposite\n# sides, gaps bg-only, ray-line B-cells bg/2-only. LOO-verified.\n\ndef _divemit_apply(g):\n    R, C = shape(g)\n    if R == 0 or C == 0:\n        return None\n    bg = bg_color(g)\n    # Divider is specifically full-length 8 (mover spine is also full-length\n    # but color 2: e.g. train0 col20 all-2, train1 row21 all-2).\n    hdiv = [r for r in range(R) if len(set(g[r])) == 1 and g[r][0] == 8]\n    vdiv = [c for c in range(C)\n            if len({g[r][c] for r in range(R)}) == 1 and g[0][c] == 8]\n    if len(hdiv) + len(vdiv) != 1:\n        return None\n    if any(v == 3 for row in g for v in row):\n        return None\n    fours = [(r, c) for r in range(R) for c in range(C) if g[r][c] == 4]\n    twos = [(r, c) for r in range(R) for c in range(C) if g[r][c] == 2]\n    if not fours or not twos:\n        return None\n    out = [row[:] for row in g]\n    if vdiv:\n        dc = vdiv[0]\n        if g[0][dc] != 8:\n            return None\n        left4 = [(r, c) for r, c in fours if c < dc]\n        right4 = [(r, c) for r, c in fours if c > dc]\n        left2 = [(r, c) for r, c in twos if c < dc]\n        right2 = [(r, c) for r, c in twos if c > dc]\n        if any(c == dc for _, c in fours + twos):\n            return None\n        if bool(left4) == bool(right4):\n            return None\n        if bool(left2) == bool(right2):\n            return None\n        a_left = bool(left4)\n        b_left = bool(left2)\n        if a_left == b_left:\n            return None\n        for r, c in fours:\n            out[r][c] = 3\n        if a_left:\n            rows = sorted(set(r for r, _ in left4))\n            for r in rows:\n                maxc = max(c for rr, c in left4 if rr == r)\n                for c in range(maxc + 1, dc):\n                    if out[r][c] != bg:\n                        return None\n                    out[r][c] = 4\n            for r in rows:\n                for c in range(dc + 1, C):\n                    if g[r][c] != bg and g[r][c] != 2:\n                        return None\n                vals = [g[r][c] for c in range(dc + 1, C) if g[r][c] == 2]\n                k = len(vals)\n                for c in range(dc + 1, C):\n                    out[r][c] = 8\n                for i in range(k):\n                    out[r][C - k + i] = vals[i]\n        else:\n            rows = sorted(set(r for r, _ in right4))\n            for r in rows:\n                minc = min(c for rr, c in right4 if rr == r)\n                for c in range(dc + 1, minc):\n                    if out[r][c] != bg:\n                        return None\n                    out[r][c] = 4\n            for r in rows:\n                for c in range(0, dc):\n                    if g[r][c] != bg and g[r][c] != 2:\n                        return None\n                vals = [g[r][c] for c in range(0, dc) if g[r][c] == 2]\n                k = len(vals)\n                for c in range(0, dc):\n                    out[r][c] = 8\n                for i in range(k):\n                    out[r][i] = vals[i]\n        return out\n    else:\n        dr = hdiv[0]\n        if g[dr][0] != 8:\n            return None\n        up4 = [(r, c) for r, c in fours if r < dr]\n        dn4 = [(r, c) for r, c in fours if r > dr]\n        up2 = [(r, c) for r, c in twos if r < dr]\n        dn2 = [(r, c) for r, c in twos if r > dr]\n        if any(r == dr for r, _ in fours + twos):\n            return None\n        if bool(up4) == bool(dn4):\n            return None\n        if bool(up2) == bool(dn2):\n            return None\n        a_up = bool(up4)\n        b_up = bool(up2)\n        if a_up == b_up:\n            return None\n        for r, c in fours:\n            out[r][c] = 3\n        if a_up:\n            cols = sorted(set(c for _, c in up4))\n            for c in cols:\n                maxr = max(r for r, cc in up4 if cc == c)\n                for r in range(maxr + 1, dr):\n                    if out[r][c] != bg:\n                        return None\n                    out[r][c] = 4\n            for c in cols:\n                for r in range(dr + 1, R):\n                    if g[r][c] != bg and g[r][c] != 2:\n                        return None\n                vals = [g[r][c] for r in range(dr + 1, R) if g[r][c] == 2]\n                k = len(vals)\n                for r in range(dr + 1, R):\n                    out[r][c] = 8\n                for i in range(k):\n                    out[R - k + i][c] = vals[i]\n        else:\n            cols = sorted(set(c for _, c in dn4))\n            for c in cols:\n                minr = min(r for r, cc in dn4 if cc == c)\n                for r in range(dr + 1, minr):\n                    if out[r][c] != bg:\n                        return None\n                    out[r][c] = 4\n            for c in cols:\n                for r in range(0, dr):\n                    if g[r][c] != bg and g[r][c] != 2:\n                        return None\n                vals = [g[r][c] for r in range(0, dr) if g[r][c] == 2]\n                k = len(vals)\n                for r in range(0, dr):\n                    out[r][c] = 8\n                for i in range(k):\n                    out[i][c] = vals[i]\n        return out\n\n\ndef try_divemit(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    return _divemit_apply(test)\n\n\ndef d_divemit(trains):\n    if not trains:\n        return False\n    for inp, out in trains:\n        if _divemit_apply(inp) != out:\n            return False\n    return verify_loo(try_divemit, trains)\n\n# \u2500\u2500\u2500 Hub-4 star wiring (0e671a1a) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Input holds singletons only: exactly one 4 (hub), every other dot\n# color 2 or 3. Each spoke wires to the hub with an L-shaped path in\n# a NEW learned paint color: 3-dots go vertical-first (own column to\n# hub row, then hub row to hub), 2-dots go horizontal-first (own row\n# to hub column, then hub column to hub). Path cells must be bg\n# (strict). Same shape in/out. LOO-verified.\n\ndef _hub4_span(a, b):\n    return range(min(a, b), max(a, b) + 1)\n\n\ndef _hub4_params(trains):\n    marks = set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        bg = bg_color(inp)\n        comps = components(inp, bg=bg)\n        if any(c[\"size\"] != 1 for c in comps):\n            return None\n        hubs = [c for c in comps if c[\"color\"] == 4]\n        spokes = [c for c in comps if c[\"color\"] != 4]\n        if len(hubs) != 1 or not spokes:\n            return None\n        if any(c[\"color\"] not in (2, 3) for c in spokes):\n            return None\n        want = [row[:] for row in inp]\n        hr, hc = hubs[0][\"anchor\"]\n        for c in spokes:\n            dr, dc = c[\"anchor\"]\n            if c[\"color\"] == 3:\n                cells = ({(r, dc) for r in _hub4_span(dr, hr) if r != dr} |\n                         {(hr, cc) for cc in _hub4_span(dc, hc) if cc != hc})\n            else:\n                cells = ({(dr, cc) for cc in _hub4_span(dc, hc) if cc != dc} |\n                         {(r, hc) for r in _hub4_span(dr, hr) if r != hr})\n            for (rr, cc) in cells:\n                if want[rr][cc] is not None and want[rr][cc] != bg:\n                    return None\n                want[rr][cc] = None\n        vals = {out[rr][cc] for rr in range(len(out))\n                for cc in range(len(out[0])) if want[rr][cc] is None}\n        if len(vals) != 1:\n            return None\n        m = next(iter(vals))\n        if m == bg or m in {v for row in inp for v in row}:\n            return None\n        for rr in range(len(out)):\n            for cc in range(len(out[0])):\n                if want[rr][cc] is None:\n                    want[rr][cc] = m\n        if want != out:\n            return None\n        marks.add(m)\n    return next(iter(marks)) if len(marks) == 1 else None\n\n\ndef try_hub4(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    m = _hub4_params(trains)\n    if m is None:\n        return None\n    bg = bg_color(test)\n    comps = components(test, bg=bg)\n    if any(c[\"size\"] != 1 for c in comps):\n        return None\n    hubs = [c for c in comps if c[\"color\"] == 4]\n    spokes = [c for c in comps if c[\"color\"] != 4]\n    if len(hubs) != 1 or not spokes:\n        return None\n    if any(c[\"color\"] not in (2, 3) for c in spokes):\n        return None\n    if m in {v for row in test for v in row}:\n        return None\n    got = [row[:] for row in test]\n    hr, hc = hubs[0][\"anchor\"]\n    for c in spokes:\n        dr, dc = c[\"anchor\"]\n        if c[\"color\"] == 3:\n            cells = ({(r, dc) for r in _hub4_span(dr, hr) if r != dr} |\n                     {(hr, cc) for cc in _hub4_span(dc, hc) if cc != hc})\n        else:\n            cells = ({(dr, cc) for cc in _hub4_span(dc, hc) if cc != dc} |\n                     {(r, hc) for r in _hub4_span(dr, hr) if r != hr})\n        for (rr, cc) in cells:\n            if got[rr][cc] != bg and got[rr][cc] != m:\n                return None\n            got[rr][cc] = m\n    return got\n\n\ndef d_hub4(trains):\n    if not trains:\n        return False\n    if _hub4_params(trains) is None:\n        return False\n    return verify_loo(try_hub4, trains)\n\n# \u2500\u2500\u2500 Halo (0ca9ddb6) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Every 1-cell paints its orthogonal neighbors, every 2-cell its\n# diagonal neighbors, in learned NEW marks (7-ish / 4-ish). Clipped\n# at the edge; halo cells must be bg (strict). Other colors pass\n# through untouched. Same shape in/out. LOO-verified.\n\n_ORTHO = [(-1, 0), (1, 0), (0, -1), (0, 1)]\n_DIAG = [(-1, -1), (-1, 1), (1, -1), (1, 1)]\n\n\ndef _halo_params(trains):\n    m1s, m2s = set(), set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        Rr, Cc = shape(inp)\n        bg = bg_color(inp)\n        cells1 = [(r, c) for r in range(Rr) for c in range(Cc)\n                  if inp[r][c] == 1]\n        cells2 = [(r, c) for r in range(Rr) for c in range(Cc)\n                  if inp[r][c] == 2]\n        if not cells1 or not cells2:\n            return None\n        want = [row[:] for row in inp]\n        for (r, c) in cells1:\n            for dr, dc in _ORTHO:\n                rr, cc = r + dr, c + dc\n                if not (0 <= rr < Rr and 0 <= cc < Cc):\n                    continue\n                if want[rr][cc] != bg:\n                    return None\n                want[rr][cc] = \"h1\"\n        for (r, c) in cells2:\n            for dr, dc in _DIAG:\n                rr, cc = r + dr, c + dc\n                if not (0 <= rr < Rr and 0 <= cc < Cc):\n                    continue\n                if want[rr][cc] != bg:\n                    return None\n                want[rr][cc] = \"h2\"\n        v1 = {out[rr][cc] for rr in range(Rr) for cc in range(Cc)\n              if want[rr][cc] == \"h1\"}\n        v2 = {out[rr][cc] for rr in range(Rr) for cc in range(Cc)\n              if want[rr][cc] == \"h2\"}\n        if len(v1) != 1 or len(v2) != 1:\n            return None\n        pal = {v for row in inp for v in row}\n        if v1 == v2 or next(iter(v1)) in pal or next(iter(v2)) in pal:\n            return None\n        m1s.add(next(iter(v1)))\n        m2s.add(next(iter(v2)))\n        for rr in range(Rr):\n            for cc in range(Cc):\n                if want[rr][cc] == \"h1\":\n                    want[rr][cc] = next(iter(v1))\n                elif want[rr][cc] == \"h2\":\n                    want[rr][cc] = next(iter(v2))\n        if want != out:\n            return None\n    if len(m1s) != 1 or len(m2s) != 1:\n        return None\n    return next(iter(m1s)), next(iter(m2s))\n\n\ndef try_halo(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    pm = _halo_params(trains)\n    if pm is None:\n        return None\n    m1, m2 = pm\n    Rr, Cc = shape(test)\n    bg = bg_color(test)\n    if m1 in {v for row in test for v in row}:\n        return None\n    if m2 in {v for row in test for v in row}:\n        return None\n    got = [row[:] for row in test]\n    for r in range(Rr):\n        for c in range(Cc):\n            if test[r][c] == 1:\n                for dr, dc in _ORTHO:\n                    rr, cc = r + dr, c + dc\n                    if not (0 <= rr < Rr and 0 <= cc < Cc):\n                        continue\n                    if got[rr][cc] != bg:\n                        return None\n                    got[rr][cc] = m1\n    for r in range(Rr):\n        for c in range(Cc):\n            if test[r][c] == 2:\n                for dr, dc in _DIAG:\n                    rr, cc = r + dr, c + dc\n                    if not (0 <= rr < Rr and 0 <= cc < Cc):\n                        continue\n                    if got[rr][cc] != bg and got[rr][cc] != m1:\n                        return None\n                    got[rr][cc] = m2\n    return got\n\n\ndef d_halo(trains):\n    if not trains:\n        return False\n    if _halo_params(trains) is None:\n        return False\n    return verify_loo(try_halo, trains)\n\n# \u2500\u2500\u2500 Diagonal focus (11e1fe23) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Dots (non-bg singletons, 3+). Exactly one diagonal (either slope)\n# holds 2+ dots: the shared axis. Its crossings with other dots'\n# diagonals yield exactly one interior non-dot cell: the focus,\n# painted a learned NEW mark. Every dot spawns a same-color copy on\n# its own diagonal ADJACENT to the focus (i.e. one step back from\n# it); the whole dot-to-copy path must be bg. Originals kept.\n# Exact integer geometry, no floats. LOO-verified.\n\ndef _diag_lines(dots):\n    from collections import defaultdict as _dd\n    by_sum, by_dif = _dd(list), _dd(list)\n    for d in dots:\n        by_sum[d[0] + d[1]].append(d)\n        by_dif[d[0] - d[1]].append(d)\n    return by_sum, by_dif\n\n\ndef _diagfocus_params(trains):\n    marks = set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        Rr, Cc = shape(inp)\n        bg = bg_color(inp)\n        comps = components(inp, bg=bg)\n        if len(comps) < 3 or any(c[\"size\"] != 1 for c in comps):\n            return None\n        dots = [c[\"anchor\"] for c in comps]\n        dset = set(dots)\n        by_sum, by_dif = _diag_lines(dots)\n        shared = ([(\"s\", k) for k, v in by_sum.items() if len(v) >= 2] +\n                  [(\"d\", k) for k, v in by_dif.items() if len(v) >= 2])\n        if len(shared) != 1:\n            return None\n        kind, kk = shared[0]\n        cands = set()\n        for (dr, dc) in dots:\n            s2 = dr + dc\n            if not (kind == \"s\" and s2 == kk):\n                for (qr, qc) in _seg_intersect_sum(s2, kind, kk, Rr, Cc):\n                    cands.add((qr, qc))\n            d2 = dr - dc\n            if not (kind == \"d\" and d2 == kk):\n                for (qr, qc) in _seg_intersect_dif(d2, kind, kk, Rr, Cc):\n                    cands.add((qr, qc))\n        cands -= dset\n        cands = {(r, c) for (r, c) in cands if 0 <= r < Rr and 0 <= c < Cc}\n        if len(cands) != 1:\n            return None\n        fr, fc = next(iter(cands))\n        if inp[fr][fc] != bg:\n            return None\n        want = [row[:] for row in inp]\n        for (dr, dc) in dots:\n            sr = (fr > dr) - (fr < dr)\n            sc = (fc > dc) - (fc < dc)\n            if sr == 0 or sc == 0:\n                return None\n            lr, lc = fr - sr, fc - sc\n            if (lr, lc) == (dr, dc):\n                return None\n            if not (0 <= lr < Rr and 0 <= lc < Cc):\n                return None\n            rr, cc = dr + sr, dc + sc\n            while (rr, cc) != (lr, lc):\n                if want[rr][cc] != bg:\n                    return None\n                rr, cc = rr + sr, cc + sc\n            if want[lr][lc] != bg:\n                return None\n            # intermediate stays bg; landing gets (\"copy\", color) below\n            want[lr][lc] = (\"copy\", inp[dr][dc])\n        vals = {out[fr][fc]}\n        for rr in range(Rr):\n            for cc in range(Cc):\n                w = want[rr][cc]\n                if isinstance(w, tuple) and out[rr][cc] != w[1]:\n                    return None\n        m = next(iter(vals))\n        pal = {v for row in inp for v in row}\n        if m == bg or m in pal:\n            return None\n        got = [row[:] for row in inp]\n        for rr in range(Rr):\n            for cc in range(Cc):\n                if isinstance(want[rr][cc], tuple):\n                    got[rr][cc] = want[rr][cc][1]\n        got[fr][fc] = m\n        if got != out:\n            return None\n        marks.add(m)\n    return next(iter(marks)) if len(marks) == 1 else None\n\n\ndef _seg_intersect_sum(s2, kind, kk, Rr, Cc):\n    # line r+c=s2  X  shared axis -> lattice points on both\n    pts = []\n    if kind == \"s\":\n        return pts  # parallel, no crossing\n    # shared is r-c=kk -> r=(s2+kk)/2, c=(s2-kk)/2\n    if (s2 + kk) % 2:\n        return pts\n    r, c = (s2 + kk) // 2, (s2 - kk) // 2\n    if 0 <= r < Rr and 0 <= c < Cc:\n        pts.append((r, c))\n    return pts\n\n\ndef _seg_intersect_dif(d2, kind, kk, Rr, Cc):\n    pts = []\n    if kind == \"d\":\n        return pts  # parallel, no crossing\n    # shared is r+c=kk -> r=(kk+d2)/2, c=(kk-d2)/2\n    if (kk + d2) % 2:\n        return pts\n    r, c = (kk + d2) // 2, (kk - d2) // 2\n    if 0 <= r < Rr and 0 <= c < Cc:\n        pts.append((r, c))\n    return pts\n\n\ndef try_diagfocus(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    m = _diagfocus_params(trains)\n    if m is None:\n        return None\n    Rr, Cc = shape(test)\n    bg = bg_color(test)\n    comps = components(test, bg=bg)\n    if len(comps) < 3 or any(c[\"size\"] != 1 for c in comps):\n        return None\n    dots = [c[\"anchor\"] for c in comps]\n    dset = set(dots)\n    by_sum, by_dif = _diag_lines(dots)\n    shared = ([(\"s\", k) for k, v in by_sum.items() if len(v) >= 2] +\n              [(\"d\", k) for k, v in by_dif.items() if len(v) >= 2])\n    if len(shared) != 1:\n        return None\n    kind, kk = shared[0]\n    cands = set()\n    for (dr, dc) in dots:\n        s2 = dr + dc\n        if not (kind == \"s\" and s2 == kk):\n            cands.update(_seg_intersect_sum(s2, kind, kk, Rr, Cc))\n        d2 = dr - dc\n        if not (kind == \"d\" and d2 == kk):\n            cands.update(_seg_intersect_dif(d2, kind, kk, Rr, Cc))\n    cands -= dset\n    cands = {(r, c) for (r, c) in cands if 0 <= r < Rr and 0 <= c < Cc}\n    if len(cands) != 1:\n        return None\n    fr, fc = next(iter(cands))\n    if test[fr][fc] != bg:\n        return None\n    if m in {v for row in test for v in row}:\n        return None\n    got = [row[:] for row in test]\n    for (dr, dc) in dots:\n        sr = (fr > dr) - (fr < dr)\n        sc = (fc > dc) - (fc < dc)\n        if sr == 0 or sc == 0:\n            return None\n        lr, lc = fr - sr, fc - sc\n        if (lr, lc) == (dr, dc):\n            return None\n        if not (0 <= lr < Rr and 0 <= lc < Cc):\n            return None\n        rr, cc = dr + sr, dc + sc\n        while (rr, cc) != (lr, lc):\n            if got[rr][cc] != bg:\n                return None\n            rr, cc = rr + sr, cc + sc\n        if got[lr][lc] != bg:\n            return None\n        got[lr][lc] = test[dr][dc]\n    got[fr][fc] = m\n    return got\n\n\ndef d_diagfocus(trains):\n    if not trains:\n        return False\n    if _diagfocus_params(trains) is None:\n        return False\n    return verify_loo(try_diagfocus, trains)\n\n# \u2500\u2500\u2500 Full cross (140c817e) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Input palette is {bg, 1} only. Every 1-cell paints its FULL row +\n# FULL column with 1; every original 1-cell becomes a learned NEW\n# dot-mark (2-ish); every diagonal neighbor of an original 1-cell\n# becomes a learned NEW diag-mark (3-ish, bg-only strict: a diag\n# landing on a bar or another dot fails). Same shape. LOO-verified.\n\ndef _cross1_params(trains):\n    mdots, mdiag = set(), set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        Rr, Cc = shape(inp)\n        bg = bg_color(inp)\n        if {v for row in inp for v in row} - {bg, 1}:\n            return None\n        ones = [(r, c) for r in range(Rr) for c in range(Cc)\n                if inp[r][c] == 1]\n        if not ones:\n            return None\n        want = [row[:] for row in inp]\n        for (r, c) in ones:\n            for cc in range(Cc):\n                if want[r][cc] != bg and want[r][cc] != 1:\n                    return None\n                want[r][cc] = 1\n            for rr in range(Rr):\n                if want[rr][c] != bg and want[rr][c] != 1:\n                    return None\n                want[rr][c] = 1\n        for (r, c) in ones:\n            want[r][c] = \"dot\"\n        for (r, c) in ones:\n            for dr, dc in _DIAG:\n                rr, cc = r + dr, c + dc\n                if not (0 <= rr < Rr and 0 <= cc < Cc):\n                    continue\n                if want[rr][cc] != bg:\n                    return None\n                want[rr][cc] = \"dg\"\n        vd = {out[rr][cc] for rr in range(Rr) for cc in range(Cc)\n              if want[rr][cc] == \"dot\"}\n        vg = {out[rr][cc] for rr in range(Rr) for cc in range(Cc)\n              if want[rr][cc] == \"dg\"}\n        if len(vd) != 1 or len(vg) != 1 or vd == vg:\n            return None\n        pal = {v for row in inp for v in row}\n        if next(iter(vd)) in pal or next(iter(vg)) in pal:\n            return None\n        mdots.add(next(iter(vd)))\n        mdiag.add(next(iter(vg)))\n        for rr in range(Rr):\n            for cc in range(Cc):\n                if want[rr][cc] == \"dot\":\n                    want[rr][cc] = next(iter(vd))\n                elif want[rr][cc] == \"dg\":\n                    want[rr][cc] = next(iter(vg))\n        if want != out:\n            return None\n    if len(mdots) != 1 or len(mdiag) != 1:\n        return None\n    return next(iter(mdots)), next(iter(mdiag))\n\n\ndef try_cross1(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    pm = _cross1_params(trains)\n    if pm is None:\n        return None\n    mdot, mdiag = pm\n    Rr, Cc = shape(test)\n    bg = bg_color(test)\n    if {v for row in test for v in row} - {bg, 1}:\n        return None\n    pal = {v for row in test for v in row}\n    if mdot in pal or mdiag in pal:\n        return None\n    ones = [(r, c) for r in range(Rr) for c in range(Cc)\n            if test[r][c] == 1]\n    if not ones:\n        return None\n    got = [row[:] for row in test]\n    for (r, c) in ones:\n        for cc in range(Cc):\n            if got[r][cc] != bg and got[r][cc] != 1:\n                return None\n            got[r][cc] = 1\n        for rr in range(Rr):\n            if got[rr][c] != bg and got[rr][c] != 1:\n                return None\n            got[rr][c] = 1\n    for (r, c) in ones:\n        got[r][c] = mdot\n    for (r, c) in ones:\n        for dr, dc in _DIAG:\n            rr, cc = r + dr, c + dc\n            if not (0 <= rr < Rr and 0 <= cc < Cc):\n                continue\n            if got[rr][cc] != bg:\n                return None\n            got[rr][cc] = mdiag\n    return got\n\n\ndef d_cross1(trains):\n    if not trains:\n        return False\n    if _cross1_params(trains) is None:\n        return False\n    return verify_loo(try_cross1, trains)\n\n# \u2500\u2500\u2500 Square frame corner marks (14b8e18c) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# For every non-bg connected component whose bbox is a SQUARE of\n# side >= 2 with all four corners in the component's own color,\n# paint the 8 corner-aligned outward neighbors (2 per corner) with\n# a learned NEW mark. Clipped at the edge; outward cells must be\n# bg (strict). Non-square bboxes and 1x1 singletons project\n# nothing. Same shape in/out. LOO-verified.\n\ndef _sqframe_targets(comps):\n    out = []\n    for c in comps:\n        cells = {(x, y) for x, y in c[\"cells\"]}\n        rs = [r for r, _ in c[\"cells\"]]\n        cs = [cc for _, cc in c[\"cells\"]]\n        r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)\n        if r1 - r0 != c1 - c0 or r1 - r0 < 1:\n            continue\n        if any((r, cc) not in cells\n               for r, cc in ((r0, c0), (r0, c1), (r1, c0), (r1, c1))):\n            continue\n        out.append((c[\"color\"], r0, r1, c0, c1))\n    return out\n\n\ndef _sqframe_params(trains):\n    marks = set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        Rr, Cc = shape(inp)\n        bg = bg_color(inp)\n        tg = _sqframe_targets(components(inp, bg=bg))\n        if not tg:\n            return None\n        want = [row[:] for row in inp]\n        for (_col, r0, r1, c0, c1) in tg:\n            for (rr, cc) in ((r0 - 1, c0), (r0 - 1, c1),\n                             (r1 + 1, c0), (r1 + 1, c1),\n                             (r0, c0 - 1), (r1, c0 - 1),\n                             (r0, c1 + 1), (r1, c1 + 1)):\n                if not (0 <= rr < Rr and 0 <= cc < Cc):\n                    continue\n                if want[rr][cc] != bg:\n                    return None\n                want[rr][cc] = None\n        vals = {out[rr][cc] for rr in range(Rr) for cc in range(Cc)\n                if want[rr][cc] is None}\n        if len(vals) != 1:\n            return None\n        m = next(iter(vals))\n        if m == bg or m in {v for row in inp for v in row}:\n            return None\n        for rr in range(Rr):\n            for cc in range(Cc):\n                if want[rr][cc] is None:\n                    want[rr][cc] = m\n        if want != out:\n            return None\n        marks.add(m)\n    return next(iter(marks)) if len(marks) == 1 else None\n\n\ndef try_sqframe(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    m = _sqframe_params(trains)\n    if m is None:\n        return None\n    Rr, Cc = shape(test)\n    bg = bg_color(test)\n    if m in {v for row in test for v in row}:\n        return None\n    tg = _sqframe_targets(components(test, bg=bg))\n    if not tg:\n        return None\n    got = [row[:] for row in test]\n    for (_col, r0, r1, c0, c1) in tg:\n        for (rr, cc) in ((r0 - 1, c0), (r0 - 1, c1),\n                         (r1 + 1, c0), (r1 + 1, c1),\n                         (r0, c0 - 1), (r1, c0 - 1),\n                         (r0, c1 + 1), (r1, c1 + 1)):\n            if not (0 <= rr < Rr and 0 <= cc < Cc):\n                continue\n            if got[rr][cc] != bg and got[rr][cc] != m:\n                return None\n            got[rr][cc] = m\n    return got\n\n\ndef d_sqframe(trains):\n    if not trains:\n        return False\n    if _sqframe_params(trains) is None:\n        return False\n    return verify_loo(try_sqframe, trains)\n\n# \u2500\u2500\u2500 Tiling gaps (137f0df0) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Full-bg rows/cols strictly INSIDE the non-bg bbox (blocks on both\n# sides) are gap lines: cells inside the bbox paint a learned NEW\n# inside-mark, cells outside a learned NEW outside-mark. Crossings\n# of a gap row with a gap column must agree. Exterior full-bg\n# lines are not gap lines (they only receive outside-marks where\n# crossed). Same shape in/out. LOO-verified.\n\ndef _tilegap_lines(g, bg, r0, r1, c0, c1):\n    Rr, Cc = shape(g)\n    grows = [r for r in range(Rr)\n             if all(g[r][c] == bg for c in range(Cc)) and r0 < r < r1]\n    gcols = [c for c in range(Cc)\n             if all(g[r][c] == bg for r in range(Rr)) and c0 < c < c1]\n    return grows, gcols\n\n\ndef _tilegap_params(trains):\n    mi, mo = set(), set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        Rr, Cc = shape(inp)\n        bg = bg_color(inp)\n        nbg = [(r, c) for r in range(Rr) for c in range(Cc)\n               if inp[r][c] != bg]\n        if not nbg:\n            return None\n        r0 = min(r for r, _ in nbg)\n        r1 = max(r for r, _ in nbg)\n        c0 = min(c for _, c in nbg)\n        c1 = max(c for _, c in nbg)\n        grows, gcols = _tilegap_lines(inp, bg, r0, r1, c0, c1)\n        want = [row[:] for row in inp]\n        for r in grows:\n            for c in range(Cc):\n                if want[r][c] != bg:\n                    return None\n                want[r][c] = \"in\" if c0 <= c <= c1 else \"out\"\n        for c in gcols:\n            for r in range(Rr):\n                w = \"in\" if r0 <= r <= r1 else \"out\"\n                if want[r][c] != bg and want[r][c] != w:\n                    return None\n                want[r][c] = w\n        vi = {out[rr][cc] for rr in range(Rr) for cc in range(Cc)\n              if want[rr][cc] == \"in\"}\n        vo = {out[rr][cc] for rr in range(Rr) for cc in range(Cc)\n              if want[rr][cc] == \"out\"}\n        if not vi or not vo or len(vi) != 1 or len(vo) != 1 or vi == vo:\n            return None\n        pal = {v for row in inp for v in row}\n        if next(iter(vi)) in pal or next(iter(vo)) in pal:\n            return None\n        mi.add(next(iter(vi)))\n        mo.add(next(iter(vo)))\n        for rr in range(Rr):\n            for cc in range(Cc):\n                if want[rr][cc] == \"in\":\n                    want[rr][cc] = next(iter(vi))\n                elif want[rr][cc] == \"out\":\n                    want[rr][cc] = next(iter(vo))\n        if want != out:\n            return None\n    if len(mi) != 1 or len(mo) != 1:\n        return None\n    return next(iter(mi)), next(iter(mo))\n\n\ndef try_tilegap(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    pm = _tilegap_params(trains)\n    if pm is None:\n        return None\n    mi, mo = pm\n    Rr, Cc = shape(test)\n    bg = bg_color(test)\n    pal = {v for row in test for v in row}\n    if mi in pal or mo in pal:\n        return None\n    nbg = [(r, c) for r in range(Rr) for c in range(Cc)\n           if test[r][c] != bg]\n    if not nbg:\n        return None\n    r0 = min(r for r, _ in nbg)\n    r1 = max(r for r, _ in nbg)\n    c0 = min(c for _, c in nbg)\n    c1 = max(c for _, c in nbg)\n    grows, gcols = _tilegap_lines(test, bg, r0, r1, c0, c1)\n    got = [row[:] for row in test]\n    for r in grows:\n        for c in range(Cc):\n            if got[r][c] != bg:\n                return None\n            got[r][c] = mi if c0 <= c <= c1 else mo\n    for c in gcols:\n        for r in range(Rr):\n            w = mi if r0 <= r <= r1 else mo\n            if got[r][c] != bg and got[r][c] != w:\n                return None\n            got[r][c] = w\n    return got\n\n\ndef d_tilegap(trains):\n    if not trains:\n        return False\n    if _tilegap_params(trains) is None:\n        return False\n    return verify_loo(try_tilegap, trains)\n\n# \u2500\u2500\u2500 Dot bars (178fcbfb) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Dots (non-bg singletons). Each color wires to an orientation,\n# LEARNED: row-colors paint their dots' full rows, col-colors\n# their dots' full columns, with the dot's own color. Row bars\n# win crossings (a col bar never overwrites a row bar). Bars paint\n# bg-only otherwise (a dot on another's bar fails closed). Same\n# shape in/out. LOO-verified.\n\ndef _dotbars_orient(trains):\n    rowcolors, colcolors = set(), set()\n    allc = set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        Rr, Cc = shape(inp)\n        bg = bg_color(inp)\n        comps = components(inp, bg=bg)\n        if any(c[\"size\"] != 1 for c in comps):\n            return None\n        for c in comps:\n            allc.add(c[\"color\"])\n            dr, dc = c[\"anchor\"]\n            if all(out[dr][cc] == c[\"color\"] for cc in range(Cc)):\n                rowcolors.add(c[\"color\"])\n    colcolors = allc - rowcolors\n    if not rowcolors and not colcolors:\n        return None\n    return rowcolors, colcolors\n\n\ndef _dotbars_apply(g, rowcolors, colcolors, bg):\n    Rr, Cc = shape(g)\n    comps = components(g, bg=bg)\n    if any(c[\"size\"] != 1 for c in comps):\n        return None\n    if {c[\"color\"] for c in comps} - (rowcolors | colcolors):\n        return None\n    dots = {(r, c) for r in range(Rr) for c in range(Cc) if g[r][c] != bg}\n    got = [row[:] for row in g]\n    for c in comps:\n        if c[\"color\"] not in rowcolors:\n            continue\n        dr, _dc = c[\"anchor\"]\n        for cc in range(Cc):\n            if (dr, cc) in dots:\n                continue\n            if got[dr][cc] != bg and got[dr][cc] != c[\"color\"]:\n                return None\n            got[dr][cc] = c[\"color\"]\n    for c in comps:\n        if c[\"color\"] not in colcolors:\n            continue\n        _dr, dc = c[\"anchor\"]\n        for rr in range(Rr):\n            if (rr, dc) in dots:\n                continue\n            if got[rr][dc] == bg or got[rr][dc] == c[\"color\"]:\n                got[rr][dc] = c[\"color\"]\n                continue\n            # row bars win crossings: keep them, fail otherwise\n            is_row = any(cc2[\"color\"] in rowcolors and\n                         cc2[\"anchor\"][0] == rr for cc2 in comps)\n            if not is_row or got[rr][dc] not in rowcolors:\n                return None\n    return got\n\n\ndef _dotbars_params(trains):\n    o = _dotbars_orient(trains)\n    if o is None:\n        return None\n    rowcolors, colcolors = o\n    for inp, out in trains:\n        got = _dotbars_apply(inp, rowcolors, colcolors, bg_color(inp))\n        if got != out:\n            return None\n    return rowcolors, colcolors\n\n\ndef try_dotbars(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    pm = _dotbars_params(trains)\n    if pm is None:\n        return None\n    return _dotbars_apply(test, pm[0], pm[1], bg_color(test))\n\n\ndef d_dotbars(trains):\n    if not trains:\n        return False\n    if _dotbars_params(trains) is None:\n        return False\n    return verify_loo(try_dotbars, trains)\n\n# \u2500\u2500\u2500 Spine tiling (12422b43) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Exactly one column opens with a top run (length >= 1) of 5s: the\n# spine; its rows are the tile. The tile cycles downward starting\n# at the first row below all content, copying every non-bg cell\n# except the spine column (bg-only strict). Same shape. No marks\n# to learn (verbatim copy). LOO-verified.\n\ndef _spinetile_params(trains):\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        Rr, Cc = shape(inp)\n        bg = bg_color(inp)\n        spines = [c for c in range(Cc) if inp[0][c] == 5]\n        if len(spines) != 1:\n            return None\n        sp = spines[0]\n        K = _runlen(inp, sp)\n        content = [r for r in range(Rr)\n                   if any(v != bg for v in inp[r])]\n        if not content:\n            return None\n        anchor = max(content) + 1\n        if anchor >= Rr:\n            return None\n        painted = [False]\n        if _spinetile_paint(inp, out, bg, sp, K, anchor,\n                            list(range(K)), painted) is None:\n            return None\n        if not painted[0]:\n            return None\n    return True\n\n\ndef _runlen(g, c):\n    n = 0\n    while n < len(g) and g[n][c] == 5:\n        n += 1\n    return n\n\n\ndef _spinetile_paint(inp, out, bg, sp, K, anchor, rows, painted):\n    Rr, Cc = shape(inp)\n    want = [row[:] for row in inp]\n    L = len(rows)\n    for r in range(anchor, Rr):\n        src = rows[(r - anchor) % L]\n        for c in range(Cc):\n            if c == sp:\n                continue\n            v = inp[src][c]\n            if v == bg:\n                continue\n            if want[r][c] != bg:\n                return None\n            want[r][c] = v\n            painted[0] = True\n    return want if want == out else None\n\n\ndef try_spinetile(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    if _spinetile_params(trains) is None:\n        return None\n    Rr, Cc = shape(test)\n    bg = bg_color(test)\n    spines = [c for c in range(Cc) if test[0][c] == 5]\n    if len(spines) != 1:\n        return None\n    sp = spines[0]\n    K = _runlen(test, sp)\n    content = [r for r in range(Rr) if any(v != bg for v in test[r])]\n    if not content:\n        return None\n    anchor = max(content) + 1\n    if anchor >= Rr:\n        return None\n    got = [row[:] for row in test]\n    L = K\n    for r in range(anchor, Rr):\n        src = (r - anchor) % L\n        for c in range(Cc):\n            if c == sp:\n                continue\n            v = test[src][c]\n            if v == bg:\n                continue\n            if got[r][c] != bg:\n                return None\n            got[r][c] = v\n    return got\n\n\ndef d_spinetile(trains):\n    if not trains:\n        return False\n    if _spinetile_params(trains) is None:\n        return False\n    return verify_loo(try_spinetile, trains)\n\n# \u2500\u2500\u2500 Odd block select (shrink-88 find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 2dc579da: full-length uniform divider lines (ANY color, even\n# bg \u2014 test uses dot row+col) partition into EQUAL blocks; output = the\n# UNIQUE block differing from the rest (counts [1, n-1]; 2-block ties\n# and non-unique odds -> None). Strict LOO detector.\n\ndef _oddblock_partition(g):\n    R, C = shape(g)\n    if R == 0 or C == 0:\n        return None\n    hdiv = [r for r in range(R) if len(set(g[r])) == 1]\n    vdiv = [c for c in range(C) if len({g[r][c] for r in range(R)}) == 1]\n    if not hdiv and not vdiv:\n        return None\n    rb = [-1] + hdiv + [R]\n    cb = [-1] + vdiv + [C]\n    hs = [rb[i + 1] - rb[i] - 1 for i in range(len(rb) - 1)]\n    ws = [cb[j + 1] - cb[j] - 1 for j in range(len(cb) - 1)]\n    if any(s <= 0 for s in hs + ws):\n        return None\n    if len(set(hs)) != 1 or len(set(ws)) != 1:\n        return None\n    blocks = []\n    for i in range(len(rb) - 1):\n        for j in range(len(cb) - 1):\n            blocks.append(tuple(tuple(row[cb[j] + 1:cb[j + 1]])\n                                for row in g[rb[i] + 1:rb[i + 1]]))\n    return blocks\n\n\ndef _oddblock_pick(blocks):\n    from collections import Counter as _C\n    if not blocks or len(blocks) < 2:\n        return None\n    cnt = _C(blocks)\n    if len(cnt) != 2:\n        return None\n    top = cnt.most_common()\n    if top[0][1] == 1 and top[1][1] == len(blocks) - 1:\n        return top[0][0]\n    if top[1][1] == 1 and top[0][1] == len(blocks) - 1:\n        return top[1][0]\n    return None\n\n\ndef try_oddblock(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    for inp, out in trains:\n        blocks = _oddblock_partition(inp)\n        if blocks is None:\n            return None\n        odd = _oddblock_pick(blocks)\n        if odd is None or [list(r) for r in odd] != out:\n            return None\n        if (len(odd), len(odd[0])) != shape(out):\n            return None\n    blocks = _oddblock_partition(test)\n    if blocks is None:\n        return None\n    odd = _oddblock_pick(blocks)\n    if odd is None:\n        return None\n    return [list(r) for r in odd]\n\n\ndef d_oddblock(trains):\n    return verify_loo(try_oddblock, trains)\n\n# \u2500\u2500\u2500 Anchor crop (shrink-88 find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in sparse->3x3: output is an exact subgrid; anchor LEARNED per\n# puzzle, exactly ONE mode may reproduce all trains (else None):\n# FIXED (5bd6f4ac: constant position), DENSEST_C (ae4f1146: unique\n# argmax of color-C density), BBOX_C (a87f7484: tight bbox of color C).\n# Strict LOO detector. 48d8fb45 fits none (below-marker max-cover rule\n# noted as future work, NOT implemented).\n\ndef _crop_at(g, r, c, h, w):\n    return [row[c:c + w] for row in g[r:r + h]]\n\n\ndef _all_crop_pos(I, O):\n    Rr, Cc, or_, oc = shape(I)[0], shape(I)[1], shape(O)[0], shape(O)[1]\n    return [(r, c) for r in range(Rr - or_ + 1) for c in range(Cc - oc + 1)\n            if _crop_at(I, r, c, or_, oc) == O]\n\n\ndef _densest_pos(I, h, w, color):\n    Rr, Cc = shape(I)\n    best, arg = -1, []\n    for r in range(Rr - h + 1):\n        for c in range(Cc - w + 1):\n            k = sum(1 for rr in range(r, r + h) for cc in range(c, c + w)\n                    if I[rr][cc] == color)\n            if k > best:\n                best, arg = k, [(r, c)]\n            elif k == best:\n                arg.append((r, c))\n    return arg\n\n\ndef _tight_bbox(I, color):\n    cells = [(r, c) for r, row in enumerate(I) for c, v in enumerate(row)\n             if v == color]\n    if not cells:\n        return None\n    r0 = min(r for r, _c in cells)\n    r1 = max(r for r, _c in cells)\n    c0 = min(c for _r, c in cells)\n    c1 = max(c for _r, c in cells)\n    return (r0, c0, r1 - r0 + 1, c1 - c0 + 1)\n\n\ndef _anchor_params(trains):\n    shapes = {shape(o) for _i, o in trains}\n    if len(shapes) != 1:\n        return None\n    (oh, ow) = next(iter(shapes))\n    if oh <= 0 or ow <= 0:\n        return None\n    modes = []\n    # FIXED: same crop position in every train\n    pos0 = None\n    fixed_ok = True\n    for inp, out in trains:\n        hits = _all_crop_pos(inp, out)\n        if len(hits) != 1:\n            fixed_ok = False\n            break\n        if pos0 is None:\n            pos0 = hits[0]\n        elif hits[0] != pos0:\n            fixed_ok = False\n            break\n    if fixed_ok and pos0 is not None:\n        modes.append((\"FIXED\", pos0))\n    # DENSEST_C / BBOX_C per color\n    colors = sorted({v for inp, _o in trains for row in inp for v in row})\n    for col in colors:\n        dok, bok = True, True\n        for inp, out in trains:\n            if _all_crop_pos(inp, out) != _densest_pos(inp, oh, ow, col) or \\\n                    len(_densest_pos(inp, oh, ow, col)) != 1:\n                dok = False\n            bb = _tight_bbox(inp, col)\n            if bb is None or bb[2:] != (oh, ow) or \\\n                    _crop_at(inp, bb[0], bb[1], oh, ow) != out:\n                bok = False\n            if not dok and not bok:\n                break\n        if dok:\n            modes.append((\"DENSEST\", col))\n        if bok:\n            modes.append((\"BBOX\", col))\n    if len(modes) != 1:\n        return None\n    return modes[0]\n\n\ndef try_anchor_crop(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    learned = _anchor_params(trains)\n    if learned is None:\n        return None\n    (oh, ow) = shape(trains[0][1])\n    mode = learned[0]\n    if mode == \"FIXED\":\n        (r, c) = learned[1]\n        Rr, Cc = shape(test)\n        if r + oh > Rr or c + ow > Cc:\n            return None\n        return _crop_at(test, r, c, oh, ow)\n    col = learned[1]\n    if mode == \"DENSEST\":\n        arg = _densest_pos(test, oh, ow, col)\n        if len(arg) != 1:\n            return None\n        (r, c) = arg[0]\n        return _crop_at(test, r, c, oh, ow)\n    bb = _tight_bbox(test, col)\n    if bb is None or bb[2:] != (oh, ow):\n        return None\n    return _crop_at(test, bb[0], bb[1], oh, ow)\n\n\ndef d_anchorcrop(trains):\n    return verify_loo(try_anchor_crop, trains)\n\n# \u2500\u2500\u2500 Half fold boolean (shrink-88 find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in v-divider->3x3: split by full-height divider (or even split),\n# overlay halves with a LEARNED boolean op (AND: 0520fde7, OR: dae9d2b5,\n# e133d23d), output monochrome learned color. Vertical only (no h-case\n# observed). First (split,op) in deterministic order reproducing ALL\n# trains wins; strict LOO detector.\n\ndef _halfold_splits(g, bg):\n    R, C = shape(g)\n    out = []\n    for c in range(C):\n        col = [g[r][c] for r in range(R)]\n        if len(set(col)) == 1 and col[0] != bg:\n            L, Rr = c, C - c - 1\n            if L == Rr and L > 0:\n                out.append((c,))\n    if C % 2 == 0 and C >= 2:\n        out.append(None)  # even split, no divider\n    return out\n\n\ndef _halfold_apply(g, bg, split, op, outcolor):\n    R, C = shape(g)\n    if split is None:\n        w = C // 2\n        L = [row[:w] for row in g]\n        Rr = [row[w:] for row in g]\n    else:\n        c = split[0]\n        L = [row[:c] for row in g]\n        Rr = [row[c + 1:] for row in g]\n    if len(L[0]) != len(Rr[0]):\n        return None\n    res = []\n    for r in range(R):\n        row = []\n        for i in range(len(L[0])):\n            a = L[r][i] != bg\n            b = Rr[r][i] != bg\n            v = (a and b) if op == \"AND\" else ((a or b) if op == \"OR\"\n                                               else (a != b))\n            row.append(outcolor if v else bg)\n        res.append(row)\n    return res\n\n\ndef _halfold_params(trains):\n    bgcands = [0]\n    for inp, _out in trains:\n        m = bg_color(inp)\n        if m not in bgcands:\n            bgcands.append(m)\n    for bg in bgcands:\n        cands = None\n        ok = True\n        for inp, out in trains:\n            or_, oc = shape(out)\n            good = []\n            for split in _halfold_splits(inp, bg):\n                for op in (\"AND\", \"OR\", \"XOR\"):\n                    for ocol in range(10):\n                        if ocol == bg:\n                            continue\n                        if _halfold_apply(inp, bg, split, op, ocol) == out:\n                            good.append((split, op, ocol))\n            # keep only (split,op) pairs that explain with a single color;\n            # color may vary per train (learned map below)\n            pairs = {(s, o) for (s, o, _c) in good}\n            if not pairs:\n                ok = False\n                break\n            cands = pairs if cands is None else (cands & pairs)\n            if not cands:\n                ok = False\n                break\n        if not ok or cands is None or len(cands) != 1:\n            continue\n        split, op = next(iter(cands))\n        # out-color: consistent across trains? (fresh-map like keydelete if not)\n        cols = set()\n        for inp, out in trains:\n            ocols = {c for row in out for c in row if c != bg}\n            if len(ocols) != 1:\n                cols = set()\n                break\n            cols.add(next(iter(ocols)))\n        if len(cols) != 1:\n            continue\n        return (bg, split, op, next(iter(cols)))\n    return None\n\n\ndef try_halfold(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    learned = _halfold_params(trains)\n    if learned is None:\n        return None\n    bg, split, op, ocol = learned\n    return _halfold_apply(test, bg, split, op, ocol)\n\n\ndef d_halfold(trains):\n    return verify_loo(try_halfold, trains)\n\n# \u2500\u2500\u2500 Keyed delete+recolor (EV-hunt find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 009d5c81 (EV 0.25): one FIXED color D vanishes entirely, one\n# FIXED color T recolors wholesale, new color keyed by the LARGEST\n# deleted component's shape sig (t1==t3 -> 3, t2==t4 -> 2, t0 unique -> 7).\n# Positions/shapes preserved exactly; only colors change. Strict: single\n# D and single T fixed across trains; new color fresh (not in input);\n# coverage-based detector (unique keys break strict LOO, as in sizefill).\n\ndef _keydelete_params(trains):\n    from collections import Counter as _C\n    Dset, Tset, mapping = None, None, {}\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        bg = bg_color(inp)\n        ci = _C(c for row in inp for c in row if c != bg)\n        co = _C(c for row in out for c in row if c != bg)\n        vanished = [c for c in ci if co.get(c, 0) == 0]\n        fresh = {c: n for c, n in co.items() if c not in ci}\n        tmap, dlist = {}, []\n        for c in vanished:\n            matches = [n for n, k in fresh.items() if k == ci[c]]\n            if len(matches) == 1:\n                tmap[c] = matches[0]\n            elif len(matches) == 0:\n                dlist.append(c)\n            else:\n                return None  # ambiguous count-match\n        if len(dlist) != 1 or len(tmap) != 1:\n            return None\n        d = dlist[0]\n        tcol, ncol = next(iter(tmap.items()))\n        Dset = {d} if Dset is None else (Dset & {d})\n        Tset = {tcol} if Tset is None else (Tset & {tcol})\n        # key shape: largest deleted component sig\n        dsigs = sorted([o[\"sig\"] for o in components(inp, bg)\n                        if o[\"color\"] == d],\n                       key=lambda s: -len(s))\n        if not dsigs:\n            return None\n        key = dsigs[0]\n        if key in mapping and mapping[key] != ncol:\n            return None\n        mapping[key] = ncol\n    if not Dset or len(Dset) != 1 or not Tset or len(Tset) != 1:\n        return None\n    return (next(iter(Dset)), next(iter(Tset)), mapping)\n\n\ndef _keydelete_keyshape(g, bg, dcolor):\n    ds = sorted([o[\"sig\"] for o in components(g, bg)\n                 if o[\"color\"] == dcolor], key=lambda s: -len(s))\n    return ds[0] if ds else None\n\n\ndef try_keydelete(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    learned = _keydelete_params(trains)\n    if learned is None:\n        return None\n    dcol, tcol, mapping = learned\n    bg = bg_color(test)\n    key = _keydelete_keyshape(test, bg, dcol)\n    if key is None or key not in mapping:\n        return None\n    newc = mapping[key]\n    R, C = shape(test)\n    out = [row[:] for row in test]\n    for r in range(R):\n        for c in range(C):\n            if out[r][c] == dcol:\n                out[r][c] = bg\n            elif out[r][c] == tcol:\n                out[r][c] = newc\n    return out\n\n\ndef d_keydelete(trains):\n    if not verify(try_keydelete, trains):\n        return False\n    learned = _keydelete_params(trains)\n    if learned is None:\n        return False\n    dcol = learned[0]\n    keys = set()\n    reps = 0\n    for inp, _out in trains:\n        bg = bg_color(inp)\n        k = _keydelete_keyshape(inp, bg, dcol)\n        if k is None:\n            return False\n        reps += 1\n        keys.add(k)\n    return len(keys) >= 2 or reps > len(keys)\n\n# \u2500\u2500\u2500 Row period extend (EV-hunt find) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 963e52fc (EV 0.25 via tiling: right neighborhood, wrong mode).\n# Each row continues its smallest prefix-period to 2x width; heights and\n# dot-rows unchanged. NOT plain doubling (fails), NOT mirror (fails):\n# out[i] = row[i % p]. Strict: same height in/out, out width exactly 2x\n# in every train; horizontal only (no vertical case observed).\n\ndef _row_period(row):\n    n = len(row)\n    for p in range(1, n + 1):\n        if all(row[i] == row[i % p] for i in range(n)):\n            return p\n    return n\n\n\ndef try_row_extend(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    for inp, out in trains:\n        ir, ic = shape(inp)\n        or_, oc = shape(out)\n        if ir == 0 or (or_, oc) != (ir, 2 * ic):\n            return None\n        for r in range(ir):\n            p = _row_period(inp[r])\n            if ([inp[r][c % p] for c in range(oc)] != list(out[r])):\n                return None\n    ir, ic = shape(test)\n    if ir == 0 or ic == 0:\n        return None\n    return [[test[r][c % _row_period(test[r])] for c in range(2 * ic)]\n            for r in range(ir)]\n\n\ndef d_rowextend(trains):\n    return verify_loo(try_row_extend, trains)\n\n# \u2500\u2500\u2500 Progression continue (draw family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 0b17323b: `#` dots on an arithmetic-diagonal step get the line\n# continued FORWARD to the grid edge, drawn in a LEARNED new color (`o`\n# in both trains). Strict: all non-bg cells are single dots of ONE color;\n# constant step (row-sorted diffs); forward-only (backward indistinguishable\n# on observed trains \u2014 conservative draws less); anti-vacuity: must draw\n# >= 1 dot across trains (else identity puzzles false-fire).\n\ndef _progression(dots):\n    if len(dots) < 2:\n        return None\n    pts = sorted(dots)\n    diffs = [(b[0] - a[0], b[1] - a[1])\n             for a, b in zip(pts, pts[1:])]\n    if len(set(diffs)) != 1 or diffs[0] == (0, 0):\n        return None\n    return pts, diffs[0]\n\n\ndef try_prog_continue(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    newcolors = set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        bg = bg_color(inp)\n        dots = [(r, c) for r, row in enumerate(inp)\n                for c, v in enumerate(row) if v != bg]\n        colors = {inp[r][c] for (r, c) in dots}\n        if len(colors) != 1:\n            return None\n        # dots must be isolated singles (no two dots 8-adjacent)\n        dotset = set(dots)\n        for (r, c) in dots:\n            for dr in (-1, 0, 1):\n                for dc in (-1, 0, 1):\n                    if (dr or dc) and (r + dr, c + dc) in dotset:\n                        return None\n        for r in range(len(inp)):\n            for c in range(len(inp[0])):\n                a, b = inp[r][c], out[r][c]\n                if a == bg and b != bg:\n                    newcolors.add(b)\n                elif a != b:\n                    return None  # only additions allowed\n    if len(newcolors) != 1:\n        return None\n    newc = next(iter(newcolors))\n\n    def apply(inp):\n        bg = bg_color(inp)\n        dots = [(r, c) for r, row in enumerate(inp)\n                for c, v in enumerate(row) if v != bg]\n        if not dots:\n            return None\n        if len({inp[r][c] for (r, c) in dots}) != 1:\n            return None\n        pr = _progression(dots)\n        if pr is None:\n            return None\n        pts, (dr, dc) = pr\n        R, C = shape(inp)\n        out = [row[:] for row in inp]\n        r, c = pts[-1][0] + dr, pts[-1][1] + dc\n        while 0 <= r < R and 0 <= c < C:\n            if out[r][c] != bg:\n                return None  # path blocked: not this dynamic\n            out[r][c] = newc\n            r, c = r + dr, c + dc\n        return out\n\n    for inp, out in trains:\n        pred = apply(inp)\n        if pred is None or not v3.grids_equal(pred, out):\n            return None\n    ndrawn = 0\n    for inp, out in trains:\n        ndrawn += sum(1 for r in range(len(inp)) for c in range(len(inp[0]))\n                      if inp[r][c] == bg_color(inp) and out[r][c] != bg_color(inp))\n    if ndrawn == 0:\n        return None\n    return apply(test)\n\n\ndef d_progcontinue(trains):\n    return verify_loo(try_prog_continue, trains)\n\n# \u2500\u2500\u2500 Frame dots (draw family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in dc1df850: every `o` dot gets a 3x3 hollow `#` square centered\n# on it (clipped at borders, center preserved). Other colors ignored.\n# Generalized as a learned map: dot_color -> frame_color, 3x3 hollow.\n# Conservative: frames paint bg cells only (no colliding example seen;\n# overwriting content would be a guess, and guesses don't ship).\n\ndef _frame_map(trains: Trains, bg):\n    dots, frames = set(), set()\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        for r in range(len(inp)):\n            for c in range(len(inp[0])):\n                a, b = inp[r][c], out[r][c]\n                if a == bg and b != bg:\n                    frames.add(b)\n                elif a != bg and b != bg and a != b:\n                    return None  # recolor: not this dynamic\n        # dot candidates: unchanged non-bg cells fully surrounded by\n        # non-bg in the OUTPUT (a frame center; edge-clipped frames count\n        # in-bounds neighbors only). Mere adjacency (e.g. * diagonal to a\n        # frame in dc1df850/train1) must NOT qualify: surroundedness does.\n        R, C = shape(inp)\n        for r in range(R):\n            for c in range(C):\n                if inp[r][c] == bg or out[r][c] != inp[r][c]:\n                    continue\n                nbrs = [(r + dr, c + dc) for dr in (-1, 0, 1)\n                        for dc in (-1, 0, 1) if (dr or dc)\n                        and 0 <= r + dr < R and 0 <= c + dc < C]\n                if nbrs and all(out[nr][nc] != bg for (nr, nc) in nbrs):\n                    dots.add(inp[r][c])\n    if len(frames) != 1 or len(dots) != 1:\n        return None\n    return (next(iter(dots)), next(iter(frames)))\n\n\ndef try_frame_dots(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    bg = bg_color(trains[0][0])\n    m = _frame_map(trains, bg)\n    if m is None:\n        return None\n    dot, frame = m\n    R, C = shape(test)\n    out = [row[:] for row in test]\n    for r in range(R):\n        for c in range(C):\n            if test[r][c] != dot:\n                continue\n            for dr in (-1, 0, 1):\n                for dc in (-1, 0, 1):\n                    if dr == 0 and dc == 0:\n                        continue\n                    nr, nc = r + dr, c + dc\n                    if (0 <= nr < R and 0 <= nc < C and\n                            out[nr][nc] == bg):\n                        out[nr][nc] = frame\n    return out\n\n\ndef d_framedots(trains):\n    return verify_loo(try_frame_dots, trains)\n\n# \u2500\u2500\u2500 Size-keyed fill (draw family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 00dbd492: hollow rectangles get interiors filled; fill COLOR\n# is keyed by interior SIZE ((7,7)->x, (5,5)->O, (3,3)->8, learned from\n# trains); pre-existing interior dots are preserved (only bg cells fill).\n\ndef _enclosed_interiors(g: Grid, bg: int):\n    R, C = shape(g)\n    reached = [[False] * C for _ in range(R)]\n    stack = ([(r, 0) for r in range(R)] + [(r, C - 1) for r in range(R)] +\n             [(0, c) for c in range(C)] + [(R - 1, c) for c in range(C)])\n    while stack:\n        r, c = stack.pop()\n        if not (0 <= r < R and 0 <= c < C):\n            continue\n        if reached[r][c] or g[r][c] != bg:\n            continue\n        reached[r][c] = True\n        stack.extend([(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)])\n    seen = set()\n    regions = []\n    for r in range(R):\n        for c in range(C):\n            if g[r][c] != bg or reached[r][c] or (r, c) in seen:\n                continue\n            cells, st = [], [(r, c)]\n            while st:\n                cr, cc = st.pop()\n                if ((cr, cc) in seen or not (0 <= cr < R and 0 <= cc < C)\n                        or g[cr][cc] != bg or reached[cr][cc]):\n                    continue\n                seen.add((cr, cc))\n                cells.append((cr, cc))\n                st.extend([(cr - 1, cc), (cr + 1, cc),\n                           (cr, cc - 1), (cr, cc + 1)])\n            if cells:\n                regions.append(cells)\n    return regions\n\n\ndef _sizefill_map(trains: Trains, bg):\n    \"\"\"(h, w) -> fill color, learned. None if inconsistent/empty.\"\"\"\n    mapping = {}\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        for cells in _enclosed_interiors(inp, bg):\n            r0 = min(r for r, c in cells)\n            r1 = max(r for r, c in cells)\n            c0 = min(c for r, c in cells)\n            c1 = max(c for r, c in cells)\n            key = (r1 - r0 + 1, c1 - c0 + 1)\n            fills = Counter(out[r][c] for (r, c) in cells\n                            if inp[r][c] == bg)\n            if not fills:\n                return None\n            color, n = fills.most_common(1)[0]\n            if n != len([1 for (r, c) in cells if inp[r][c] == bg]):\n                return None  # interior bg did not uniformly fill\n            if color == bg:\n                return None\n            if key in mapping and mapping[key] != color:\n                return None\n            mapping[key] = color\n    return mapping or None\n\n\ndef try_sizefill(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    bg = bg_color(trains[0][0])\n    mapping = _sizefill_map(trains, bg)\n    if not mapping:\n        return None\n    out = [row[:] for row in test]\n    for cells in _enclosed_interiors(test, bg):\n        r0 = min(r for r, c in cells)\n        r1 = max(r for r, c in cells)\n        c0 = min(c for r, c in cells)\n        c1 = max(c for r, c in cells)\n        key = (r1 - r0 + 1, c1 - c0 + 1)\n        if key not in mapping:\n            return None\n        for (r, c) in cells:\n            if out[r][c] == bg:\n                out[r][c] = mapping[key]\n    return out\n\n\ndef d_sizefill(trains):\n    # NOT verify_loo: map-learning needs coverage, not redundancy.\n    # LOO demands each pair's sizes appear in OTHER pairs; sparse maps\n    # (00dbd492: (7,7) only in t0) fail LOO on a TRUE rule. Instead:\n    # plain consistency + evidence bar (distinct keys >= 2, or a key\n    # repeated across trains). Memorizers can't pass: output is fully\n    # determined by the small key->color map, not by copying a train.\n    if not verify(try_sizefill, trains):\n        return False\n    bg = bg_color(trains[0][0])\n    keys = []\n    for inp, _out in trains:\n        for cells in _enclosed_interiors(inp, bg):\n            r0 = min(r for r, c in cells)\n            r1 = max(r for r, c in cells)\n            c0 = min(c for r, c in cells)\n            c1 = max(c for r, c in cells)\n            keys.append((r1 - r0 + 1, c1 - c0 + 1))\n    return len(set(keys)) >= 2 or len(keys) > len(set(keys))\n\n# \u2500\u2500\u2500 Staircase pack (stack family, H4) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 03560426. H1 (column gravity), H2 (delete empty rows/cols) and\n# H3 (compact+pad) all REFUTED computationally on its 3 trains.\n# True rule (fit on 10 object motions, 3 trains): order objects by\n# (min_col, min_row); obj0 anchor -> (0,0); then staircase:\n#   top[i+1]  = top[i] + height[i] - 1\n#   left[i+1] = left[i] + width[i] - 1\n# (FULL input-shape bbox dims), paint sequentially, later overwrites\n# (overlap cells resolve to the later object: the \"lost\" cells).\n\ndef try_staircase_pack(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    bg = bg_color(trains[0][0])\n    R, C = shape(test)\n    tcomps = components(test, bg)\n    if not tcomps:\n        return None\n    tcomps.sort(key=lambda o: (o[\"anchor\"][1], o[\"anchor\"][0]))\n    out = [[bg] * C for _ in range(R)]\n    top, left = 0, 0\n    first = True\n    for comp in tcomps:\n        (ar, ac) = comp[\"anchor\"]\n        cells = [(r - ar + top, c - ac + left) for (r, c) in comp[\"cells\"]]\n        if any(not (0 <= r < R and 0 <= c < C) for (r, c) in cells):\n            return None\n        for (r, c) in cells:\n            out[r][c] = comp[\"color\"]\n        _r0, _r1, _c0, _c1 = _bbox(comp[\"cells\"])\n        h, w = _r1 - _r0 + 1, _c1 - _c0 + 1\n        top, left = top + h - 1, left + w - 1\n        first = False\n    return out\n\n\ndef d_staircase(trains):\n    return verify_loo(try_staircase_pack, trains)\n\n# \u2500\u2500\u2500 Crop + quadrant extraction (extract family) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Finding 2026-09-09: \"extract\" (output\u2282input, 52 puzzles) is a FAMILY,\n# not one dynamic. bbox-crop holds on trains for 1/52 (refuted as the\n# cluster rule). Consistent-quadrant holds for 2/52. Both implemented as\n# LOO-gated micro-rules: zero regression risk, honest p@1 accounting.\n\ndef _content_box(g, bg):\n    cells = [(r, c) for r, row in enumerate(g)\n             for c, v in enumerate(row) if v != bg]\n    if not cells:\n        return None\n    r0 = min(r for r, c in cells)\n    r1 = max(r for r, c in cells)\n    c0 = min(c for r, c in cells)\n    c1 = max(c for r, c in cells)\n    return (r0, r1, c0, c1)\n\n\ndef try_crop_content(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    bg = bg_color(trains[0][0])\n    for inp, out in trains:\n        box = _content_box(inp, bg)\n        if box is None:\n            return None\n        r0, r1, c0, c1 = box\n        crop = [row[c0:c1 + 1] for row in inp[r0:r1 + 1]]\n        if not v3.grids_equal(crop, out):\n            return None\n    box = _content_box(test, bg)\n    if box is None:\n        return None\n    r0, r1, c0, c1 = box\n    return [row[c0:c1 + 1] for row in test[r0:r1 + 1]]\n\n\ndef _quadrants(box):\n    H, W = len(box), len(box[0])\n    if H % 2 or W % 2 or H < 2 or W < 2:\n        return None\n    mh, mw = H // 2, W // 2\n    return {\"TL\": [r[:mw] for r in box[:mh]],\n            \"TR\": [r[mw:] for r in box[:mh]],\n            \"BL\": [r[:mw] for r in box[mh:]],\n            \"BR\": [r[mw:] for r in box[mh:]]}\n\n\ndef try_extract_quadrant(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    bg = bg_color(trains[0][0])\n    for q in (\"TL\", \"TR\", \"BL\", \"BR\"):\n        ok = True\n        for inp, out in trains:\n            box = _content_box(inp, bg)\n            if box is None:\n                ok = False\n                break\n            r0, r1, c0, c1 = box\n            Q = _quadrants([row[c0:c1 + 1] for row in inp[r0:r1 + 1]])\n            if Q is None or not v3.grids_equal(Q[q], out):\n                ok = False\n                break\n        if ok:\n            box = _content_box(test, bg)\n            if box is None:\n                return None\n            r0, r1, c0, c1 = box\n            Q = _quadrants([row[c0:c1 + 1] for row in test[r0:r1 + 1]])\n            return Q[q] if Q is not None else None\n    return None\n\n\ndef d_crop(trains):\n    return verify_loo(try_crop_content, trains)\n\n\ndef d_quadrant(trains):\n    return verify_loo(try_extract_quadrant, trains)\n\n# \u2500\u2500\u2500 Flip-tiling (extends rsi_module.try_adaptive_tiling) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Seen in 00576224: 2x2 \u2192 6x6 where odd TILE ROWS are mirrored horizontally.\n# rsi adaptive covers vertical-flip-on-odd-row / horizontal-flip-on-odd-col\n# but not horizontal-flip-on-odd-row. Full mode matrix below.\n\ndef _tile_transform(tile, flip_h, flip_v):\n    t = [row[:] for row in tile]\n    if flip_h:\n        t = [row[::-1] for row in t]\n    if flip_v:\n        t = t[::-1]\n    return t\n\n\nFLIP_MODES = {\n    \"repeat\":    lambda ti, tj: (False, False),\n    \"altrow_h\":  lambda ti, tj: (ti % 2 == 1, False),\n    \"altrow_v\":  lambda ti, tj: (False, ti % 2 == 1),\n    \"altcol_h\":  lambda ti, tj: (tj % 2 == 1, False),\n    \"altcol_v\":  lambda ti, tj: (False, tj % 2 == 1),\n    \"checker_h\": lambda ti, tj: ((ti + tj) % 2 == 1, False),\n    \"checker_v\": lambda ti, tj: (False, (ti + tj) % 2 == 1),\n}\n\n\ndef _check_flip_mode(inp, out, mode, tr, tc):\n    ir, ic = shape(inp)\n    fn = FLIP_MODES[mode]\n    for ti in range(tr):\n        for tj in range(tc):\n            fh, fv = fn(ti, tj)\n            exp = _tile_transform(inp, fh, fv)\n            for di in range(ir):\n                for dj in range(ic):\n                    if out[ti * ir + di][tj * ic + dj] != exp[di][dj]:\n                        return False\n    return True\n\n\ndef try_flip_tiling(trains: Trains, test: Grid) -> Optional[Grid]:\n    if not trains:\n        return None\n    i0, o0 = trains[0]\n    ir, ic = shape(i0)\n    or_, oc = shape(o0)\n    if ir == 0 or ic == 0 or or_ % ir != 0 or oc % ic != 0:\n        return None\n    tr, tc = or_ // ir, oc // ic\n    if tr < 1 or tc < 1 or (tr == 1 and tc == 1):\n        return None\n    for mode in FLIP_MODES:\n        if all(shape(i) == (ir, ic) and shape(o) == (or_, oc)\n               and _check_flip_mode(i, o, mode, tr, tc) for i, o in trains):\n            R, C = shape(test)\n            if R * tr == 0 or C * tc == 0:\n                return None\n            fn = FLIP_MODES[mode]\n            res = [[0] * (C * tc) for _ in range(R * tr)]\n            for ti in range(tr):\n                for tj in range(tc):\n                    fh, fv = fn(ti, tj)\n                    exp = _tile_transform(test, fh, fv)\n                    for di in range(R):\n                        for dj in range(C):\n                            res[ti * R + di][tj * C + dj] = exp[di][dj]\n            return res\n    return None\n\n# \u2500\u2500\u2500 New strategy: move object to anchor \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Dynamic seen in 05f2a901: one object (mover) slides along one axis until\n# 4-adjacent to a static object (anchor). Axis varies per pair, so the rule\n# is relational (toward-anchor), not a fixed vector (cf. try_object_shift).\n\ndef _cells_adjacent(a_cells, b_cells) -> bool:\n    b = set(b_cells)\n    for (r, c) in a_cells:\n        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n            if (r + dr, c + dc) in b:\n                return True\n    return False\n\n\ndef _bbox(cells):\n    rs = [x[0] for x in cells]\n    cs = [x[1] for x in cells]\n    return (min(rs), max(rs), min(cs), max(cs))\n\n\ndef identify_anchor_mover(trains: Trains, bg) -> Tuple[Optional[int], Optional[int]]:\n    \"\"\"(anchor_color, mover_color): anchor static in every train, mover keeps\n    shape but changes position and ends 4-adjacent to the anchor.\"\"\"\n    colors = None\n    for inp, _out in trains:\n        cs = {c for row in inp for c in row if c != bg}\n        colors = cs if colors is None else (colors & cs)\n    if not colors:\n        return None, None\n\n    def cellset(g, color):\n        return {(r, c) for r, row in enumerate(g)\n                for c, v in enumerate(row) if v == color}\n\n    def sigs(g, color):\n        return sorted(o[\"sig\"] for o in components(g, bg)\n                      if o[\"color\"] == color)\n\n    anchor_cand, mover_cand = [], []\n    for color in colors:\n        static = all(cellset(i, color) == cellset(o, color)\n                     for i, o in trains)\n        moved = all(sigs(i, color) == sigs(o, color) and\n                    cellset(i, color) != cellset(o, color)\n                    for i, o in trains)\n        if static:\n            anchor_cand.append(color)\n        if moved:\n            mover_cand.append(color)\n    if not anchor_cand or not mover_cand:\n        return None, None\n\n    # disambiguate by adjacency in every output\n    ok_anchors = [a for a in anchor_cand\n                  if all(any(_cells_adjacent(cellset(o, m), cellset(o, a))\n                             for m in mover_cand)\n                         for _, o in trains)]\n    ok_movers = [m for m in mover_cand\n                 if all(any(_cells_adjacent(cellset(o, m), cellset(o, a))\n                            for a in anchor_cand)\n                        for _, o in trains)]\n    if len(ok_anchors) != 1 or len(ok_movers) != 1:\n        return None, None\n    return ok_anchors[0], ok_movers[0]\n\n\ndef _slide_to_anchor(mover_cells, anchor_cells):\n    \"\"\"Shift mover per-axis (bbox) until 4-adjacent. None if impossible.\"\"\"\n    mr0, mr1, mc0, mc1 = _bbox(mover_cells)\n    ar0, ar1, ac0, ac1 = _bbox(anchor_cells)\n    anchor_set = set(anchor_cells)\n    dr = dc = 0\n    while mr1 < ar0 - 1:\n        dr += 1\n        mr0 += 1\n        mr1 += 1\n        if dr > 64:\n            return None\n    while mr0 > ar1 + 1:\n        dr -= 1\n        mr0 -= 1\n        mr1 -= 1\n        if dr < -64:\n            return None\n    while mc1 < ac0 - 1:\n        dc += 1\n        mc0 += 1\n        mc1 += 1\n        if dc > 64:\n            return None\n    while mc0 > ac1 + 1:\n        dc -= 1\n        mc0 -= 1\n        mc1 -= 1\n        if dc < -64:\n            return None\n    moved = [(r + dr, c + dc) for (r, c) in mover_cells]\n    if set(moved) & anchor_set:\n        return None  # overlap, not adjacency\n    if not _cells_adjacent(moved, anchor_cells):\n        return None\n    return moved\n\n\ndef try_move_to_anchor(trains: Trains, test: Grid) -> Optional[Grid]:\n    bg = bg_color(trains[0][0])\n    anchor_c, mover_c = identify_anchor_mover(trains, bg)\n    if anchor_c is None:\n        return None\n    R, C = shape(test)\n    anchor_cells = [(r, c) for r, row in enumerate(test)\n                    for c, v in enumerate(row) if v == anchor_c]\n    mover_comps = [o for o in components(test, bg) if o[\"color\"] == mover_c]\n    if not anchor_cells or not mover_comps:\n        return None\n    out = [row[:] for row in test]\n    for comp in mover_comps:\n        for (r, c) in comp[\"cells\"]:\n            out[r][c] = bg\n    for comp in mover_comps:\n        moved = _slide_to_anchor(comp[\"cells\"], anchor_cells)\n        if moved is None:\n            return None\n        for (r, c) in moved:\n            if not (0 <= r < R and 0 <= c < C) or out[r][c] != bg:\n                return None\n            out[r][c] = mover_c\n    return out\n\n\ndef d_anchor(trains):\n    bg = bg_color(trains[0][0])\n    a, m = identify_anchor_mover(trains, bg)\n    if a is None:\n        return False\n    return verify_loo(try_move_to_anchor, trains)\n\n# \u2500\u2500\u2500 Composition (hunter-grown toolbox, data not code) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# A verified composition {\"ops\": [f, g]} becomes a strategy WITHOUT code\n# edits: compose(f, g)(trains, x) = g(trains, f(trains, x)), shared trains\n# context, None-propagating. Registry = state/rsi_reports/compositions.json\n# (hunter writes it). Loaded only via --compose PATH (opt-in: default runs\n# stay exactly comparable). Detector = whichever entry LOO-verifies.\n\ndef compose(f, g):\n    def _composed(trains, test):\n        try:\n            mid = f(trains, test)\n        except Exception:\n            return None\n        if mid is None:\n            return None\n        try:\n            return g(trains, mid)\n        except Exception:\n            return None\n    _composed.__name__ = f\"{f.__name__}+{g.__name__}\"\n    return _composed\n\n\nCOMPOSED_FNS = []\n\n\ndef load_compositions(path):\n    import json as _json\n    try:\n        reg = _json.load(open(path))\n    except Exception:\n        return []\n    by_name = {name: fn for name, fn in FALLBACK}\n    out = []\n    for e in reg.get(\"compositions\", []):\n        ops = e.get(\"ops\", [])\n        if len(ops) != 2 or not all(o in by_name for o in ops):\n            continue\n        fn = compose(by_name[ops[0]], by_name[ops[1]])\n        out.append((f\"composed_{ops[0]}+{ops[1]}\", fn))\n    return out\n\n\ndef d_composed(trains):\n    for _name, fn in COMPOSED_FNS:\n        if verify_loo(fn, trains):\n            return True\n    return False\n\n\ndef composed_fns_for(trains):\n    return [fn for _name, fn in COMPOSED_FNS\n            if verify_loo(fn, trains)]\n\n# \u2500\u2500\u2500 Dynamic detectors (ordered, most specific first) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef d_extract(trains):\n    return all(contains_subgrid(i, o) for i, o in trains)\n\n\ndef d_retiling(trains):\n    return (verify_loo(v3.try_tiling, trains)\n            or verify_loo(v3.try_diagonal_tiling, trains)\n            or verify_loo(try_adaptive_tiling, trains)\n            or verify_loo(try_flip_tiling, trains))\n\n\ndef d_rotation(trains):\n    return verify_loo(v3.try_rotation, trains) or verify_loo(v3.try_reflection, trains)\n\n\ndef d_recolor(trains):\n    if any(shape(i) != shape(o) for i, o in trains):\n        return False\n    return (verify_loo(v3.try_color_mapping, trains)\n            or verify_loo(v2.try_position_color_map, trains))\n\n\ndef d_fill(trains):\n    if any(shape(i) != shape(o) for i, o in trains):\n        return False\n    return (verify_loo(v3.try_interior_fill, trains)\n            or verify_loo(v2.try_pattern_fill, trains)\n            or verify_loo(v2.try_flood_fill, trains))\n\n\ndef d_counting(trains):\n    if not all(shape(o)[0] * shape(o)[1] <= 25 for _, o in trains):\n        return False\n    return (verify_loo(v3.try_object_count, trains)\n            or verify_loo(v2.try_color_counting, trains))\n\n\ndef d_move(trains):\n    return verify_loo(try_object_shift, trains)\n\n\ndef d_expand(trains):\n    if not any(shape(o) != shape(i) for i, o in trains):\n        return False\n    return verify_loo(v3.try_shape_expansion, trains)\n\n\ndef d_spread(trains):\n    # nearest_neighbor deliberately excluded: it memorizes train outputs\n    # (returns the most similar pair's output) instead of spreading.\n    return verify_loo(v3.try_color_spread, trains)\n\n# dynamic -> (detector, strategy fns to try in order, ARC-AGI-3 action family)\nDYNAMICS = [\n    (\"composed\", d_composed, None,  # fns resolved per-puzzle (registry)\n     \"composed\"),\n    # crop/quadrant BEFORE extract: their outputs are also subgrids,\n    # but these two carry solvers while bare extract does not.\n    (\"crop\",     d_crop,     [try_crop_content],\n     \"explore-cursor\"),\n    (\"quadrant\", d_quadrant, [try_extract_quadrant],\n     \"explore-cursor\"),\n    (\"anchorcrop\", d_anchorcrop, [try_anchor_crop],\n     \"cursor-move(ACTION1/2)\"),\n    (\"oddblock\", d_oddblock, [try_oddblock],\n     \"explore-cursor\"),\n    (\"blockcensus\", d_blockcensus, [try_blockcensus],\n     \"explore-cursor\"),\n    (\"norhalves\", d_norhalves, [try_norhalves],  # solver before extract\n     \"explore-cursor\"),\n    (\"xorhalves\", d_xorhalves, [try_xorhalves],  # solver before extract\n     \"explore-cursor\"),\n    (\"orhalves\", d_orhalves, [try_orhalves],  # solver before extract\n     \"explore-cursor\"),\n    (\"pack\", d_pack, [try_pack],  # solver before extract\n     \"explore-cursor\"),\n    (\"anomaly\", d_anomaly, [try_anomaly],  # solver before extract\n     \"explore-cursor\"),\n    (\"qcollage\", d_qcollage, [try_qcollage],  # solver before extract\n     \"explore-cursor\"),\n    (\"qstack\", d_qstack, [try_qstack],  # solver before extract\n     \"explore-cursor\"),\n    (\"qstack2\", d_qstack2, [try_qstack2],  # solver before extract\n     \"explore-cursor\"),\n    (\"qstackL\", d_qstackL, [try_qstackL],  # solver before extract\n     \"explore-cursor\"),\n    (\"vstack\", d_vstack, [try_vstack],  # solver before extract\n     \"explore-cursor\"),\n    (\"vunion\", d_vunion, [try_vunion],  # solver before extract\n     \"explore-cursor\"),\n    (\"vfirst\", d_vfirst, [try_vfirst],  # solver before extract\n     \"explore-cursor\"),\n    (\"vrows\", d_vrows, [try_vrows],  # solver before extract\n     \"explore-cursor\"),\n    (\"vcols\", d_vcols, [try_vcols],  # solver before extract\n     \"explore-cursor\"),\n    (\"frames\", d_frames, [try_frames],  # solver before extract\n     \"explore-cursor\"),\n    (\"dotframe\", d_dotframe, [try_dotframe],  # solver before extract\n     \"explore-cursor\"),\n    (\"below\", d_below, [try_below],  # solver before extract\n     \"explore-cursor\"),\n    (\"midline\", d_midline, [try_midline],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"asym\", d_asym, [try_asym],  # solver before extract\n     \"explore-cursor\"),\n    (\"divemit\", d_divemit, [try_divemit],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"hub4\", d_hub4, [try_hub4],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"halo\", d_halo, [try_halo],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"diagfocus\", d_diagfocus, [try_diagfocus],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"cross1\", d_cross1, [try_cross1],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"sqframe\", d_sqframe, [try_sqframe],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"tilegap\", d_tilegap, [try_tilegap],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"dotbars\", d_dotbars, [try_dotbars],  # solver before extract\n     \"interact-paint(ACTION5)\"),\n    (\"spinetile\", d_spinetile, [try_spinetile],  # solver before extract\n     \"explore-cursor\"),\n    (\"extract\",  d_extract,  [],  # no solver without output \u2192 fallback\n     \"explore-cursor\"),\n    (\"retiling\", d_retiling, [v3.try_tiling, v3.try_diagonal_tiling,\n                              try_adaptive_tiling, try_flip_tiling],\n     \"transform-bulk(ACTION3/4)\"),\n    (\"mirrordouble\", d_mirrordouble, [try_mirror_double],\n     \"transform-bulk(ACTION3/4)\"),\n    (\"rowextend\", d_rowextend, [try_row_extend],\n     \"transform-bulk(ACTION3/4)\"),\n    (\"halfold\", d_halfold, [try_halfold],\n     \"interact-paint(ACTION5)\"),\n    (\"rotation\", d_rotation, [v3.try_rotation, v3.try_reflection],\n     \"transform-bulk(ACTION3/4)\"),\n    (\"recolor\",  d_recolor,  [v3.try_color_mapping, v2.try_position_color_map],\n     \"interact-paint(ACTION5)\"),\n    (\"fill\",     d_fill,     [v3.try_interior_fill, v2.try_pattern_fill,\n                              v2.try_flood_fill],\n     \"interact-paint(ACTION5)\"),\n    (\"sizefill\", d_sizefill, [try_sizefill],\n     \"interact-paint(ACTION5)\"),\n    (\"framedots\", d_framedots, [try_frame_dots],\n     \"interact-paint(ACTION5)\"),\n    (\"symcomplete\", d_symcomplete, [try_symmetry_complete],\n     \"interact-paint(ACTION5)\"),\n    (\"growring\", d_growring, [try_grow_ring],\n     \"interact-paint(ACTION5)\"),\n    (\"progcont\", d_progcontinue, [try_prog_continue],\n     \"interact-paint(ACTION5)\"),\n    (\"keydelete\", d_keydelete, [try_keydelete],\n     \"interact-paint(ACTION5)\"),\n    (\"invertrecolor\", d_invertrecolor, [try_invert_recolor],\n     \"interact-paint(ACTION5)\"),\n    (\"connect\", d_connect, [try_connect_dots],\n     \"interact-paint(ACTION5)\"),\n    (\"counting\", d_counting, [v3.try_object_count, v2.try_color_counting],\n     \"explore-cursor\"),\n    (\"anchor\",   d_anchor,   [try_move_to_anchor],\n     \"move-cursor(ACTION1-4)\"),\n    (\"staircase\", d_staircase, [try_staircase_pack],\n     \"move-cursor(ACTION1-4)\"),\n    (\"legendswap\", d_legendswap, [try_legend_swap],\n     \"move-cursor(ACTION1-4)\"),\n    (\"bandmatch\", d_bandmatch, [try_band_match],\n     \"move-cursor(ACTION1-4)\"),\n    # lattice UNPARKED 2026-09-10: step now from bg grid lines (not\n    # gcd-of-gaps); 06df4c85 all 3 trains reproduce (multi-texture was\n    # one mechanism: interpolate on the bg lattice, bg 1/8/4 per train).\n    (\"lattice\", d_lattice, [try_lattice_interp],\n     \"move-cursor(ACTION1-4)\"),\n    (\"move\",     d_move,     [try_object_shift],\n     \"move-cursor(ACTION1-4)\"),\n    (\"expand\",   d_expand,   [v3.try_shape_expansion],\n     \"transform-bulk(ACTION3/4)\"),\n    (\"spread\",   d_spread,   [v3.try_color_spread, v3.try_nearest_neighbor],\n     \"interact-paint(ACTION5)\"),\n]\n\n# Full blind chain = v3 order + missing toolbox fns (dedup). Baseline superset.\nFALLBACK: List[Tuple[str, Callable]] = list(v3.STRATEGIES)\n_seen = {id(f) for _, f in FALLBACK}\nfor _name, _fn in [(\"object_count\", v3.try_object_count),\n                   (\"adaptive_tiling\", try_adaptive_tiling),\n                   (\"flip_tiling\", try_flip_tiling),\n                   (\"flood_fill\", v2.try_flood_fill),\n                   (\"position_color_map\", v2.try_position_color_map),\n                   (\"color_counting\", v2.try_color_counting),\n                   (\"pattern_fill\", v2.try_pattern_fill),\n                   (\"object_shift\", try_object_shift),\n                   (\"move_to_anchor\", try_move_to_anchor),\n                   (\"crop_content\", try_crop_content),\n                   (\"extract_quadrant\", try_extract_quadrant),\n                   (\"frame_dots\", try_frame_dots),\n                   (\"half_fold\", try_halfold),\n                   (\"anchor_crop\", try_anchor_crop),\n                   (\"odd_block\", try_oddblock),\n                   (\"block_census\", try_blockcensus),\n                   (\"row_extend\", try_row_extend),\n                   (\"mirror_double\", try_mirror_double),\n                   (\"symmetry_complete\", try_symmetry_complete),\n                   (\"grow_ring\", try_grow_ring),\n                   (\"prog_continue\", try_prog_continue),\n                   (\"legend_swap\", try_legend_swap),\n                   (\"band_match\", try_band_match),\n                   (\"lattice_interp\", try_lattice_interp),\n                   (\"connect_dots\", try_connect_dots),\n                   (\"sizefill\", try_sizefill),\n                   (\"staircase_pack\", try_staircase_pack),\n                   (\"nor_halves\", try_norhalves),\n                   (\"xor_halves\", try_xorhalves),\n                   (\"or_halves\", try_orhalves),\n                   (\"pack_pieces\", try_pack),\n                   (\"anomaly_crop\", try_anomaly),\n                   (\"quad_collage\", try_qcollage),\n                   (\"quad_stack\", try_qstack),\n                   (\"quad_stack2\", try_qstack2),\n                   (\"quad_stackL\", try_qstackL),\n                   (\"vert_stack\", try_vstack),\n                   (\"vert_union\", try_vunion),\n                   (\"vert_first\", try_vfirst),\n                   (\"vert_rows\", try_vrows),\n                   (\"vert_cols\", try_vcols),\n                   (\"frames_overlay\", try_frames),\n                   (\"dot_frame\", try_dotframe),\n                    (\"below_marker\", try_below),\n                    (\"mid_line\", try_midline),\n                    (\"asym_block\", try_asym),\n                    (\"div_emit\", try_divemit),\n                    (\"hub_star\", try_hub4),\n                    (\"halo_dots\", try_halo),\n                    (\"diag_focus\", try_diagfocus),\n                    (\"cross_ones\", try_cross1),\n                    (\"square_frame\", try_sqframe),\n                    (\"tile_gaps\", try_tilegap),\n                    (\"dot_bars\", try_dotbars),\n                    (\"spine_tile\", try_spinetile)]:\n    if id(_fn) not in _seen:\n        FALLBACK.append((_name, _fn))\n        _seen.add(id(_fn))\n\n\ndef classify(trains: Trains) -> Tuple[str, Dict[str, Any]]:\n    \"\"\"First dynamic whose detector verifies on all train pairs.\"\"\"\n    evidence: Dict[str, Any] = {\"n_train\": len(trains)}\n    for name, det, _fns, _arc3 in DYNAMICS:\n        try:\n            hit = det(trains)\n        except Exception as e:\n            hit = False\n            evidence[f\"{name}_error\"] = str(e)[:60]\n        if hit:\n            return name, evidence\n    return \"unknown\", evidence\n\n\ndef solve_routed(trains: Trains, test: Grid, use_fallback=True):\n    \"\"\"Apply predicted strategy only; fallback to blind chain on miss.\"\"\"\n    dynamic, evidence = classify(trains)\n    tried = []\n    for name, _det, fns, _arc3 in DYNAMICS:\n        if name != dynamic:\n            continue\n        if fns is None:  # composed: resolve verifying entries per puzzle\n            fns = composed_fns_for(trains)\n        for fn in fns:\n            tried.append(fn.__name__)\n            pred = safe_apply(fn, trains, test)\n            if pred is not None:\n                return {\"pred\": pred, \"dynamic\": dynamic,\n                        \"tried\": tried, \"fallback\": False,\n                        \"evidence\": evidence}\n    if use_fallback:\n        for fname, fn in FALLBACK:\n            if fn.__name__ in tried:\n                continue\n            tried.append(fn.__name__)\n            pred = safe_apply(fn, trains, test)\n            if pred is not None:\n                return {\"pred\": pred, \"dynamic\": dynamic,\n                        \"tried\": tried, \"fallback\": True,\n                        \"evidence\": evidence}\n    return {\"pred\": None, \"dynamic\": dynamic, \"tried\": tried,\n            \"fallback\": True, \"evidence\": evidence}\n\n\ndef oracle_solvers(trains: Trains, test: Grid, expected: Grid) -> List[str]:\n    \"\"\"Which toolbox strategies solve this puzzle (upper bound)?\"\"\"\n    winners = []\n    for fname, fn in FALLBACK:\n        pred = safe_apply(fn, trains, test)\n        if pred is not None and v3.grids_equal(pred, expected):\n            winners.append(fname)\n    return winners\n\n# \u2500\u2500\u2500 Cluster reporter (Fase 3: automate the orient step) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Groups UNKNOWN puzzles by a COARSE structural signature so the next\n# hunting target emerges without eyeballing. Buckets are deliberately\n# coarse: exact counts gave all-singletons (session 2026-09-09).\n\ndef _symmetric(g: Grid) -> bool:\n    R, C = shape(g)\n    h = all(g[r] == g[R - 1 - r] for r in range(R // 2))\n    v = all(g[r][c] == g[r][C - 1 - c]\n            for r in range(R) for c in range(C // 2))\n    return h or v\n\n\ndef coarse_signature(puzzle: Dict) -> tuple:\n    trains = v2.get_train_pairs(puzzle)\n    if not trains:\n        return (\"empty\",)\n    same_shape = all(shape(i) == shape(o) for i, o in trains)\n    if same_shape:\n        shp = \"same_shape\"\n    elif all(shape(o)[0] % shape(i)[0] == 0 and shape(o)[1] % shape(i)[1] == 0\n             and shape(i)[0] and shape(i)[1] for i, o in trains):\n        shp = \"int_scale\"\n    else:\n        shp = \"reshape\"\n    in_area = sum(shape(i)[0] * shape(i)[1] for i, _ in trains)\n    out_area = sum(shape(o)[0] * shape(o)[1] for _, o in trains)\n    ratio = out_area / in_area if in_area else 0\n    size = (\"tiny\" if ratio <= 0.05 else \"smaller\" if ratio < 0.9\n            else \"same\" if ratio <= 1.1 else \"bigger\")\n    ci = sum(len({c for row in i for c in row}) for i, _ in trains)\n    co = sum(len({c for row in o for c in row}) for _, o in trains)\n    col = (\"same_colors\" if ci == co else\n           \"more_colors\" if co > ci else \"fewer_colors\")\n    ni = sum(len(components(i)) for i, _ in trains)\n    no = sum(len(components(o)) for _, o in trains)\n    obj = (\"same_objs\" if ni == no else\n           \"fewer_objs\" if no < ni else \"more_objs\")\n    mi = sum(sum(1 for v in row if v != bg_color(i))\n             for i, _ in trains for row in i)\n    mo = sum(sum(1 for v in row if v != bg_color(o))\n             for _, o in trains for row in o)\n    mass = (\"mass_same\" if mi == mo else\n            \"mass_lost\" if mo < mi else \"mass_gained\")\n    sym = (\"sym_out\" if all(_symmetric(o) for _, o in trains)\n           else \"asym_out\")\n    return (shp, size, col, obj, mass, sym)\n\n\ndef cluster_unknowns(rows: List[Dict], puzzles: Dict, top: int = 5):\n    \"\"\"Rank unknown clusters by size. Returns [(key, count, [ids])].\"\"\"\n    groups = defaultdict(list)\n    for r in rows:\n        if r[\"dynamic\"] != \"unknown\":\n            continue\n        p = puzzles.get(r[\"id\"])\n        if p is None:\n            continue\n        try:\n            groups[coarse_signature(p)].append(r[\"id\"])\n        except Exception:\n            groups[(\"error\",)].append(r[\"id\"])\n    ranked = sorted(groups.items(), key=lambda kv: -len(kv[1]))\n    return [(k, len(v), v[:3]) for k, v in ranked[:top]]\n\n# \u2500\u2500\u2500 Machine-maintained taxonomy \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# New categories register THEMSELVES here: every dynamic that solves is\n# catalogued with precision, examples and ARC-AGI-3 family, no hand docs.\n# Human writes strategies; the system writes the catalog. Origin ladder:\n# \"toolbox\" (hand code) -> \"composed\" (hunter registry) -> proposal.\n\ndef build_taxonomy(rep, out_path=None):\n    dyn_meta = {}\n    for name, _det, fns, arc3 in DYNAMICS:\n        dyn_meta[name] = {\n            \"strategies\": [fn.__name__ if fn is not None else None\n                           for fn in (fns or [])],\n            \"arc3\": arc3,\n        }\n    by_dyn = defaultdict(list)\n    for r in rep[\"rows\"]:\n        by_dyn[r[\"dynamic\"]].append(r)\n    dynamics = []\n    for name in sorted(by_dyn):\n        rows = by_dyn[name]\n        seen, ex = set(), []\n        for r in [x for x in rows if x[\"solved\"]] + rows:\n            if r[\"id\"] not in seen:\n                seen.add(r[\"id\"])\n                ex.append(r[\"id\"])\n            if len(ex) >= 3:\n                break\n        fns = dyn_meta.get(name, {}).get(\"strategies\", [])\n        origin = (\"composed\" if any((f or \"\").startswith(\"composed_\")\n                                    for f in fns) else \"toolbox\")\n        dynamics.append({\n            \"name\": name,\n            \"n\": len(rows),\n            \"p1\": sum(1 for r in rows if r.get(\"p1\")),\n            \"solved\": sum(1 for r in rows if r[\"solved\"]),\n            \"strategies\": fns,\n            \"arc3\": dyn_meta.get(name, {}).get(\"arc3\", \"?\"),\n            \"examples\": ex,\n            \"origin\": origin,\n        })\n    tax = {\n        \"updated\": datetime.now(timezone.utc).isoformat(),\n        \"totals\": {k: rep[k] for k in\n                   (\"n\", \"solved\", \"accuracy\", \"precision_at_1\",\n                    \"oracle_solved\", \"oracle_acc\", \"avg_strategies_tried\")\n                   if k in rep},\n        \"dynamics\": dynamics,\n        \"clusters\": rep.get(\"clusters\", []),\n    }\n    if out_path:\n        os.makedirs(os.path.dirname(out_path), exist_ok=True)\n        with open(out_path, \"w\") as f:\n            json.dump(tax, f, indent=1)\n    return tax\n\n# \u2500\u2500\u2500 Evaluation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef evaluate(data_dir: str, max_puzzles: int = 1000, oracle: bool = True):\n    puzzles = v2.load_puzzles(data_dir)\n    ids = sorted(puzzles.keys())[:max_puzzles]\n    rows = []\n    dyn_stats = defaultdict(lambda: {\"n\": 0, \"p1\": 0, \"solved\": 0})\n    for tid in ids:\n        p = puzzles[tid]\n        trains = v2.get_train_pairs(p)\n        if not p.get(\"test\"):\n            continue\n        test_in = v2.get_test_input(p)\n        expected = p[\"test\"][0][\"output\"]\n        r = solve_routed(trains, test_in, use_fallback=True)\n        ok = r[\"pred\"] is not None and v3.grids_equal(r[\"pred\"], expected)\n        p1 = (not r[\"fallback\"] and r[\"pred\"] is not None\n              and v3.grids_equal(r[\"pred\"], expected))\n        wins = oracle_solvers(trains, test_in, expected) if oracle else []\n        st = dyn_stats[r[\"dynamic\"]]\n        st[\"n\"] += 1\n        st[\"p1\"] += int(p1)\n        st[\"solved\"] += int(ok)\n        rows.append({\"id\": tid, \"dynamic\": r[\"dynamic\"],\n                     \"tried\": len(r[\"tried\"]), \"solved\": ok, \"p1\": p1,\n                     \"fallback\": r[\"fallback\"], \"oracle\": wins})\n    n = len(rows)\n    solved = sum(1 for x in rows if x[\"solved\"])\n    p1 = sum(1 for x in rows if x[\"p1\"])\n    orac = sum(1 for x in rows if x[\"oracle\"])\n    avg_tried = sum(x[\"tried\"] for x in rows) / n if n else 0\n    return {\n        \"n\": n, \"solved\": solved, \"accuracy\": solved / n if n else 0,\n        \"precision_at_1\": p1 / n if n else 0,\n        \"oracle_solved\": orac, \"oracle_acc\": orac / n if n else 0,\n        \"avg_strategies_tried\": round(avg_tried, 2),\n        \"blind_chain_len\": len(FALLBACK),\n        \"per_dynamic\": {k: {\"n\": v[\"n\"], \"p1\": v[\"p1\"],\n                            \"solved\": v[\"solved\"]} for k, v in dyn_stats.items()},\n        \"rows\": rows,\n    }\n\n\ndef main():\n    import argparse\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--data\", default=\"/tmp/opencode/arc-training/training\")\n    ap.add_argument(\"--max\", type=int, default=1000)\n    ap.add_argument(\"--no-oracle\", action=\"store_true\")\n    ap.add_argument(\"--out\", default=None)\n    ap.add_argument(\"--compose\", default=None,\n                    help=\"compositions registry JSON (hunter-grown toolbox)\")\n    ap.add_argument(\"--taxonomize\", default=None, metavar=\"OUT\",\n                    help=\"write machine-maintained taxonomy JSON and exit \"\n                         \"(also saved: no solve run needed, uses --max run)\")\n    args = ap.parse_args()\n\n    global COMPOSED_FNS\n    if args.compose:\n        COMPOSED_FNS = load_compositions(args.compose)\n        have = {id(f) for _, f in FALLBACK}\n        for name, fn in COMPOSED_FNS:\n            if id(fn) not in have:\n                FALLBACK.append((name, fn))\n                have.add(id(fn))\n        print(f\"composed strategies loaded: {len(COMPOSED_FNS)}\")\n\n    rep = evaluate(args.data, args.max, oracle=not args.no_oracle)\n    print(f\"n={rep['n']} solved={rep['solved']} \"\n          f\"acc={rep['accuracy']:.1%} p@1={rep['precision_at_1']:.1%} \"\n          f\"oracle={rep['oracle_solved']} ({rep['oracle_acc']:.1%}) \"\n          f\"avg_tried={rep['avg_strategies_tried']}/{rep['blind_chain_len']}\")\n    print(\"per-dynamic (n / p1 / solved):\")\n    for k in sorted(rep[\"per_dynamic\"]):\n        v = rep[\"per_dynamic\"][k]\n        print(f\"  {k:10s} n={v['n']:4d} p1={v['p1']:3d} solved={v['solved']:3d}\")\n\n    puzzles = v2.load_puzzles(args.data)\n    clusters = cluster_unknowns(rep[\"rows\"], puzzles)\n    rep[\"clusters\"] = [{\"key\": list(k), \"n\": n, \"examples\": e}\n                       for k, n, e in clusters]\n    print(\"top unknown clusters (key / n / examples):\")\n    for k, n, e in clusters:\n        print(f\"  n={n:4d} {'|'.join(k)}\")\n        print(f\"           e.g. {e}\")\n\n    out = args.out or str(Path(__file__).parent / \"state\" / \"rsi_reports\"\n                          / f\"recognizer_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.json\")\n    os.makedirs(os.path.dirname(out), exist_ok=True)\n    with open(out, \"w\") as f:\n        json.dump(rep, f, indent=1)\n    print(f\"full report: {out}\")\n    if args.taxonomize:\n        tax = build_taxonomy(rep, args.taxonomize)\n        print(f\"taxonomy: {args.taxonomize} \"\n              f\"({len(tax['dynamics'])} dynamics, \"\n              f\"{sum(1 for d in tax['dynamics'] if d['solved'])} solving)\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "solver_smart_v2.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"\nARC-AGI-2 Smart Solver v2 \u2014 VSF Agent\nMulti-strategy solver with comprehensive pattern detection.\n\"\"\"\nimport json\nimport os\nimport random\nimport copy\nimport math\nfrom pathlib import Path\nfrom typing import List, Tuple, Optional, Dict, Any, Callable\nfrom collections import Counter, defaultdict\n\n# \u2500\u2500\u2500 Data Loading \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_puzzles(data_dir: str) -> Dict[str, Any]:\n    puzzles = {}\n    data_path = Path(data_dir)\n    for f in sorted(data_path.glob(\"*.json\")):\n        task_id = f.stem\n        with open(f) as fp:\n            puzzles[task_id] = json.load(fp)\n    return puzzles\n\ndef get_train_pairs(puzzle: Dict) -> List[Tuple]:\n    return [(p[\"input\"], p[\"output\"]) for p in puzzle.get(\"train\", [])]\n\ndef get_test_input(puzzle: Dict) -> List[List[int]]:\n    tests = puzzle.get(\"test\", [])\n    return tests[0][\"input\"] if tests else []\n\n# \u2500\u2500\u2500 Grid Utilities \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef grid_shape(g):\n    return (len(g), len(g[0]) if g else 0) if g else (0, 0)\n\ndef grids_equal(a, b):\n    if len(a) != len(b): return False\n    for r in range(len(a)):\n        if len(a[r]) != len(b[r]): return False\n        for c in range(len(a[r])):\n            if a[r][c] != b[r][c]: return False\n    return True\n\ndef rotate_90(g):\n    R, C = grid_shape(g)\n    return [[g[R-1-c][r] for c in range(R)] for r in range(C)] if R and C else g\n\ndef rotate_180(g):\n    R, C = grid_shape(g)\n    return [[g[R-1-r][C-1-c] for c in range(C)] for r in range(R)]\n\ndef rotate_270(g):\n    R, C = grid_shape(g)\n    return [[g[c][C-1-r] for c in range(R)] for r in range(C)] if R and C else g\n\ndef reflect_h(g):\n    return [row[::-1] for row in g]\n\ndef reflect_v(g):\n    return g[::-1]\n\ndef reflect_d1(g):  # transpose\n    R, C = grid_shape(g)\n    return [[g[r][c] for r in range(R)] for c in range(C)] if R and C else g\n\ndef reflect_d2(g):  # anti-transpose\n    R, C = grid_shape(g)\n    return [[g[C-1-c][R-1-r] for c in range(C)] for r in range(R)] if R and C else g\n\ndef flood_fill(grid, sr, sc, nc):\n    R, C = grid_shape(grid)\n    result = [row[:] for row in grid]\n    oc = result[sr][sc]\n    if oc == nc: return result\n    stack = [(sr, sc)]\n    visited = set()\n    while stack:\n        r, c = stack.pop()\n        if (r,c) in visited or r<0 or r>=R or c<0 or c>=C or result[r][c]!=oc: continue\n        visited.add((r,c))\n        result[r][c] = nc\n        stack.extend([(r-1,c),(r+1,c),(r,c-1),(r,c+1)])\n    return result\n\n# \u2500\u2500\u2500 Strategy 1: Color Mapping \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_color_mapping(train_pairs, test_input):\n    \"\"\"Consistent color mapping across all pairs.\"\"\"\n    maps = []\n    for inp, out in train_pairs:\n        cm = {}\n        ok = True\n        for r in range(min(len(inp), len(out))):\n            for c in range(min(len(inp[r]), len(out[r]))):\n                k, v = inp[r][c], out[r][c]\n                if k in cm:\n                    if cm[k] != v: ok = False; break\n                else: cm[k] = v\n            if not ok: break\n        if not ok: return None\n        maps.append(cm)\n    \n    if not maps: return None\n    base = maps[0]\n    for cm in maps[1:]:\n        for k, v in cm.items():\n            if k in base and base[k] != v: return None\n    \n    result = [row[:] for row in test_input]\n    for r in range(len(result)):\n        for c in range(len(result[r])):\n            if result[r][c] in base:\n                result[r][c] = base[result[r][c]]\n    return result\n\n# \u2500\u2500\u2500 Strategy 2: Scaling/Tiling \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_tiling(train_pairs, test_input):\n    \"\"\"Output is a tiled/scaled version of input.\"\"\"\n    for inp, out in train_pairs:\n        ir, ic = grid_shape(inp)\n        or_, oc = grid_shape(out)\n        if ir == 0 or ic == 0: continue\n        if or_ % ir != 0 or oc % ic != 0: continue\n        tr, tc = or_ // ir, oc // ic\n        if tr < 2 and tc < 2: continue\n        \n        # Try simple tiling\n        ok = True\n        for r in range(or_):\n            for c in range(oc):\n                if out[r][c] != inp[r % ir][c % ic]:\n                    ok = False; break\n            if not ok: break\n        if ok:\n            tr, tc = or_ // ir, oc // ic\n            result = [[0]*(len(test_input[0])*tc) for _ in range(len(test_input)*tr)]\n            for r in range(len(result)):\n                for c in range(len(result[0])):\n                    result[r][c] = test_input[r % len(test_input)][c % len(test_input[0])]\n            return result\n        \n        # Try alternating tiling (flip rows/cols)\n        for flip_r in [False, True]:\n            for flip_c in [False, True]:\n                ok = True\n                for r in range(or_):\n                    for c in range(oc):\n                        sr = r % ir\n                        sc = c % ic\n                        tr_idx = r // ir\n                        tc_idx = c // ic\n                        if flip_r and tr_idx % 2 == 1: sr = ir - 1 - sr\n                        if flip_c and tc_idx % 2 == 1: sc = ic - 1 - sc\n                        if out[r][c] != inp[sr][sc]:\n                            ok = False; break\n                    if not ok: break\n                if ok:\n                    tr, tc = or_ // ir, oc // ic\n                    result = [[0]*(len(test_input[0])*tc) for _ in range(len(test_input)*tr)]\n                    for r in range(len(result)):\n                        for c in range(len(result[0])):\n                            sr = r % len(test_input)\n                            sc = c % len(test_input[0])\n                            tr_idx = r // len(test_input)\n                            tc_idx = c // len(test_input[0])\n                            if flip_r and tr_idx % 2 == 1: sr = len(test_input) - 1 - sr\n                            if flip_c and tc_idx % 2 == 1: sc = len(test_input[0]) - 1 - sc\n                            result[r][c] = test_input[sr][sc]\n                    return result\n    \n    return None\n\n# \u2500\u2500\u2500 Strategy 3: Rotation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_rotation(train_pairs, test_input):\n    for inp, out in train_pairs:\n        if grids_equal(rotate_90(inp), out):\n            return rotate_90(test_input)\n        if grids_equal(rotate_180(inp), out):\n            return rotate_180(test_input)\n        if grids_equal(rotate_270(inp), out):\n            return rotate_270(test_input)\n    return None\n\n# \u2500\u2500\u2500 Strategy 4: Reflection \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_reflection(train_pairs, test_input):\n    for inp, out in train_pairs:\n        if grids_equal(reflect_h(inp), out):\n            return reflect_h(test_input)\n        if grids_equal(reflect_v(inp), out):\n            return reflect_v(test_input)\n        if grids_equal(reflect_d1(inp), out):\n            return reflect_d1(test_input)\n        if grids_equal(reflect_d2(inp), out):\n            return reflect_d2(test_input)\n    return None\n\n# \u2500\u2500\u2500 Strategy 5: Flood Fill \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_flood_fill(train_pairs, test_input):\n    for inp, out in train_pairs:\n        ir, ic = grid_shape(inp)\n        or_, oc = grid_shape(out)\n        if (ir, ic) != (or_, oc): continue\n        changes = []\n        for r in range(ir):\n            for c in range(ic):\n                if inp[r][c] != out[r][c]:\n                    changes.append((r, c, inp[r][c], out[r][c]))\n        if len(changes) == 1:\n            r, c, _, new = changes[0]\n            filled = flood_fill(inp, r, c, new)\n            if grids_equal(filled, out):\n                return flood_fill(test_input, r, c, new)\n    return None\n\n# \u2500\u2500\u2500 Strategy 6: Position-based Color Map \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_position_color_map(train_pairs, test_input):\n    \"\"\"Color mapping that depends on position.\"\"\"\n    if len(train_pairs) < 2: return None\n    \n    # Check if each train pair has a consistent per-position color map\n    # but the maps differ between pairs\n    all_maps = []\n    for inp, out in train_pairs:\n        R, C = grid_shape(inp)\n        oR, oC = grid_shape(out)\n        if R != oR or C != oC: return None\n        pm = {}\n        for r in range(R):\n            for c in range(C):\n                pm[(r,c)] = (inp[r][c], out[r][c])\n        all_maps.append(pm)\n    \n    # Check if the mapping pattern is consistent (same input\u2192output for same position)\n    # but different positions have different mappings\n    # This is just a per-position map from first pair\n    pm = all_maps[0]\n    result = [row[:] for row in test_input]\n    R, C = grid_shape(test_input)\n    for r in range(R):\n        for c in range(C):\n            if (r,c) in pm:\n                inp_val, out_val = pm[(r,c)]\n                if test_input[r][c] == inp_val:\n                    result[r][c] = out_val\n    return result\n\n# \u2500\u2500\u2500 Strategy 7: Diagonal Tiling \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_diagonal_tiling(train_pairs, test_input):\n    \"\"\"Input placed along diagonal of output, zeros elsewhere.\"\"\"\n    for inp, out in train_pairs:\n        ir, ic = grid_shape(inp)\n        or_, oc = grid_shape(out)\n        if or_ % ir != 0 or oc % ic != 0: continue\n        tr, tc = or_ // ir, oc // ic\n        \n        ok = True\n        for r in range(or_):\n            for c in range(oc):\n                tile_r, tile_c = r // ir, c // ic\n                src_r, src_c = r % ir, c % ic\n                if tile_r == tile_c:\n                    if out[r][c] != inp[src_r][src_c]:\n                        ok = False; break\n                else:\n                    if out[r][c] != 0:\n                        ok = False; break\n            if not ok: break\n        \n        if ok:\n            tr, tc = or_ // ir, oc // ic\n            result = [[0]*(len(test_input[0])*tc) for _ in range(len(test_input)*tr)]\n            for tile_idx in range(min(tr, tc)):\n                r0 = tile_idx * len(test_input)\n                c0 = tile_idx * len(test_input[0])\n                for r in range(len(test_input)):\n                    for c in range(len(test_input[0])):\n                        result[r0+r][c0+c] = test_input[r][c]\n            return result\n    \n    return None\n\n# \u2500\u2500\u2500 Strategy 8: Color Counting \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_color_counting(train_pairs, test_input):\n    \"\"\"Output has same colors but different counts.\"\"\"\n    for inp, out in train_pairs:\n        in_colors = Counter(c for row in inp for c in row)\n        out_colors = Counter(c for row in out for c in row)\n        if in_colors == out_colors:\n            # Same multiset of colors, just rearranged\n            # Use nearest neighbor approach\n            best_sim = -1\n            best_out = None\n            for inp2, out2 in train_pairs:\n                sim = sum(1 for r in range(min(len(inp2), len(test_input)))\n                         for c in range(min(len(inp2[0]) if inp2 else 0, len(test_input[0]) if test_input else 0))\n                         if inp2[r][c] == test_input[r][c])\n                if sim > best_sim:\n                    best_sim = sim\n                    best_out = out2\n            if best_out:\n                R, C = grid_shape(test_input)\n                result = [[0]*C for _ in range(R)]\n                for r in range(min(R, len(best_out))):\n                    for c in range(min(C, len(best_out[r]))):\n                        result[r][c] = best_out[r][c]\n                return result\n    return None\n\n# \u2500\u2500\u2500 Strategy 9: Pattern Fill \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_pattern_fill(train_pairs, test_input):\n    \"\"\"Output uses input as a pattern but fills with different colors.\"\"\"\n    for inp, out in train_pairs:\n        ir, ic = grid_shape(inp)\n        or_, oc = grid_shape(out)\n        if ir == 0 or ic == 0: continue\n        \n        # Check if shapes match\n        if ir == or_ and ic == oc:\n            # Check if there's a consistent color substitution per position\n            subs = {}\n            ok = True\n            for r in range(ir):\n                for c in range(ic):\n                    k = inp[r][c]\n                    v = out[r][c]\n                    if k not in subs:\n                        subs[k] = v\n                    elif subs[k] != v:\n                        ok = False; break\n                if not ok: break\n            \n            if ok and len(subs) > 0:\n                result = [row[:] for row in test_input]\n                for r in range(len(result)):\n                    for c in range(len(result[r])):\n                        if result[r][c] in subs:\n                            result[r][c] = subs[result[r][c]]\n                return result\n    \n    return None\n\n# \u2500\u2500\u2500 Strategy 10: Nearest Neighbor \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_nearest_neighbor(train_pairs, test_input):\n    \"\"\"Find most similar train input, use its output as template.\"\"\"\n    if not train_pairs: return None\n    \n    best_sim = -1\n    best_out = None\n    for inp, out in train_pairs:\n        sim = sum(1 for r in range(min(len(inp), len(test_input)))\n                 for c in range(min(len(inp[0]) if inp else 0, len(test_input[0]) if test_input else 0))\n                 if inp[r][c] == test_input[r][c])\n        if sim > best_sim:\n            best_sim = sim\n            best_out = out\n    \n    if best_out:\n        R, C = grid_shape(test_input)\n        result = [[0]*C for _ in range(R)]\n        for r in range(min(R, len(best_out))):\n            for c in range(min(C, len(best_out[r]))):\n                result[r][c] = best_out[r][c]\n        return result\n    return None\n\n# \u2500\u2500\u2500 Master Solver \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nSTRATEGIES = [\n    (\"color_mapping\", try_color_mapping),\n    (\"tiling\", try_tiling),\n    (\"rotation\", try_rotation),\n    (\"reflection\", try_reflection),\n    (\"flood_fill\", try_flood_fill),\n    (\"position_color_map\", try_position_color_map),\n    (\"diagonal_tiling\", try_diagonal_tiling),\n    (\"color_counting\", try_color_counting),\n    (\"pattern_fill\", try_pattern_fill),\n    (\"nearest_neighbor\", try_nearest_neighbor),\n]\n\ndef solver_smart_v2(train_pairs, test_input):\n    \"\"\"Try each strategy in order, return first success.\"\"\"\n    for name, fn in STRATEGIES:\n        try:\n            result = fn(train_pairs, test_input)\n            if result is not None:\n                return result\n        except:\n            continue\n    return [row[:] for row in test_input]\n\n# \u2500\u2500\u2500 Evaluation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef evaluate_solver(solver_fn, puzzles, max_puzzles=200):\n    correct = 0\n    total = 0\n    strategy_hits = defaultdict(int)\n    task_ids = list(puzzles.keys())[:max_puzzles]\n    \n    for task_id in task_ids:\n        puzzle = puzzles[task_id]\n        train_pairs = get_train_pairs(puzzle)\n        test_input = get_test_input(puzzle)\n        if not test_input: continue\n        \n        expected = puzzle[\"test\"][0][\"output\"]\n        \n        try:\n            # Find which strategy worked\n            solved = False\n            for name, fn in STRATEGIES:\n                try:\n                    result = fn(train_pairs, test_input)\n                    if result is not None and grids_equal(result, expected):\n                        correct += 1\n                        strategy_hits[name] += 1\n                        solved = True\n                        break\n                except:\n                    continue\n            \n            if not solved:\n                # Try the master solver\n                result = solver_fn(train_pairs, test_input)\n                if grids_equal(result, expected):\n                    correct += 1\n        except Exception as e:\n            pass\n        \n        total += 1\n    \n    accuracy = correct / total if total > 0 else 0\n    \n    return {\n        \"solver\": solver_fn.__name__,\n        \"correct\": correct,\n        \"total\": total,\n        \"accuracy\": accuracy,\n        \"strategy_hits\": dict(strategy_hits),\n    }\n\n# \u2500\u2500\u2500 Main \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nif __name__ == \"__main__\":\n    data_dir = \"/root/arc-data/training\"\n    print(f\"Loading puzzles from {data_dir}...\")\n    puzzles = load_puzzles(data_dir)\n    print(f\"Loaded {len(puzzles)} puzzles\")\n    \n    print(\"\\n=== Smart Solver v2 Evaluation ===\\n\")\n    result = evaluate_solver(solver_smart_v2, puzzles, max_puzzles=200)\n    print(f\"Total: {result['correct']}/{result['total']} | accuracy: {result['accuracy']:.1%}\")\n    print(f\"\\nStrategy breakdown:\")\n    for strat, count in sorted(result['strategy_hits'].items(), key=lambda x: -x[1]):\n        print(f\"  {strat:25s}: {count}\")\n    \n    # Also run on full 1000\n    print(\"\\n=== Full 1000 Puzzles ===\\n\")\n    result_full = evaluate_solver(solver_smart_v2, puzzles, max_puzzles=1000)\n    print(f\"Total: {result_full['correct']}/{result_full['total']} | accuracy: {result_full['accuracy']:.1%}\")\n    print(f\"\\nStrategy breakdown:\")\n    for strat, count in sorted(result_full['strategy_hits'].items(), key=lambda x: -x[1]):\n        print(f\"  {strat:25s}: {count}\")\n", "solver_v3.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"\nARC-AGI-2 Solver v3 \u2014 RSI-enhanced with interior fill and more strategies\n\"\"\"\nimport json\nimport os\nimport math\nfrom pathlib import Path\nfrom typing import List, Tuple, Dict, Any, Optional\nfrom collections import Counter, defaultdict\n\n# \u2500\u2500\u2500 Data Loading \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_puzzles(data_dir):\n    puzzles = {}\n    for f in sorted(Path(data_dir).glob(\"*.json\")):\n        with open(f) as fp:\n            puzzles[f.stem] = json.load(fp)\n    return puzzles\n\ndef get_train_pairs(puzzle):\n    return [(p[\"input\"], p[\"output\"]) for p in puzzle.get(\"train\", [])]\n\ndef get_test_input(puzzle):\n    tests = puzzle.get(\"test\", [])\n    return tests[0][\"input\"] if tests else []\n\n# \u2500\u2500\u2500 Grid Utilities \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef grids_equal(a, b):\n    if len(a) != len(b): return False\n    for r in range(len(a)):\n        if len(a[r]) != len(b[r]): return False\n        for c in range(len(a[r])):\n            if a[r][c] != b[r][c]: return False\n    return True\n\ndef rotate_90(g):\n    R, C = len(g), len(g[0]) if g else 0\n    return [[g[R-1-c][r] for c in range(R)] for r in range(C)] if R and C else g\n\ndef rotate_180(g):\n    R, C = len(g), len(g[0]) if g else 0\n    return [[g[R-1-r][C-1-c] for c in range(C)] for r in range(R)]\n\ndef rotate_270(g):\n    R, C = len(g), len(g[0]) if g else 0\n    return [[g[c][C-1-r] for c in range(R)] for r in range(C)] if R and C else g\n\ndef reflect_h(g):\n    return [row[::-1] for row in g]\n\ndef reflect_v(g):\n    return g[::-1]\n\ndef flood_fill(grid, sr, sc, nc):\n    R, C = len(grid), len(grid[0]) if grid else 0\n    result = [row[:] for row in grid]\n    oc = result[sr][sc]\n    if oc == nc: return result\n    stack = [(sr, sc)]\n    visited = set()\n    while stack:\n        r, c = stack.pop()\n        if (r,c) in visited or r<0 or r>=R or c<0 or c>=C or result[r][c]!=oc: continue\n        visited.add((r,c))\n        result[r][c] = nc\n        stack.extend([(r-1,c),(r+1,c),(r,c-1),(r,c+1)])\n    return result\n\n# \u2500\u2500\u2500 Strategy 1: Color Mapping \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_color_mapping(train_pairs, test_input):\n    cm = None\n    for inp, out in train_pairs:\n        pair_cm = {}\n        ok = True\n        for r in range(min(len(inp), len(out))):\n            for c in range(min(len(inp[r]), len(out[r]))):\n                k, v = inp[r][c], out[r][c]\n                if k in pair_cm:\n                    if pair_cm[k] != v: ok = False; break\n                else: pair_cm[k] = v\n            if not ok: break\n        if not ok: return None\n        if cm is None: cm = pair_cm\n        else:\n            for k, v in pair_cm.items():\n                if k in cm and cm[k] != v: return None\n    if not cm: return None\n    result = [row[:] for row in test_input]\n    for r in range(len(result)):\n        for c in range(len(result[r])):\n            if result[r][c] in cm: result[r][c] = cm[result[r][c]]\n    return result\n\n# \u2500\u2500\u2500 Strategy 2: Tiling \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_tiling(train_pairs, test_input):\n    for inp, out in train_pairs:\n        ir, ic = len(inp), len(inp[0]) if inp else 0\n        or_, oc = len(out), len(out[0]) if out else 0\n        if ir == 0 or ic == 0: continue\n        if or_ % ir != 0 or oc % ic != 0: continue\n        tr, tc = or_ // ir, oc // ic\n        if tr < 2 and tc < 2: continue\n        ok = all(out[r][c] == inp[r % ir][c % ic] for r in range(or_) for c in range(oc))\n        if ok:\n            R, C = len(test_input), len(test_input[0]) if test_input else 0\n            return [[test_input[r % R][c % C] for c in range(C * tc)] for r in range(R * tr)]\n    return None\n\n# \u2500\u2500\u2500 Strategy 3: Rotation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_rotation(train_pairs, test_input):\n    for inp, out in train_pairs:\n        if inp and out:\n            if rotate_90(inp) == out: return rotate_90(test_input)\n            if rotate_180(inp) == out: return rotate_180(test_input)\n            if rotate_270(inp) == out: return rotate_270(test_input)\n    return None\n\n# \u2500\u2500\u2500 Strategy 4: Reflection \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_reflection(train_pairs, test_input):\n    for inp, out in train_pairs:\n        if inp and out:\n            if reflect_h(inp) == out: return reflect_h(test_input)\n            if reflect_v(inp) == out: return reflect_v(test_input)\n    return None\n\n# \u2500\u2500\u2500 Strategy 5: Interior Fill (NEW) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_interior_fill(train_pairs, test_input):\n    \"\"\"Fill enclosed 0-regions with a fill color.\"\"\"\n    for inp, out in train_pairs:\n        R, C = len(inp), len(inp[0]) if inp else 0\n        oR, oC = len(out), len(out[0]) if out else 0\n        if (R, C) != (oR, oC): continue\n        \n        # Find fill color: color that appears in output but not in input\n        in_colors = set(c for row in inp for c in row)\n        out_colors = set(c for row in out for c in row)\n        fill_colors = out_colors - in_colors\n        \n        if not fill_colors: continue\n        fill_color = list(fill_colors)[0]\n        \n        # Find which 0-pixels were filled\n        filled_pixels = set()\n        for r in range(R):\n            for c in range(C):\n                if inp[r][c] == 0 and out[r][c] == fill_color:\n                    filled_pixels.add((r, c))\n        \n        if not filled_pixels: continue\n        \n        # Verify: filled pixels should be \"interior\" (surrounded by non-zero)\n        # Use flood fill from edges to find exterior\n        exterior = set()\n        # Flood fill from all edge 0-pixels\n        for r in range(R):\n            for c in [0, C-1]:\n                if inp[r][c] == 0:\n                    stack = [(r, c)]\n                    while stack:\n                        cr, cc = stack.pop()\n                        if (cr, cc) in exterior: continue\n                        if cr < 0 or cr >= R or cc < 0 or cc >= C: continue\n                        if inp[cr][cc] != 0: continue\n                        exterior.add((cr, cc))\n                        stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])\n        for c in range(C):\n            for r in [0, R-1]:\n                if inp[r][c] == 0:\n                    stack = [(r, c)]\n                    while stack:\n                        cr, cc = stack.pop()\n                        if (cr, cc) in exterior: continue\n                        if cr < 0 or cr >= R or cc < 0 or cc >= C: continue\n                        if inp[cr][cc] != 0: continue\n                        exterior.add((cr, cc))\n                        stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])\n        \n        # Interior = 0-pixels not in exterior\n        interior = set()\n        for r in range(R):\n            for c in range(C):\n                if inp[r][c] == 0 and (r, c) not in exterior:\n                    interior.add((r, c))\n        \n        # Check if interior matches filled pixels\n        if interior == filled_pixels:\n            # Apply to test input\n            tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n            result = [row[:] for row in test_input]\n            \n            # Find fill color for test: use same logic\n            # For test, we need to determine what fill color to use\n            # Use the most common fill color from training\n            test_exterior = set()\n            for r in range(tR):\n                for c in [0, tC-1]:\n                    if test_input[r][c] == 0:\n                        stack = [(r, c)]\n                        while stack:\n                            cr, cc = stack.pop()\n                            if (cr, cc) in test_exterior: continue\n                            if cr < 0 or cr >= tR or cc < 0 or cc >= tC: continue\n                            if test_input[cr][cc] != 0: continue\n                            test_exterior.add((cr, cc))\n                            stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])\n            for c in range(tC):\n                for r in [0, tR-1]:\n                    if test_input[r][c] == 0:\n                        stack = [(r, c)]\n                        while stack:\n                            cr, cc = stack.pop()\n                            if (cr, cc) in test_exterior: continue\n                            if cr < 0 or cr >= tR or cc < 0 or cc >= tC: continue\n                            if test_input[cr][cc] != 0: continue\n                            test_exterior.add((cr, cc))\n                            stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])\n            \n            test_interior = set()\n            for r in range(tR):\n                for c in range(tC):\n                    if test_input[r][c] == 0 and (r, c) not in test_exterior:\n                        test_interior.add((r, c))\n            \n            # Fill interior with the fill color\n            for r, c in test_interior:\n                result[r][c] = fill_color\n            \n            return result\n    \n    return None\n\n# \u2500\u2500\u2500 Strategy 6: Shape Expansion \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_shape_expansion(train_pairs, test_input):\n    \"\"\"Output adds new elements around/inside input shapes.\"\"\"\n    for inp, out in train_pairs:\n        R, C = len(inp), len(inp[0]) if inp else 0\n        oR, oC = len(out), len(out[0]) if out else 0\n        \n        if R == 0 or C == 0: continue\n        \n        # Check if output is input with added border\n        if oR == R + 2 and oC == C + 2:\n            ok = True\n            for r in range(R):\n                for c in range(C):\n                    if out[r+1][c+1] != inp[r][c]:\n                        ok = False; break\n                if not ok: break\n            if ok:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                result = [[0]*(tC+2) for _ in range(tR+2)]\n                for r in range(tR):\n                    for c in range(tC):\n                        result[r+1][c+1] = test_input[r][c]\n                return result\n        \n        # Check if output is input with added row/col\n        if oR == R + 1 and oC == C:\n            ok = all(out[r][c] == inp[r][c] for r in range(R) for c in range(C))\n            if ok:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                result = [row[:] for row in test_input]\n                result.append([0]*tC)\n                return result\n        \n        if oR == R and oC == C + 1:\n            ok = all(out[r][c] == inp[r][c] for r in range(R) for c in range(C))\n            if ok:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                result = [row + [0] for row in test_input]\n                return result\n    \n    return None\n\n# \u2500\u2500\u2500 Strategy 7: Color Spread \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_color_spread(train_pairs, test_input):\n    \"\"\"Non-zero colors spread to adjacent 0-cells.\"\"\"\n    for inp, out in train_pairs:\n        R, C = len(inp), len(inp[0]) if inp else 0\n        oR, oC = len(out), len(out[0]) if out else 0\n        if (R, C) != (oR, oC): continue\n        \n        # Check if output has more non-zero cells than input\n        in_nonzero = sum(1 for r in range(R) for c in range(C) if inp[r][c] != 0)\n        out_nonzero = sum(1 for r in range(R) for c in range(C) if out[r][c] != 0)\n        \n        if out_nonzero <= in_nonzero: continue\n        \n        # Find spread pattern: 0-cells that became non-zero\n        spreads = []\n        for r in range(R):\n            for c in range(C):\n                if inp[r][c] == 0 and out[r][c] != 0:\n                    # Check neighbors\n                    neighbors = []\n                    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:\n                        nr, nc = r+dr, c+dc\n                        if 0 <= nr < R and 0 <= nc < C and inp[nr][nc] != 0:\n                            neighbors.append(inp[nr][nc])\n                    spreads.append((r, c, out[r][c], neighbors))\n        \n        if not spreads: continue\n        \n        # Check if spread color matches a neighbor\n        all_match = all(len(n) > 0 and s[2] in n for s in spreads if s[3])\n        if all_match:\n            # Apply to test input\n            tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n            result = [row[:] for row in test_input]\n            \n            # Spread: for each 0-cell adjacent to non-zero, copy the neighbor's color\n            changed = True\n            while changed:\n                changed = False\n                for r in range(tR):\n                    for c in range(tC):\n                        if result[r][c] == 0:\n                            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:\n                                nr, nc = r+dr, c+dc\n                                if 0 <= nr < tR and 0 <= nc < tC and test_input[nr][nc] != 0:\n                                    result[r][c] = test_input[nr][nc]\n                                    changed = True\n                                    break\n            \n            return result\n    \n    return None\n\n# \u2500\u2500\u2500 Strategy 8: Nearest Neighbor \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_nearest_neighbor(train_pairs, test_input):\n    if not train_pairs: return None\n    best_sim = -1\n    best_out = None\n    for inp, out in train_pairs:\n        sim = sum(1 for r in range(min(len(inp), len(test_input)))\n                 for c in range(min(len(inp[0]) if inp else 0, len(test_input[0]) if test_input else 0))\n                 if inp[r][c] == test_input[r][c])\n        if sim > best_sim:\n            best_sim = sim\n            best_out = out\n    if best_out:\n        R, C = len(test_input), len(test_input[0]) if test_input else 0\n        result = [[0]*C for _ in range(R)]\n        for r in range(min(R, len(best_out))):\n            for c in range(min(C, len(best_out[r]))):\n                result[r][c] = best_out[r][c]\n        return result\n    return None\n\n# \u2500\u2500\u2500 Strategy 9: Diagonal Tiling \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_diagonal_tiling(train_pairs, test_input):\n    for inp, out in train_pairs:\n        ir, ic = len(inp), len(inp[0]) if inp else 0\n        or_, oc = len(out), len(out[0]) if out else 0\n        if or_ % ir != 0 or oc % ic != 0: continue\n        tr, tc = or_ // ir, oc // ic\n        ok = True\n        for r in range(or_):\n            for c in range(oc):\n                tile_r, tile_c = r // ir, c // ic\n                if tile_r == tile_c:\n                    if out[r][c] != inp[r % ir][c % ic]:\n                        ok = False; break\n                else:\n                    if out[r][c] != 0:\n                        ok = False; break\n            if not ok: break\n        if ok:\n            tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n            result = [[0]*(tC*tc) for _ in range(tR*tr)]\n            for tile_idx in range(min(tr, tc)):\n                r0 = tile_idx * tR\n                c0 = tile_idx * tC\n                for r in range(tR):\n                    for c in range(tC):\n                        result[r0+r][c0+c] = test_input[r][c]\n            return result\n    return None\n\n# \u2500\u2500\u2500 Strategy 10: Object Count \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_object_count(train_pairs, test_input):\n    \"\"\"If output has different number of objects, try to match count.\"\"\"\n    for inp, out in train_pairs:\n        # Count connected components\n        def count_objects(grid):\n            R, C = len(grid), len(grid[0]) if grid else 0\n            visited = set()\n            count = 0\n            for r in range(R):\n                for c in range(C):\n                    if (r,c) not in visited and grid[r][c] != 0:\n                        count += 1\n                        stack = [(r,c)]\n                        while stack:\n                            cr, cc = stack.pop()\n                            if (cr,cc) in visited or cr<0 or cr>=R or cc<0 or cc>=C: continue\n                            if grid[cr][cc] != grid[r][c]: continue\n                            visited.add((cr,cc))\n                            stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])\n            return count\n        \n        in_count = count_objects(inp)\n        out_count = count_objects(out)\n        \n        if in_count != out_count:\n            # Try to adjust object count in test\n            test_count = count_objects(test_input)\n            if test_count != out_count:\n                # This strategy is too complex to implement generically\n                pass\n    \n    return None\n\n# \u2500\u2500\u2500 Master Solver \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nSTRATEGIES = [\n    (\"color_mapping\", try_color_mapping),\n    (\"interior_fill\", try_interior_fill),\n    (\"tiling\", try_tiling),\n    (\"rotation\", try_rotation),\n    (\"reflection\", try_reflection),\n    (\"shape_expansion\", try_shape_expansion),\n    (\"color_spread\", try_color_spread),\n    (\"diagonal_tiling\", try_diagonal_tiling),\n    (\"nearest_neighbor\", try_nearest_neighbor),\n]\n\ndef solver_v3(train_pairs, test_input):\n    \"\"\"Try all strategies, return best match (or first non-None if no match).\"\"\"\n    candidates = []\n    for name, fn in STRATEGIES:\n        try:\n            result = fn(train_pairs, test_input)\n            if result is not None:\n                candidates.append((name, result))\n        except:\n            continue\n    if candidates:\n        return candidates[0][1]\n    return [row[:] for row in test_input]\n\n# \u2500\u2500\u2500 Evaluation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef evaluate_solver(solver_fn, puzzles, max_puzzles=200):\n    correct = 0\n    total = 0\n    strategy_hits = defaultdict(int)\n    task_ids = list(puzzles.keys())[:max_puzzles]\n    \n    for task_id in task_ids:\n        puzzle = puzzles[task_id]\n        train_pairs = get_train_pairs(puzzle)\n        test_input = get_test_input(puzzle)\n        if not test_input: continue\n        expected = puzzle[\"test\"][0][\"output\"]\n        \n        solved = False\n        for name, fn in STRATEGIES:\n            try:\n                result = fn(train_pairs, test_input)\n                if result is not None and grids_equal(result, expected):\n                    correct += 1\n                    strategy_hits[name] += 1\n                    solved = True\n                    break\n            except:\n                continue\n        \n        total += 1\n    \n    return {\n        \"solver\": solver_fn.__name__,\n        \"correct\": correct,\n        \"total\": total,\n        \"accuracy\": correct / total if total > 0 else 0,\n        \"strategy_hits\": dict(strategy_hits),\n    }\n\n# \u2500\u2500\u2500 Main \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nif __name__ == \"__main__\":\n    data_dir = \"/root/arc-data/training\"\n    print(f\"Loading puzzles from {data_dir}...\")\n    puzzles = load_puzzles(data_dir)\n    print(f\"Loaded {len(puzzles)} puzzles\")\n    \n    print(\"\\n=== Solver v3 Evaluation (200 puzzles) ===\\n\")\n    result = evaluate_solver(solver_v3, puzzles, max_puzzles=200)\n    print(f\"Total: {result['correct']}/{result['total']} | accuracy: {result['accuracy']:.1%}\")\n    print(f\"\\nStrategy breakdown:\")\n    for strat, count in sorted(result['strategy_hits'].items(), key=lambda x: -x[1]):\n        print(f\"  {strat:25s}: {count}\")\n    \n    print(\"\\n=== Solver v3 Evaluation (1000 puzzles) ===\\n\")\n    result_full = evaluate_solver(solver_v3, puzzles, max_puzzles=1000)\n    print(f\"Total: {result_full['correct']}/{result_full['total']} | accuracy: {result_full['accuracy']:.1%}\")\n    print(f\"\\nStrategy breakdown:\")\n    for strat, count in sorted(result_full['strategy_hits'].items(), key=lambda x: -x[1]):\n        print(f\"  {strat:25s}: {count}\")\n", "rsi_module.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"\nARC-AGI-2 RSI Module \u2014 Recursive Self-Improvement for ARC Solvers\nAnalyzes failures, generates hypotheses, implements new strategies.\n\"\"\"\nimport json\nimport os\nimport math\nfrom pathlib import Path\nfrom typing import List, Tuple, Dict, Any, Optional, Callable\nfrom collections import Counter, defaultdict\n\n# \u2500\u2500\u2500 RSI Core \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass ARC_RSI:\n    \"\"\"Compressed RSI cycle for ARC puzzle solving.\"\"\"\n    \n    def __init__(self, data_dir: str):\n        self.data_dir = data_dir\n        self.puzzles = self._load_puzzles()\n        self.failures = []\n        self.strategies = []\n        self.solved = []\n        self.hypotheses = []\n    \n    def _load_puzzles(self) -> Dict:\n        puzzles = {}\n        for f in sorted(Path(self.data_dir).glob(\"*.json\")):\n            with open(f) as fp:\n                puzzles[f.stem] = json.load(fp)\n        return puzzles\n    \n    # \u2500\u2500\u2500 OBSERVE: Analyze failures \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    \n    def observe(self, solver_fn, max_puzzles=200):\n        \"\"\"Run solver, collect failures, categorize.\"\"\"\n        self.failures = []\n        self.solved = []\n        \n        task_ids = list(self.puzzles.keys())[:max_puzzles]\n        \n        for task_id in task_ids:\n            puzzle = self.puzzles[task_id]\n            trains = [(p[\"input\"], p[\"output\"]) for p in puzzle.get(\"train\", [])]\n            tests = puzzle.get(\"test\", [])\n            if not tests: continue\n            \n            test_input = tests[0][\"input\"]\n            expected = tests[0][\"output\"]\n            \n            try:\n                result = solver_fn(trains, test_input)\n                if result == expected:\n                    self.solved.append(task_id)\n                    continue\n            except:\n                pass\n            \n            # Analyze failure\n            failure = self._analyze_failure(task_id, trains, test_input, expected)\n            self.failures.append(failure)\n        \n        return {\n            \"solved\": len(self.solved),\n            \"failed\": len(self.failures),\n            \"total\": len(task_ids),\n        }\n    \n    def _analyze_failure(self, task_id, trains, test_input, expected):\n        \"\"\"Deep analysis of why a puzzle failed.\"\"\"\n        analysis = {\"task_id\": task_id}\n        \n        # Shape analysis\n        in_shapes = [(len(p[0]), len(p[0][0]) if p[0] else 0) for p in trains]\n        out_shapes = [(len(p[1]), len(p[1][0]) if p[1] else 0) for p in trains]\n        test_shape = (len(test_input), len(test_input[0]) if test_input else 0)\n        expected_shape = (len(expected), len(expected[0]) if expected else 0)\n        \n        analysis[\"input_shapes\"] = in_shapes\n        analysis[\"output_shapes\"] = out_shapes\n        analysis[\"test_shape\"] = test_shape\n        analysis[\"expected_shape\"] = expected_shape\n        analysis[\"shape_change\"] = in_shapes != out_shapes\n        \n        # Scale factor\n        if in_shapes and out_shapes:\n            avg_in_area = sum(r*c for r,c in in_shapes) / len(in_shapes)\n            avg_out_area = sum(r*c for r,c in out_shapes) / len(out_shapes)\n            analysis[\"scale_factor\"] = avg_out_area / avg_in_area if avg_in_area > 0 else 1\n        \n        # Color analysis\n        all_colors = set()\n        for inp, out in trains:\n            all_colors.update(c for row in inp for c in row)\n            all_colors.update(c for row in out for c in row)\n        analysis[\"n_colors\"] = len(all_colors)\n        \n        # Color map consistency\n        cm = self._find_color_map(trains)\n        analysis[\"consistent_color_map\"] = cm is not None\n        if cm:\n            analysis[\"color_map\"] = cm\n        \n        # Spatial pattern detection\n        analysis[\"has_symmetry\"] = self._detect_symmetry(trains)\n        analysis[\"has_progression\"] = self._detect_progression(trains)\n        analysis[\"has_object_movement\"] = self._detect_object_movement(trains)\n        \n        return analysis\n    \n    def _find_color_map(self, trains):\n        \"\"\"Find consistent color mapping.\"\"\"\n        cm = None\n        for inp, out in trains:\n            pair_cm = {}\n            ok = True\n            for r in range(min(len(inp), len(out))):\n                for c in range(min(len(inp[r]), len(out[r]))):\n                    k, v = inp[r][c], out[r][c]\n                    if k in pair_cm:\n                        if pair_cm[k] != v: ok = False; break\n                    else: pair_cm[k] = v\n                if not ok: break\n            if not ok: return None\n            if cm is None: cm = pair_cm\n            else:\n                for k, v in pair_cm.items():\n                    if k in cm and cm[k] != v: return None\n        return cm\n    \n    def _detect_symmetry(self, trains):\n        \"\"\"Detect if output has symmetry patterns.\"\"\"\n        for inp, out in trains:\n            R, C = len(out), len(out[0]) if out else 0\n            if R == 0 or C == 0: continue\n            \n            # Check horizontal symmetry\n            h_sym = all(out[r] == out[R-1-r] for r in range(R//2))\n            # Check vertical symmetry\n            v_sym = all(out[r][c] == out[r][C-1-c] for r in range(R) for c in range(C//2))\n            \n            if h_sym or v_sym:\n                return {\"horizontal\": h_sym, \"vertical\": v_sym}\n        return None\n    \n    def _detect_progression(self, trains):\n        \"\"\"Detect if there's a progression pattern (e.g., color changes incrementally).\"\"\"\n        if len(trains) < 2: return None\n        \n        # Check if color counts change progressively\n        color_counts = []\n        for inp, out in trains:\n            cc = Counter(c for row in out for c in row)\n            color_counts.append(cc)\n        \n        # Check if any color count changes linearly\n        for color in set(c for cc in color_counts for c in cc):\n            counts = [cc.get(color, 0) for cc in color_counts]\n            if len(counts) >= 2:\n                diffs = [counts[i+1] - counts[i] for i in range(len(counts)-1)]\n                if len(set(diffs)) == 1:  # constant difference\n                    return {\"color\": color, \"counts\": counts, \"diff\": diffs[0]}\n        return None\n    \n    def _detect_object_movement(self, trains):\n        \"\"\"Detect if objects (connected components) move between input and output.\"\"\"\n        for inp, out in trains[:1]:  # Check first pair\n            # Find connected components in input\n            in_objects = self._find_objects(inp)\n            out_objects = self._find_objects(out)\n            \n            if len(in_objects) > 0 and len(out_objects) > 0:\n                # Check if object count changes\n                if len(in_objects) != len(out_objects):\n                    return {\"in_count\": len(in_objects), \"out_count\": len(out_objects)}\n        return None\n    \n    def _find_objects(self, grid):\n        \"\"\"Find connected components (objects) in a grid.\"\"\"\n        R, C = len(grid), len(grid[0]) if grid else 0\n        visited = set()\n        objects = []\n        \n        for r in range(R):\n            for c in range(C):\n                if (r,c) not in visited and grid[r][c] != 0:\n                    # BFS\n                    obj = []\n                    stack = [(r,c)]\n                    while stack:\n                        cr, cc = stack.pop()\n                        if (cr,cc) in visited or cr<0 or cr>=R or cc<0 or cc>=C:\n                            continue\n                        if grid[cr][cc] != grid[r][c]:\n                            continue\n                        visited.add((cr,cc))\n                        obj.append((cr,cc))\n                        stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])\n                    objects.append(obj)\n        \n        return objects\n    \n    # \u2500\u2500\u2500 ORIENT: Generate hypotheses \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    \n    def orient(self):\n        \"\"\"Analyze failure patterns, generate strategy hypotheses.\"\"\"\n        self.hypotheses = []\n        \n        # Pattern 1: Shape changes with scaling\n        shape_changes = [f for f in self.failures if f.get(\"shape_change\")]\n        if shape_changes:\n            scales = [f.get(\"scale_factor\", 1) for f in shape_changes]\n            avg_scale = sum(scales) / len(scales)\n            self.hypotheses.append({\n                \"type\": \"scaling\",\n                \"description\": f\"Average scale factor: {avg_scale:.1f}x\",\n                \"count\": len(shape_changes),\n                \"strategy\": \"adaptive_tiling\",\n            })\n        \n        # Pattern 2: Symmetry in output\n        symmetries = [f for f in self.failures if f.get(\"has_symmetry\")]\n        if symmetries:\n            self.hypotheses.append({\n                \"type\": \"symmetry\",\n                \"description\": f\"{len(symmetries)} puzzles have symmetry in output\",\n                \"count\": len(symmetries),\n                \"strategy\": \"symmetry_generation\",\n            })\n        \n        # Pattern 3: Progression patterns\n        progressions = [f for f in self.failures if f.get(\"has_progression\")]\n        if progressions:\n            self.hypotheses.append({\n                \"type\": \"progression\",\n                \"description\": f\"{len(progressions)} puzzles have progression patterns\",\n                \"count\": len(progressions),\n                \"strategy\": \"progression_solver\",\n            })\n        \n        # Pattern 4: Object movement\n        movements = [f for f in self.failures if f.get(\"has_object_movement\")]\n        if movements:\n            self.hypotheses.append({\n                \"type\": \"movement\",\n                \"description\": f\"{len(movements)} puzzles have object count changes\",\n                \"count\": len(movements),\n                \"strategy\": \"object_manipulation\",\n            })\n        \n        # Pattern 5: Many colors, no shape change\n        complex_same_shape = [f for f in self.failures \n                             if not f.get(\"shape_change\") and f.get(\"n_colors\", 0) >= 6]\n        if complex_same_shape:\n            self.hypotheses.append({\n                \"type\": \"complex_mapping\",\n                \"description\": f\"{len(complex_same_shape)} puzzles with many colors, same shape\",\n                \"count\": len(complex_same_shape),\n                \"strategy\": \"context_aware_color_map\",\n            })\n        \n        return self.hypotheses\n    \n    # \u2500\u2500\u2500 DECIDE: Select strategies to implement \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    \n    def decide(self):\n        \"\"\"Prioritize hypotheses by potential impact.\"\"\"\n        # Sort by count (most common pattern first)\n        self.hypotheses.sort(key=lambda h: -h.get(\"count\", 0))\n        return self.hypotheses\n    \n    # \u2500\u2500\u2500 REPORT \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    \n    def report(self):\n        \"\"\"Generate RSI report.\"\"\"\n        return {\n            \"observed\": {\n                \"solved\": len(self.solved),\n                \"failed\": len(self.failures),\n            },\n            \"hypotheses\": self.hypotheses,\n            \"top_strategies\": [h[\"strategy\"] for h in self.hypotheses[:3]],\n        }\n\n\n# \u2500\u2500\u2500 New Strategies from RSI \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef try_adaptive_tiling(train_pairs, test_input):\n    \"\"\"Tiling that adapts based on scale factor.\"\"\"\n    for inp, out in train_pairs:\n        ir, ic = len(inp), len(inp[0]) if inp else 0\n        or_, oc = len(out), len(out[0]) if out else 0\n        if ir == 0 or ic == 0: continue\n        if or_ % ir != 0 or oc % ic != 0: continue\n        tr, tc = or_ // ir, oc // ic\n        \n        # Try multiple tiling patterns\n        # 1. Simple repeat\n        ok = all(out[r][c] == inp[r % ir][c % ic] for r in range(or_) for c in range(oc))\n        if ok:\n            R, C = len(test_input), len(test_input[0]) if test_input else 0\n            return [[test_input[r % R][c % C] for c in range(C * tc)] for r in range(R * tr)]\n        \n        # 2. Alternating flip (checkerboard)\n        for flip_axis in ['row', 'col', 'both']:\n            ok = True\n            for r in range(or_):\n                for c in range(oc):\n                    sr, sc = r % ir, c % ic\n                    tr_i, tc_i = r // ir, c // ic\n                    if flip_axis in ['row', 'both'] and tr_i % 2 == 1:\n                        sr = ir - 1 - sr\n                    if flip_axis in ['col', 'both'] and tc_i % 2 == 1:\n                        sc = ic - 1 - sc\n                    if out[r][c] != inp[sr][sc]:\n                        ok = False; break\n                if not ok: break\n            if ok:\n                R, C = len(test_input), len(test_input[0]) if test_input else 0\n                result = [[0]*(C*tc) for _ in range(R*tr)]\n                for r in range(R*tr):\n                    for c in range(C*tc):\n                        sr, sc = r % R, c % C\n                        tr_i, tc_i = r // R, c // C\n                        if flip_axis in ['row', 'both'] and tr_i % 2 == 1:\n                            sr = R - 1 - sr\n                        if flip_axis in ['col', 'both'] and tc_i % 2 == 1:\n                            sc = C - 1 - sc\n                        result[r][c] = test_input[sr][sc]\n                return result\n        \n        # 3. Rotation tiling\n        rotations = [lambda g: g, \n                     lambda g: [list(reversed(row)) for row in g],  # flip_h\n                     lambda g: list(reversed(g)),  # flip_v\n                     lambda g: list(zip(*reversed(g)))]  # rotate_90 approx\n        for rot_fn in rotations:\n            try:\n                rotated = rot_fn(inp)\n                if len(rotated) != ir or (rotated and len(rotated[0]) != ic):\n                    continue\n                ok = all(out[r][c] == rotated[r % ir][c % ic] for r in range(or_) for c in range(oc))\n                if ok:\n                    R, C = len(test_input), len(test_input[0]) if test_input else 0\n                    rot_test = rot_fn(test_input)\n                    return [[rot_test[r % R][c % C] for c in range(C * tc)] for r in range(R * tr)]\n            except:\n                continue\n    \n    return None\n\n\ndef try_symmetry_generation(train_pairs, test_input):\n    \"\"\"Generate symmetric output from input.\"\"\"\n    for inp, out in train_pairs:\n        R, C = len(out), len(out[0]) if out else 0\n        iR, iC = len(inp), len(inp[0]) if inp else 0\n        \n        if iR == 0 or iC == 0: continue\n        \n        # Check if output is input reflected horizontally\n        if R == iR and C == iC * 2:\n            ok = all(out[r][c] == inp[r][c % iC] for r in range(R) for c in range(C))\n            if ok:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                return [[test_input[r][c % tC] for c in range(tC * 2)] for r in range(tR)]\n        \n        # Check if output is input reflected vertically\n        if R == iR * 2 and C == iC:\n            ok = all(out[r][c] == inp[r % iR][c] for r in range(R) for c in range(C))\n            if ok:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                return [[test_input[r % tR][c] for c in range(tC)] for r in range(tR * 2)]\n        \n        # Check if output is 4-way symmetry\n        if R == iR * 2 and C == iC * 2:\n            ok = True\n            for r in range(R):\n                for c in range(C):\n                    sr, sc = r % iR, c % iC\n                    if r >= iR: sr = iR - 1 - sr\n                    if c >= iC: sc = iC - 1 - sc\n                    if out[r][c] != inp[sr][sc]:\n                        ok = False; break\n                if not ok: break\n            if ok:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                result = [[0]*(tC*2) for _ in range(tR*2)]\n                for r in range(tR*2):\n                    for c in range(tC*2):\n                        sr, sc = r % tR, c % tC\n                        if r >= tR: sr = tR - 1 - sr\n                        if c >= tC: sc = tC - 1 - sc\n                        result[r][c] = test_input[sr][sc]\n                return result\n    \n    return None\n\n\ndef try_context_aware_color_map(train_pairs, test_input):\n    \"\"\"Color mapping that considers spatial context.\"\"\"\n    if len(train_pairs) < 2: return None\n    \n    # Build position-dependent color map\n    R, C = len(test_input), len(test_input[0]) if test_input else 0\n    \n    # For each position, collect all input\u2192output mappings\n    pos_maps = defaultdict(lambda: defaultdict(int))\n    for inp, out in train_pairs:\n        for r in range(min(len(inp), len(out))):\n            for c in range(min(len(inp[r]), len(out[r]))):\n                key = (inp[r][c], r, c)\n                pos_maps[(r,c)][inp[r][c]] = out[r][c]\n    \n    # Apply: for each position, use the most common mapping\n    result = [row[:] for row in test_input]\n    for r in range(R):\n        for c in range(C):\n            if (r,c) in pos_maps and test_input[r][c] in pos_maps[(r,c)]:\n                result[r][c] = pos_maps[(r,c)][test_input[r][c]]\n    \n    return result\n\n\ndef try_shape_change_solver(train_pairs, test_input):\n    \"\"\"Handle puzzles where output is a different shape than input.\"\"\"\n    for inp, out in train_pairs:\n        ir, ic = len(inp), len(inp[0]) if inp else 0\n        or_, oc = len(out), len(out[0]) if out else 0\n        \n        if ir == 0 or ic == 0: continue\n        \n        # Case 1: Output is taller (add rows)\n        if or_ > ir and oc == ic:\n            # Find what rows are added\n            row_pattern = []\n            for r in range(or_):\n                src_r = r % ir\n                if out[r] == inp[src_r]:\n                    row_pattern.append(\"same\")\n                elif r > 0 and out[r] == out[r-1]:\n                    row_pattern.append(\"repeat\")\n                else:\n                    row_pattern.append(\"new\")\n            \n            if \"repeat\" in row_pattern or \"same\" in row_pattern:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                result = []\n                for r in range(or_):\n                    src_r = r % tR\n                    result.append(test_input[src_r][:])\n                return result\n        \n        # Case 2: Output is wider (add columns)\n        if or_ == ir and oc > ic:\n            col_pattern = []\n            for c in range(oc):\n                src_c = c % ic\n                col_match = all(out[r][c] == inp[r][src_c] for r in range(ir))\n                if col_match:\n                    col_pattern.append(\"same\")\n                else:\n                    col_pattern.append(\"new\")\n            \n            if \"same\" in col_pattern:\n                tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n                result = []\n                for r in range(or_):\n                    src_r = r % tR\n                    row = []\n                    for c in range(oc):\n                        src_c = c % tC\n                        row.append(test_input[src_r][src_c])\n                    result.append(row)\n                return result\n        \n        # Case 3: Both taller and wider\n        if or_ > ir and oc > ic:\n            tR, tC = len(test_input), len(test_input[0]) if test_input else 0\n            result = []\n            for r in range(or_):\n                src_r = r % tR\n                row = []\n                for c in range(oc):\n                    src_c = c % tC\n                    row.append(test_input[src_r][src_c])\n                result.append(row)\n            return result\n    \n    return None\n\n\n# \u2500\u2500\u2500 Combined Smart Solver with RSI \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef solver_rsi(train_pairs, test_input):\n    \"\"\"Solver enhanced with RSI-generated strategies.\"\"\"\n    strategies = [\n        # Original strategies\n        (\"color_mapping\", _try_color_mapping),\n        (\"tiling\", _try_tiling),\n        (\"rotation\", _try_rotation),\n        (\"reflection\", _try_reflection),\n        (\"flood_fill\", _try_flood_fill),\n        # RSI-generated strategies\n        (\"adaptive_tiling\", try_adaptive_tiling),\n        (\"symmetry_generation\", try_symmetry_generation),\n        (\"context_aware_color_map\", try_context_aware_color_map),\n        (\"shape_change_solver\", try_shape_change_solver),\n    ]\n    \n    for name, fn in strategies:\n        try:\n            result = fn(train_pairs, test_input)\n            if result is not None:\n                return result\n        except:\n            continue\n    \n    return [row[:] for row in test_input]\n\n\n# \u2500\u2500\u2500 Helper functions (copied from solver_smart_v2) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef _try_color_mapping(train_pairs, test_input):\n    cm = None\n    for inp, out in train_pairs:\n        pair_cm = {}\n        ok = True\n        for r in range(min(len(inp), len(out))):\n            for c in range(min(len(inp[r]), len(out[r]))):\n                k, v = inp[r][c], out[r][c]\n                if k in pair_cm:\n                    if pair_cm[k] != v: ok = False; break\n                else: pair_cm[k] = v\n            if not ok: break\n        if not ok: return None\n        if cm is None: cm = pair_cm\n        else:\n            for k, v in pair_cm.items():\n                if k in cm and cm[k] != v: return None\n    \n    if not cm: return None\n    result = [row[:] for row in test_input]\n    for r in range(len(result)):\n        for c in range(len(result[r])):\n            if result[r][c] in cm:\n                result[r][c] = cm[result[r][c]]\n    return result\n\ndef _try_tiling(train_pairs, test_input):\n    for inp, out in train_pairs:\n        ir, ic = len(inp), len(inp[0]) if inp else 0\n        or_, oc = len(out), len(out[0]) if out else 0\n        if ir == 0 or ic == 0: continue\n        if or_ % ir != 0 or oc % ic != 0: continue\n        tr, tc = or_ // ir, oc // ic\n        if tr < 2 and tc < 2: continue\n        ok = all(out[r][c] == inp[r % ir][c % ic] for r in range(or_) for c in range(oc))\n        if ok:\n            R, C = len(test_input), len(test_input[0]) if test_input else 0\n            return [[test_input[r % R][c % C] for c in range(C * tc)] for r in range(R * tr)]\n    return None\n\ndef _try_rotation(train_pairs, test_input):\n    def rot90(g):\n        R, C = len(g), len(g[0]) if g else 0\n        return [[g[R-1-c][r] for c in range(R)] for r in range(C)] if R and C else g\n    def rot180(g):\n        R, C = len(g), len(g[0]) if g else 0\n        return [[g[R-1-r][C-1-c] for c in range(C)] for r in range(R)]\n    def rot270(g):\n        R, C = len(g), len(g[0]) if g else 0\n        return [[g[c][C-1-r] for c in range(R)] for r in range(C)] if R and C else g\n    \n    for inp, out in train_pairs:\n        if inp and out:\n            if rot90(inp) == out: return rot90(test_input)\n            if rot180(inp) == out: return rot180(test_input)\n            if rot270(inp) == out: return rot270(test_input)\n    return None\n\ndef _try_reflection(train_pairs, test_input):\n    for inp, out in train_pairs:\n        if inp and out:\n            if [row[::-1] for row in inp] == out: return [row[::-1] for row in test_input]\n            if inp[::-1] == out: return test_input[::-1]\n    return None\n\ndef _try_flood_fill(train_pairs, test_input):\n    def ff(grid, sr, sc, nc):\n        R, C = len(grid), len(grid[0]) if grid else 0\n        result = [row[:] for row in grid]\n        oc = result[sr][sc]\n        if oc == nc: return result\n        stack = [(sr, sc)]\n        visited = set()\n        while stack:\n            r, c = stack.pop()\n            if (r,c) in visited or r<0 or r>=R or c<0 or c>=C or result[r][c]!=oc: continue\n            visited.add((r,c))\n            result[r][c] = nc\n            stack.extend([(r-1,c),(r+1,c),(r,c-1),(r,c+1)])\n        return result\n    \n    for inp, out in train_pairs:\n        R, C = len(inp), len(inp[0]) if inp else 0\n        oR, oC = len(out), len(out[0]) if out else 0\n        if (R, C) != (oR, oC): continue\n        changes = [(r,c,inp[r][c],out[r][c]) for r in range(R) for c in range(C) if inp[r][c]!=out[r][c]]\n        if len(changes) == 1:\n            r, c, _, new = changes[0]\n            if ff(inp, r, c, new) == out:\n                return ff(test_input, r, c, new)\n    return None\n\n\n# \u2500\u2500\u2500 Main \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nif __name__ == \"__main__\":\n    data_dir = \"/root/arc-data/training\"\n    \n    print(\"=== ARC RSI Cycle ===\\n\")\n    \n    # Initialize RSI\n    rsi = ARC_RSI(data_dir)\n    \n    # Import the solver for observation\n    from solver_smart_v2 import solver_smart_v2\n    \n    # OBSERVE\n    print(\"1. OBSERVE: Analyzing failures...\")\n    obs = rsi.observe(solver_smart_v2, max_puzzles=200)\n    print(f\"   Solved: {obs['solved']}, Failed: {obs['failed']}\")\n    \n    # ORIENT\n    print(\"\\n2. ORIENT: Generating hypotheses...\")\n    hyps = rsi.orient()\n    for h in hyps:\n        print(f\"   - {h['type']}: {h['description']}\")\n    \n    # DECIDE\n    print(\"\\n3. DECIDE: Prioritizing strategies...\")\n    strategies = rsi.decide()\n    for i, s in enumerate(strategies[:3]):\n        print(f\"   {i+1}. {s['strategy']} (impact: {s['count']} puzzles)\")\n    \n    # Test RSI-enhanced solver\n    print(\"\\n4. ACT: Testing RSI-enhanced solver...\")\n    from solver_smart_v2 import load_puzzles, evaluate_solver\n    \n    puzzles = load_puzzles(data_dir)\n    result = evaluate_solver(solver_rsi, puzzles, max_puzzles=200)\n    print(f\"   RSI Solver: {result['correct']}/{result['total']} | accuracy: {result['accuracy']:.1%}\")\n    \n    # Compare with v2\n    result_v2 = evaluate_solver(solver_smart_v2, puzzles, max_puzzles=200)\n    print(f\"   V2 Solver:  {result_v2['correct']}/{result_v2['total']} | accuracy: {result_v2['accuracy']:.1%}\")\n    \n    improvement = result['accuracy'] - result_v2['accuracy']\n    print(f\"\\n   Improvement: {improvement:+.1%}\")\n    \n    # Save RSI report\n    report = rsi.report()\n    with open(\"/root/rsi_report.json\", \"w\") as f:\n        json.dump(report, f, indent=2)\n    print(\"\\n   RSI report saved to /root/rsi_report.json\")\n", "agent.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"Inference-time agent v1: perceive -> propose -> critic -> decide.\n\nThe offline agent S5 asked for: understands the presented puzzle and\ndecides, alone, with no internet. Deterministic (R10: no random, no\nwall-clock branching \u2014 caps are fixed counts), stdlib-only.\n\n- perceive: classify() dynamic + palette stats (orientation only).\n- propose: EVERY LOO-verifying dynamic's strategies (not just the\n  first hit) + fallback chain -> candidates with provenance.\n  Grids deduped; agreement counted across producers.\n- critic (NO ground truth available): score =\n  0.5 * loo_frac + 0.3 * agreement + 0.2 * prior, where loo_frac\n  = fraction of train pairs verified via LOO (0.0-1.0, more signal\n  than binary), agreement = share of VERIFYING producers emitting\n  this exact grid (fallback noise that never verifies cannot boost),\n  prior = per-dynamic training p@1 (PRIORS below, era-93 snapshot).\n  Tie-break: DYNAMICS order, then fn order (stable, deterministic;\n  reproduces legacy routed-first among full ties).\n- decide: attempt_1 = argmax; attempt_2 = best DISTINCT grid\n  (fixes blind second-attempt p@2=+0 with a ranked runner-up).\n- trace: full candidate table + score parts in meta. Accountability:\n  every prediction explains itself (dynamic, producers, scores).\n\nPRIORS regen (era-93 taxonomy; rerun when taxonomy moves):\n  python3 -c \"import json;t=json.load(open('knowledge/arc_taxonomy.json'));print({d['name']:round(d['p1']/d['n'],4) for d in t['dynamics']})\"\n\"\"\"\nimport sys\nfrom pathlib import Path\n\nHERE = Path(__file__).parent\nif str(HERE) not in sys.path:\n    sys.path.insert(0, str(HERE))\n\nimport recognizer as R  # noqa: E402\n\n# RSI bridge: load RSI-generated strategies if available\ntry:\n    import rsi_bridge as RSI  # noqa: E402\nexcept Exception:\n    RSI = None\n\n# Priors: empirical p@1 per dynamic over the full ARC training benchmark\n# (n>=3). Regen after any taxonomy/bench move:\n#   python3 /tmp/opencode/bench_agi2.py 1000   # writes per_dynamic\n#   (copy {d: p1/n for n>=3} below)\n# A2 (2026-10-16 audit): the era-93 snapshot had n=1..2 per dynamic\n# (p1=1.0 trivially, e.g. 1/1) \u2014 pure noise. Only dynamics with a real\n# training history get a prior now.\nPRIORS = {\n    'halfold': 0.67,      # 4/6\n    'norhalves': 0.86,    # 6/7\n    'orhalves': 1.0,      # 3/3\n    'recolor': 1.0,       # 3/3\n    'retiling': 1.0,      # 11/11\n    'rotation': 1.0,      # 3/3\n    'sizefill': 1.0,      # 3/3\n}\n\nW_VERIFIES, W_AGREE, W_PRIOR = 0.5, 0.3, 0.2\n\n\ndef grid_key(g):\n    return tuple(tuple(r) for r in g)\n\n\ndef propose(trains, test):\n    \"\"\"All candidate predictions with provenance.\n\n    Returns list of dicts: {grid, key, producers=[(order, fn, dynamic,\n    verifies)], verifies, agreement, prior, score}. Sorted best-first.\n    \"\"\"\n    order_of = {name: i for i, (name, _d, _f, _a) in enumerate(R.DYNAMICS)}\n    groups = {}\n\n    def add(order, dynamic, fn, pred, verifies, fn_idx=0):\n        if pred is None:\n            return\n        try:\n            key = grid_key(pred)\n        except Exception:\n            return\n        prod = (order, fn_idx, fn.__name__, dynamic, verifies)\n        g = groups.get(key)\n        if g is None:\n            groups[key] = {\"grid\": pred, \"key\": key, \"producers\": [prod]}\n        else:\n            g[\"producers\"].append(prod)\n\n    routed_ids = set()\n    for name, det, fns, _arc3 in R.DYNAMICS:\n        try:\n            hit = det(trains)\n        except Exception:\n            continue\n        if not hit:\n            continue\n        fns_r = R.composed_fns_for(trains) if fns is None else fns\n        for fi, fn in enumerate(fns_r or []):\n            routed_ids.add(id(fn))\n            try:\n                pred = R.safe_apply(fn, trains, test)\n            except Exception:\n                continue\n            if pred is None:\n                continue\n            try:\n                loo_f = R.verify_loo_fraction(fn, trains)\n            except Exception:\n                loo_f = 0.0\n            add(order_of.get(name, 999), name, fn, pred, loo_f,\n                fi)\n    # B (2026-10-16 A+B): composer \u2014 program enumeration certified by\n    # the same LOO verify. Sits between fixed dynamics and blind\n    # fallback so verified compositions outrank unverified guesses.\n    try:\n        import composer as C\n        for cc in C.candidates(trains, test):\n            add(500, \"composer:\" + cc[\"name\"], cc[\"fn\"], cc[\"pred\"],\n                cc[\"loo\"], 0)\n    except Exception:\n        pass\n\n    for fbi, (fname, fn) in enumerate(R.FALLBACK):\n        if id(fn) in routed_ids:\n            continue\n        try:\n            pred = R.safe_apply(fn, trains, test)\n        except Exception:\n            continue\n        if pred is None:\n            continue\n        try:\n            loo_f = R.verify_loo_fraction(fn, trains)\n        except Exception:\n            loo_f = 0.0\n        add(1000 + fbi, \"fallback:\" + fname, fn, pred, loo_f, 0)\n\n    # RSI-generated strategies: merge as additional candidates\n    if RSI is not None:\n        try:\n            rsi_cands = RSI.rsi_propose(trains, test)\n            for rc in rsi_cands:\n                add(2000, rc[\"dynamic\"], type(\"RSIFn\", (), {\n                    \"__name__\": rc[\"fn_name\"]\n                })(), rc[\"grid\"], rc[\"loo_frac\"], 0)\n        except Exception:\n            pass\n\n    cands = []\n    total_loo = sum(p[4] for g in groups.values() for p in g[\"producers\"])\n    for g in groups.values():\n        prods = sorted(g[\"producers\"])\n        loo_max = max(p[4] for p in prods)\n        grid_loo = sum(p[4] for p in prods)\n        agreement = (grid_loo / total_loo) if total_loo else 0.0\n        prior = max(PRIORS.get(p[3].split(\":\")[-1], 0.0) for p in prods)\n        score = W_VERIFIES * loo_max + W_AGREE * agreement + W_PRIOR * prior\n        cands.append({\"grid\": g[\"grid\"], \"key\": g[\"key\"],\n                      \"producers\": prods, \"verifies\": loo_max,\n                      \"agreement\": round(agreement, 4), \"prior\": prior,\n                      \"score\": round(score, 4)})\n    cands.sort(key=lambda c: (-c[\"score\"],\n                              min(p[0] for p in c[\"producers\"]),\n                              min(p[1] for p in c[\"producers\"])))\n    return cands\n\n\ndef decide(trains, test):\n    \"\"\"(attempt_1, attempt_2, meta with trace). attempt_2 may be None.\"\"\"\n    cands = propose(trains, test)\n    trace = [{\"rank\": i + 1,\n              \"dynamic\": c[\"producers\"][0][3],\n              \"fn\": c[\"producers\"][0][2],\n              \"score\": c[\"score\"],\n              \"loo_frac\": c[\"verifies\"],\n              \"agreement\": c[\"agreement\"],\n              \"prior\": c[\"prior\"],\n              \"n_producers\": len(c[\"producers\"])} for i, c in enumerate(cands)]\n    if not cands:\n        dyn, _ev = R.classify(trains)\n        return None, None, {\"dynamic\": dyn, \"fallback\": True,\n                            \"trace\": trace, \"n_candidates\": 0}\n    top = cands[0]\n    # A1 (2026-10-16 audit): attempt_2 must be an INDEPENDENT hypothesis\n    # (different strategy family), not just a distinct grid \u2014 a same-\n    # family twin never saves the miss. Kaggle scores either attempt, so\n    # p@2 uplift comes from genuinely different bets. Distinct-grid\n    # remains last resort only when every candidate shares the family.\n    top_fam = top[\"producers\"][0][3]\n    second = next((c for c in cands[1:]\n                   if c[\"producers\"][0][3] != top_fam), None)\n    if second is None:\n        second = next((c for c in cands[1:] if c[\"key\"] != top[\"key\"]),\n                      None)\n    dyn = top[\"producers\"][0][3]\n    if dyn.startswith(\"fallback:\"):\n        dyn = \"fallback\"\n    return top[\"grid\"], (second[\"grid\"] if second else None), {\n        \"dynamic\": dyn, \"fallback\": top[\"verifies\"] < 0.01,\n        \"trace\": trace, \"n_candidates\": len(cands)}\n", "solve.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"arc-solver public inference API \u2014 pure function, no IO, no state.\n\nUsage:\n    from solve import solve_task\n    out = solve_task(task)   # task: ARC JSON dict with train/test\n    out[\"attempt_1\"]         # primary prediction (p@1 path)\n    out[\"attempt_2\"]         # independent blind second opinion or None\n\nOnly stdlib + the four vendored engine files in this directory.\n\"\"\"\nimport os\nimport sys\nimport time\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nif HERE not in sys.path:\n    sys.path.insert(0, HERE)\n\nimport recognizer as R  # noqa: E402\n\ntry:\n    import agent as AG  # noqa: E402\nexcept Exception:\n    AG = None\n\ntry:\n    import rsi_bridge as RSI  # noqa: E402\nexcept Exception:\n    RSI = None\n\ntry:\n    import llm_harness as LH  # noqa: E402\nexcept Exception:\n    LH = None\n\n\n# Run-level tally of decision sources (printed by the kernel at the end:\n# \"llm+llm-certified\" vs dynamics vs fallback \u2014 evidence of harness mix).\nSOURCE_TALLY = {}\n\n\ndef _solve_legacy(trains, test):\n    \"\"\"Pre-agent path (routed + blind second). Fallback if agent missing.\"\"\"\n    r = R.solve_routed(trains, test, use_fallback=True)\n    a1 = r[\"pred\"]\n    a2 = None\n    seen = set(r.get(\"tried\", []))\n    for _fname, fn in R.FALLBACK:\n        if fn.__name__ in seen:\n            continue\n        try:\n            pred = R.safe_apply(fn, trains, test)\n        except Exception:\n            continue\n        if pred is not None and pred != a1:\n            a2 = pred\n            break\n    # RSI-generated strategies: try as second attempt\n    if a2 is None and RSI is not None:\n        try:\n            for rc in RSI.rsi_propose(trains, test):\n                if rc[\"grid\"] is not None:\n                    grid_bytes = rc[\"grid\"].tobytes()\n                    a1_bytes = a1.tobytes() if a1 is not None else None\n                    if grid_bytes != a1_bytes:\n                        a2 = rc[\"grid\"]\n                        break\n        except Exception:\n            pass\n    return a1, a2, {\"dynamic\": r[\"dynamic\"], \"fallback\": r[\"fallback\"],\n                    \"trace\": [], \"n_candidates\": len(seen)}\n\n\ndef _sym_as_dict(trains, test):\n    \"\"\"agent.decide -> the ensemble's symbolic adapter.\"\"\"\n    a1, a2, meta = AG.decide(trains, test)\n    trace = meta.get(\"trace\") or []\n    loo = trace[0].get(\"loo_frac\") if trace else 0.0\n    return {\"a1\": a1, \"a2\": a2, \"loo\": loo or 0.0, \"meta\": meta}\n\n\ndef solve_one(trains, test):\n    \"\"\"Core: one test input -> (attempt_1, attempt_2, meta).\n\n    M1 (2026-10-16): when a staged LLM model is present (Kaggle input),\n    proposals go through llm_harness.ensemble_decide \u2014 certified LLM\n    candidates compete with the symbolic solver; certified programs\n    outrank fluent guesses. No model (local/smoke): symbolic only.\n    \"\"\"\n    t0 = time.time()\n    if AG is None:\n        a1, a2, meta = _solve_legacy(trains, test)\n    elif LH is not None and LH.model_path():\n        try:\n            ens = LH.ensemble_decide(trains, test,\n                                     lambda tr, te: _sym_as_dict(tr, te))\n            a1, a2 = ens[\"a1\"], ens[\"a2\"]\n            meta = {\"dynamic\": \"llm+\" + str(ens.get(\"source\")),\n                    \"fallback\": a1 is None,\n                    \"trace\": [], \"n_candidates\": 0,\n                    \"llm_meta\": ens.get(\"llm_meta\"),\n                    \"sym_loo\": ens.get(\"sym_loo\")}\n        except Exception:\n            d = _sym_as_dict(trains, test)\n            a1, a2, meta = d[\"a1\"], d[\"a2\"], d[\"meta\"]\n    else:\n        try:\n            a1, a2, meta = AG.decide(trains, test)\n        except Exception:\n            a1, a2, meta = _solve_legacy(trains, test)\n    ms = (time.time() - t0) * 1000.0\n    meta[\"ms\"] = ms\n    dyn = str(meta.get(\"dynamic\", \"?\"))\n    SOURCE_TALLY[dyn] = SOURCE_TALLY.get(dyn, 0) + 1\n    return a1, a2, meta\n\n\ndef solve_task(task):\n    \"\"\"Solve one ARC task. Returns dict with attempt_1/attempt_2.\"\"\"\n    trains = [(p[\"input\"], p[\"output\"]) for p in task.get(\"train\", [])]\n    tests = task.get(\"test\", [])\n    if not trains or not tests:\n        return {\"attempt_1\": None, \"attempt_2\": None, \"dynamic\": \"empty\",\n                \"fallback\": True, \"ms\": 0.0}\n    a1, a2, meta = solve_one(trains, tests[0][\"input\"])\n    return {\"attempt_1\": a1, \"attempt_2\": a2, \"dynamic\": meta[\"dynamic\"],\n            \"fallback\": meta[\"fallback\"], \"ms\": meta[\"ms\"],\n            \"trace\": meta.get(\"trace\", []),\n            \"n_candidates\": meta.get(\"n_candidates\", 0)}\n\n\ndef solve_all_tests(task):\n    \"\"\"All test inputs -> list of {attempt_1, attempt_2} (Kaggle order).\"\"\"\n    trains = [(p[\"input\"], p[\"output\"]) for p in task.get(\"train\", [])]\n    out = []\n    for t in task.get(\"test\", []):\n        a1, a2, _meta = solve_one(trains, t[\"input\"])\n        out.append({\"attempt_1\": a1 if a1 is not None else t[\"input\"],\n                    \"attempt_2\": a2 if a2 is not None else\n                    (a1 if a1 is not None else t[\"input\"])})\n    return out\n", "rsi_bridge.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"rsi_bridge.py \u2014 Bridge: RSI-generated strategies \u2192 agent.py.\n\nCarga estrategias generadas por el RSI loop (pattern_memory) y las\npresenta como candidatos al agent. Stdlib-only, determin\u00edstico.\n\nUsage:\n    from rsi_bridge import rsi_propose\n    candidates = rsi_propose(trains, test)\n    # Returns list of {grid, key, dynamic, fn_name, loo_frac, prior}\n\"\"\"\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nHERE = Path(__file__).parent\nif str(HERE) not in sys.path:\n    sys.path.insert(0, str(HERE))\n\nimport numpy as np\nimport recognizer as R\n\n# State directory for RSI patterns\n_STATE_DIR = HERE.parent / \"arc-agi3\" / \"state\"\n_MEMORY_FILE = _STATE_DIR / \"pattern_memory.json\"\n\n\ndef _load_rsi_patterns():\n    \"\"\"Load RSI-learned patterns from pattern_memory.json.\"\"\"\n    if not _MEMORY_FILE.exists():\n        return {}\n    try:\n        with open(_MEMORY_FILE) as f:\n            data = json.load(f)\n        # Only return active, llm-generated patterns\n        return {\n            k: v for k, v in data.get(\"patterns\", {}).items()\n            if v.get(\"status\") == \"active\"\n            and v.get(\"source\") in (\"llm_proposed\", \"rsi_generated\")\n        }\n    except Exception:\n        return {}\n\n\ndef _try_rsi_strategy(code_str, trains, test):\n    \"\"\"Execute an RSI-generated strategy in a restricted sandbox.\"\"\"\n    safe_builtins = {\n        \"abs\": abs, \"all\": all, \"any\": any, \"bool\": bool,\n        \"dict\": dict, \"enumerate\": enumerate, \"filter\": filter,\n        \"float\": float, \"int\": int, \"isinstance\": isinstance,\n        \"len\": len, \"list\": list, \"map\": map, \"max\": max, \"min\": min,\n        \"print\": print, \"range\": range, \"round\": round, \"set\": set,\n        \"sorted\": sorted, \"str\": str, \"sum\": sum, \"tuple\": tuple,\n        \"type\": type, \"zip\": zip,\n    }\n\n    def _safe_import(name, *args, **kwargs):\n        mod = name.split(\".\")[0]\n        if mod not in (\"numpy\", \"math\", \"collections\", \"itertools\"):\n            raise ImportError(f\"Import not allowed: {name}\")\n        import importlib\n        return importlib.import_module(name)\n\n    safe_builtins[\"__import__\"] = _safe_import\n\n    namespace = {\"__builtins__\": safe_builtins, \"np\": np, \"numpy\": np}\n    try:\n        compiled = compile(code_str, \"<rsi_generated>\", \"exec\")\n        exec(compiled, namespace)\n    except Exception:\n        return None\n\n    solve_fn = namespace.get(\"solve\")\n    if not callable(solve_fn):\n        return None\n\n    try:\n        result = solve_fn(trains, test)\n        if result is not None and isinstance(result, np.ndarray):\n            return result\n    except Exception:\n        pass\n    return None\n\n\ndef rsi_propose(trains, test, max_candidates=3):\n    \"\"\"Generate candidates from RSI-learned strategies.\n\n    Returns list of dicts compatible with agent.py's propose() output:\n        {grid, key, dynamic, fn_name, loo_frac, prior}\n    \"\"\"\n    patterns = _load_rsi_patterns()\n    if not patterns:\n        return []\n\n    candidates = []\n    for name, info in list(patterns.items())[:max_candidates]:\n        code = info.get(\"code\", \"\")\n        if not code:\n            continue\n\n        grid = _try_rsi_strategy(code, trains, test)\n        if grid is None:\n            continue\n\n        try:\n            key = tuple(tuple(r) for r in grid)\n        except Exception:\n            continue\n\n        # Quick LOO check\n        try:\n            fn = lambda tr, t: _try_rsi_strategy(code, tr, t)\n            loo_f = R.verify_loo_fraction(fn, trains)\n        except Exception:\n            loo_f = 0.0\n\n        candidates.append({\n            \"grid\": grid,\n            \"key\": key,\n            \"dynamic\": f\"rsi:{name}\",\n            \"fn_name\": f\"rsi_{name}\",\n            \"loo_frac\": loo_f,\n            \"prior\": 0.5,  # RSI strategies start with neutral prior\n        })\n\n    return candidates\n\n\ndef has_rsi_strategies():\n    \"\"\"Check if any RSI strategies are available.\"\"\"\n    return bool(_load_rsi_patterns())\n", "composer.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"composer \u2014 B-phase coverage motor: program enumeration + LOO verify.\n\nAudit 2026-10-16: the 56 fixed dynamics cover 9% of training; the other\n91% falls to unverified fallback. The propose\u2192LOO-verify\u2192critic frame\nwas always right \u2014 this module supplies the missing generator: atomic\ngrid operations (geometry, palette, structure, census) composed as\npairs, every candidate certified by the SAME verify_loo_fraction the\ndynamics use. Deterministic (R10): fixed op order, learned params from\ntrains only, no randomness.\n\nIntegration: agent.propose calls composer.candidates(trains, test) and\nmerges results as producers (order 500). Cost: ~120-180 fns \u00d7 LOO,\nbounded by INVARIANT prefilters (shape ratio, palette size).\n\"\"\"\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom typing import Callable, List, Optional, Tuple\n\nGrid = List[List[int]]\nTrains = List[Tuple[Grid, Grid]]\n\n# \u2500\u2500\u2500 grid primitives (lists, stdlib only) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef shape(g):\n    return (len(g), len(g[0]) if g else 0)\n\n\ndef rot90(g):\n    R, C = shape(g)\n    return [[g[R - 1 - c][r] for c in range(R)] for r in range(C)] if R else g\n\n\ndef flip_h(g):\n    return [row[::-1] for row in g]\n\n\ndef flip_v(g):\n    return g[::-1]\n\n\ndef transpose(g):\n    if not g or not g[0]:\n        return g\n    return [list(row) for row in zip(*g)]\n\n\ndef anti_transpose(g):\n    return transpose(flip_h(g))\n\n\ndef colors(g):\n    return {c for row in g for c in row}\n\n\ndef bg_of(trains):\n    \"\"\"Most common corner color across train inputs (background).\"\"\"\n    cnt = Counter()\n    for inp, _out in trains:\n        for r in (0, len(inp) - 1):\n            for c in (0, len(inp[r]) - 1):\n                cnt[inp[r][c]] += 1\n    return cnt.most_common(1)[0][0] if cnt else 0\n\n\ndef content_bbox(g, bg):\n    rows = [r for r in range(len(g))\n            if any(v != bg for v in g[r])]\n    if not rows:\n        return None\n    cols = [c for c in range(len(g[0]))\n            if any(g[r][c] != bg for r in rows)]\n    if not cols:\n        return None\n    r0, r1, c0, c1 = rows[0], rows[-1], cols[0], cols[-1]\n    return [row[c0:c1 + 1] for row in g[r0:r1 + 1]]\n\n\ndef components_of(g, bg=None):\n    \"\"\"4-connected non-bg components: list of set((r, c)).\"\"\"\n    if bg is None:\n        bg = bg_of([(g, g)])\n    R, C = shape(g)\n    seen = set()\n    out = []\n    for r0 in range(R):\n        for c0 in range(C):\n            if g[r0][c0] == bg or (r0, c0) in seen:\n                continue\n            stack, comp = [(r0, c0)], set()\n            seen.add((r0, c0))\n            while stack:\n                r, c = stack.pop()\n                comp.add((r, c))\n                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                    nr, nc = r + dr, c + dc\n                    if (0 <= nr < R and 0 <= nc < C\n                            and (nr, nc) not in seen\n                            and g[nr][nc] != bg):\n                        seen.add((nr, nc))\n                        stack.append((nr, nc))\n            out.append(comp)\n    return out\n\n\n# \u2500\u2500\u2500 atomic ops: factory(trains) -> fn(trains, grid) -> grid|None \u2500\u2500\u2500\n\ndef _pure(name, f):\n    return (name, lambda trains, _f=f: (lambda _t, g, __f=_f: __f(g)))\n\n\nATOMIC = [\n    _pure(\"rot90\", rot90),\n    _pure(\"rot180\", lambda g: rot90(rot90(g))),\n    _pure(\"rot270\", lambda g: rot90(rot90(rot90(g)))),\n    _pure(\"flip_h\", flip_h),\n    _pure(\"flip_v\", flip_v),\n    _pure(\"transpose\", transpose),\n    _pure(\"anti_transpose\", anti_transpose),\n]\n\nLEARNABLE: List[Tuple[str, Callable[[Trains], Optional[Callable]]]] = []\n\n\ndef _fact_crop(trains):\n    \"\"\"Crop to train-input content bbox (contract: fn(trains, grid)).\"\"\"\n    bg = bg_of(trains)\n    for inp, _out in trains:\n        if content_bbox(inp, bg) is None:\n            return None\n    return lambda _t, inp: (content_bbox(inp, bg) or inp)\n\n\ndef _fact_tile_learn(trains):\n    \"\"\"Tile input to match train output shape (constant integer factor).\"\"\"\n    factors = set()\n    for inp, out in trains:\n        ir, ic = shape(inp)\n        orr, oc = shape(out)\n        if ir and ic and orr % ir == 0 and oc % ic == 0:\n            factors.add((orr // ir, oc // ic))\n        else:\n            return None\n    if len(factors) != 1:\n        return None\n    fr, fc = factors.pop()\n\n    def fn(_t, inp):\n        R, C = shape(inp)\n        return [[inp[r % R][c % C] for c in range(fc * C)]\n                for r in range(fr * R)]\n    return fn\n\n\ndef _fact_invert_mono(trains):\n    \"\"\"Invert non-bg colors into a consecutive low palette (train-learned).\"\"\"\n    bg = bg_of(trains)\n    seen = set()\n    for inp, out in trains:\n        seen |= (colors(inp) | colors(out)) - {bg}\n    if not seen or len(seen) > 9:\n        return None\n    order = sorted(seen)\n    cmap = {old: i + 1 for i, old in enumerate(order)}\n\n    def fn(_t, inp):\n        return [[bg if v == bg else cmap.get(v, v) for v in row]\n                for row in inp]\n    return fn\n\n\ndef _fact_count_to_color(trains):\n    \"\"\"Per-cell color := count of that cell's color in its row (census).\"\"\"\n\n    def fn(_t, inp):\n        out = []\n        for row in inp:\n            cnt = Counter(row)\n            out.append([cnt[v] % 10 for v in row])\n        return out\n    return fn\n\n\ndef _fact_obj_fill(trains):\n    \"\"\"Fill every component with its own modal color (object uniformity).\"\"\"\n    bg = bg_of(trains)\n\n    def fn(_t, inp):\n        g = [row[:] for row in inp]\n        for comp in components_of(inp, bg):\n            modal = Counter(inp[r][c] for r, c in comp).most_common(1)[0][0]\n            for r, c in comp:\n                g[r][c] = modal\n        return g\n    return fn\n\n\ndef _fact_clear_border(trains):\n    \"\"\"Border cells -> background (train-consistent bg).\"\"\"\n    bg = bg_of(trains)\n\n    def fn(_t, inp):\n        R, C = shape(inp)\n        g = [row[:] for row in inp]\n        for c in range(C):\n            g[0][c] = bg\n            g[R - 1][c] = bg\n        for r in range(R):\n            g[r][0] = bg\n            g[r][C - 1] = bg\n        return g\n    return fn\n\n\ndef _fact_color_map(trains):\n    \"\"\"Pointwise color mapping learned from train pairs (consistent only).\n\n    Covers the large 'recolor' family the strict dynamics detectors miss.\n    \"\"\"\n    cmap = {}\n    for inp, out in trains:\n        if shape(inp) != shape(out):\n            return None\n        for rin, rout in zip(inp, out):\n            for a, b in zip(rin, rout):\n                if a in cmap and cmap[a] != b:\n                    return None\n                cmap[a] = b\n    if not cmap or len(set(cmap.values())) == len(colors(\n            trains[0][1])) and all(k == v for k, v in cmap.items()):\n        return None  # identity or empty: no signal\n    return lambda _t, g: [[cmap.get(v, v) for v in row] for row in g]\n\n\ndef _fact_sym_h(trains):\n    \"\"\"Complete horizontal symmetry: left half mirrors onto right.\"\"\"\n\n    def fn(_t, g):\n        R, C = shape(g)\n        if C < 2:\n            return None\n        out = [row[:] for row in g]\n        for r in range(R):\n            for c in range((C + 1) // 2):\n                out[r][C - 1 - c] = g[r][c]\n        return out\n    return fn\n\n\ndef _fact_sym_v(trains):\n    \"\"\"Complete vertical symmetry: top half mirrors onto bottom.\"\"\"\n\n    def fn(_t, g):\n        R, C = shape(g)\n        if R < 2:\n            return None\n        out = [row[:] for row in g]\n        for r in range((R + 1) // 2):\n            out[R - 1 - r] = g[r][:]\n        return out\n    return fn\n\n\ndef _fact_quadrant_copy(trains):\n    \"\"\"Tile top-left quadrant across the whole grid.\"\"\"\n\n    def fn(_t, g):\n        R, C = shape(g)\n        hr, hc = R // 2, C // 2\n        if hr == 0 or hc == 0:\n            return None\n        q = [row[:hc] for row in g[:hr]]\n        return [[q[r % hr][c % hc] for c in range(C)] for r in range(R)]\n    return fn\n\n\ndef _fact_objcount_row(trains):\n    \"\"\"Per-cell color := number of objects intersecting that row (mod 10).\"\"\"\n    bg = bg_of(trains)\n\n    def fn(_t, g):\n        comps = components_of(g, bg)\n        cnt_rows = Counter()\n        for comp in comps:\n            for r in {rr for rr, _cc in comp}:\n                cnt_rows[r] += 1\n        return [[cnt_rows[r] % 10 for _c in g[r]] for r in range(len(g))]\n    return fn\n\n\nLEARNABLE = [\n    (\"crop_content\", _fact_crop),\n    (\"tile_learn\", _fact_tile_learn),\n    (\"invert_mono\", _fact_invert_mono),\n    (\"count_to_color\", _fact_count_to_color),\n    (\"obj_fill\", _fact_obj_fill),\n    (\"clear_border\", _fact_clear_border),\n    (\"color_map\", _fact_color_map),\n    (\"sym_h\", _fact_sym_h),\n    (\"sym_v\", _fact_sym_v),\n    (\"quadrant_copy\", _fact_quadrant_copy),\n    (\"objcount_row\", _fact_objcount_row),\n]\n\n\n# \u2500\u2500\u2500 composition + candidates \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef _shape_preserving(trains):\n    \"\"\"True when every train output matches its input shape exactly.\"\"\"\n    return all(shape(i) == shape(o) for i, o in trains)\n\n\ndef _palette_bound(trains):\n    \"\"\"Palette size of train outputs (census ops capped by this).\"\"\"\n    out_cols = set()\n    for _i, o in trains:\n        out_cols |= colors(o)\n    return len(out_cols)\n\n\ndef candidates(trains, test):\n    \"\"\"Enumerate composed programs, return verified-friendly list.\n\n    Each item: {name, fn, pred, loo} \u2014 pred already applied to test via\n    safe semantics (fn may return None). Caller runs verify_loo_fraction.\n    Prefilters: pure pairs always (cheap); learnables only when their\n    factory resolves; census ops only when train palette > 2 (else a\n    constant census cannot express the output).\n    \"\"\"\n    import recognizer as R  # local import: keep composer standalone-testable\n\n    built = []  # (name, fn)\n    for name, fac in ATOMIC:\n        fn = fac(trains)\n        if fn is None:\n            continue\n        built.append((name, fn))\n    for name, fac in LEARNABLE:\n        try:\n            fn = fac(trains)\n        except Exception:\n            fn = None\n        if fn is None:\n            continue\n        built.append((name, fn))\n\n    # census op is only meaningful when outputs have >2 colors\n    palette_out = _palette_bound(trains)\n\n    out = []\n    seen = set()\n\n    def emit(name, fn):\n        if name in seen:\n            return\n        pred = R.safe_apply(fn, trains, test)\n        if pred is None:\n            return\n        seen.add(name)\n        try:\n            loo = R.verify_loo_fraction(fn, trains)\n        except Exception:\n            loo = 0.0\n        out.append({\"name\": name, \"fn\": fn, \"pred\": pred, \"loo\": loo})\n\n    for name, fn in built:\n        emit(\"c:\" + name, fn)\n\n    # pairs: op2 \u2218 op1 (skip census-containing pairs unless palette ok)\n    census_names = {\"count_to_color\"}\n    for n1, f1 in built:\n        for n2, f2 in built:\n            if (n1 in census_names or n2 in census_names) and palette_out <= 2:\n                continue\n\n            def composed(trains, g, _f1=f1, _f2=f2):\n                g1 = _f1(trains, g)\n                if g1 is None:\n                    return None\n                return _f2(trains, g1)\n            emit(f\"c:{n2}\u2218{n1}\", composed)\n    return out\n", "llm_serialize.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"llm_serialize \u2014 grid <-> text for LLM prompting (M1 harness).\n\nFormat (community grid-serialization style): every grid is rows of\ndigit characters (colors 0-9), one row per line. The prompt shows all\ntrain pairs as Input/Output blocks and asks for the test output grid\nONLY. Deterministic (R10): pure string transforms, no randomness.\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import List, Optional, Tuple\n\nGrid = List[List[int]]\nTrains = List[Tuple[Grid, Grid]]\n\n\ndef grid_to_text(g: Grid) -> str:\n    return \"\\n\".join(\"\".join(str(int(v)) for v in row) for row in g)\n\n\ndef grid_from_text(text: str) -> Optional[Grid]:\n    \"\"\"Parse the FIRST valid grid block from LLM text (robust).\"\"\"\n    rows: List[List[int]] = []\n    for raw in text.splitlines():\n        line = raw.strip().strip(\"`\").strip()\n        if not line:\n            if rows:\n                break\n            continue\n        # accept only pure digit rows (optionally space/comma separated)\n        cleaned = line.replace(\" \", \"\").replace(\",\", \"\")\n        if cleaned.isdigit() and len(cleaned) <= 64:\n            rows.append([int(ch) for ch in cleaned])\n            continue\n        if rows:\n            break\n    if not rows:\n        return None\n    width = len(rows[0])\n    if any(len(r) != width for r in rows):\n        return None\n    return rows\n\n\ndef build_prompt(trains: Trains, test: Grid) -> str:\n    parts = [\"Solve this abstract reasoning task. The test OUTPUT grid \"\n             \"follows the same rules as the train examples.\",\n             \"Output ONLY the test output grid: one row per line, digits \"\n             \"0-9, no explanation, no markdown.\"]\n    for i, (inp, out) in enumerate(trains, 1):\n        parts.append(f\"Train example {i} input:\\n{grid_to_text(inp)}\")\n        parts.append(f\"Train example {i} output:\\n{grid_to_text(out)}\")\n    parts.append(f\"Test input:\\n{grid_to_text(test)}\")\n    parts.append(\"Test output:\")\n    return \"\\n\\n\".join(parts)\n\n\ndef sanity_shape(pred: Grid, trains: Trains, max_factor: int = 4) -> bool:\n    \"\"\"Reject absurd shapes: at most max_factor x the largest train out.\"\"\"\n    if not pred:\n        return False\n    pr, pc = len(pred), len(pred[0])\n    if pr < 1 or pc < 1 or pr > 64 or pc > 64:\n        return False\n    max_r = max(len(o) for _i, o in trains)\n    max_c = max(len(o[0]) for _i, o in trains)\n    return pr <= max_r * max_factor and pc <= max_c * max_factor\n\n\nSYSTEM = (\"You are an expert at the ARC abstract reasoning benchmark. \"\n          \"You output ONLY grids as rows of digit characters.\")\n", "llm_harness.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"llm_harness \u2014 M1: LLM proposals certified before they compete.\n\nS5 hypothesis (2026-10-16): a local LLM (pre-trained models allowed,\nno internet during eval) proposes candidates; OUR verifier filters\nhallucinations. Two certify layers, deterministic where possible:\n\nV1 self-consistency: K samples -> majority grid + unanimity ratio\n   (greedy/seeded sampling; the same trick as consensus decoding).\nV2 LOO-understanding: leave-one-train-out prompts \u2014 the model must\n   reproduce held-out train outputs before its test prediction is\n   trusted (budget-gated: only when V1 is not unanimous).\n\nEnsemble with the symbolic solver: a LOO-verified symbolic PROGRAM\nalways outranks an LLM guess (certified > fluent); the LLM only takes\nattempt_1 when the symbolic side is unverified. attempt_2 stays the\nbest distinct-family candidate from either side.\n\nRuntime budget is enforced by the caller (kernel watchdog); this\nmodule only counts calls. Model loading is lazy + path-gated so the\nlocal/symbolic pipeline keeps working with no torch installed.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom collections import Counter\nfrom typing import Callable, List, Optional, Tuple\n\nimport llm_serialize as S\n\nGrid = List[List[int]]\nTrains = List[Tuple[Grid, Grid]]\n\nK_SAMPLES = int(os.environ.get(\"LLM_K_SAMPLES\", \"4\"))\nLOO_MAX = int(os.environ.get(\"LLM_LOO_MAX\", \"2\"))\nUNANIMOUS = float(os.environ.get(\"LLM_UNANIMOUS\", \"0.75\"))\n\n_MODEL = None\n_TOKENIZER = None\n_DEADLINE = None\n\n\ndef set_deadline(epoch_seconds: float) -> None:\n    \"\"\"Hard wall for LLM calls (kernel watchdog lives in the notebook).\"\"\"\n    global _DEADLINE\n    _DEADLINE = float(epoch_seconds)\n\n\ndef _past_deadline() -> bool:\n    if _DEADLINE is None:\n        return False\n    import time\n    return time.time() >= _DEADLINE\n\n\ndef model_path() -> Optional[str]:\n    \"\"\"Weight location: env override first, then scan /kaggle/input.\n\n    Datasets mount flat (slug) or nested (owner/slug) depending on\n    Kaggle's mounting \u2014 walk two levels looking for HF layout\n    (config.json + safetensors), never deeper (the eval set is huge).\n    \"\"\"\n    env = os.environ.get(\"LLM_MODEL_PATH\", \"\")\n    if env and os.path.isdir(env):\n        return env\n    root = \"/kaggle/input\"\n    if not os.path.isdir(root):\n        return None\n    try:\n        for name in sorted(os.listdir(root)):\n            d1 = os.path.join(root, name)\n            cands = [d1]\n            try:\n                cands += [os.path.join(d1, s) for s in sorted(os.listdir(d1))\n                          if os.path.isdir(os.path.join(d1, s))]\n            except OSError:\n                pass\n            for cand in cands:\n                try:\n                    files = set(os.listdir(cand))\n                except OSError:\n                    continue\n                if \"model.safetensors.index.json\" in files or (\n                        \"config.json\" in files and\n                        any(f.endswith(\".safetensors\") for f in files)):\n                    return cand\n    except OSError:\n        return None\n    return None\n\n\ndef _load():\n    global _MODEL, _TOKENIZER\n    if _MODEL is not None:\n        return True\n    path = model_path()\n    if not path:\n        return False\n    try:\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)\n        model = AutoModelForCausalLM.from_pretrained(\n            path, torch_dtype=torch.bfloat16, device_map=\"cuda\",\n            trust_remote_code=True)\n        model.eval()\n        _TOKENIZER, _MODEL = tok, model\n        return True\n    except Exception:\n        _MODEL = None\n        return False\n\n\ndef _generate(prompt: str, max_new_tokens: int = 512,\n              temperature: float = 0.7) -> str:\n    import torch\n    tok, model = _TOKENIZER, _MODEL\n    messages = [{\"role\": \"system\", \"content\": S.SYSTEM},\n                {\"role\": \"user\", \"content\": prompt}]\n    text = tok.apply_chat_template(messages, tokenize=False,\n                                   add_generation_prompt=True)\n    inputs = tok(text, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        out = model.generate(\n            **inputs, max_new_tokens=max_new_tokens,\n            do_sample=temperature > 0, temperature=max(temperature, 1e-5),\n            top_p=0.95, pad_token_id=tok.eos_token_id)\n    return tok.decode(out[0][inputs[\"input_ids\"].shape[1]:],\n                      skip_special_tokens=True)\n\n\ndef propose(trains: Trains, test: Grid) -> Optional[dict]:\n    \"\"\"K sampled candidates -> V1 majority certify. Returns dict or None.\n\n    {grid, key, agreement, n_unique, loo_llm} \u2014 loo_llm filled by v2\n    only when the caller asks (certify_understanding).\n    \"\"\"\n    if not _load() or _past_deadline():\n        return None\n    prompt = S.build_prompt(trains, test)\n    grids = []\n    for _ in range(K_SAMPLES):\n        raw = _generate(prompt, temperature=0.7)\n        g = S.grid_from_text(raw)\n        if g is not None and S.sanity_shape(g, trains):\n            grids.append(g)\n    if not grids:\n        return None\n    keys = Counter(S.grid_to_text(g) for g in grids)\n    best_text, votes = keys.most_common(1)[0]\n    best = next(g for g in grids if S.grid_to_text(g) == best_text)\n    return {\"grid\": best, \"key\": tuple(tuple(r) for r in best),\n            \"agreement\": votes / K_SAMPLES,\n            \"n_unique\": len(keys), \"loo_llm\": None}\n\n\ndef certify_understanding(trains: Trains, test: Grid,\n                          candidate: Grid) -> float:\n    \"\"\"V2 LOO: how many left-out train outputs the model reproduces.\n\n    A fluent hallucinator fails V2; a model that 'got' the rule passes.\n    Fraction in [0,1]; caller treats >= 0.5 as understanding.\n    \"\"\"\n    if not _load() or _past_deadline() or len(trains) < 2:\n        return 0.0\n    budget = min(LOO_MAX, len(trains))\n    ok = 0\n    for i in range(budget):\n        others = trains[:i] + trains[i + 1:]\n        held_in, held_out = trains[i]\n        prompt = S.build_prompt(others, held_in)\n        raw = _generate(prompt, max_new_tokens=512, temperature=0.0)\n        g = S.grid_from_text(raw)\n        if g is not None and g == held_out:\n            ok += 1\n    return ok / budget\n\n\ndef ensemble_decide(trains: Trains, test: Grid,\n                    symbolic: Callable[[Trains, Grid], dict],\n                    ) -> dict:\n    \"\"\"Combine certified LLM with the symbolic propose().\n\n    symbolic(trains, test) -> {\"a1\": grid|None, \"a2\": grid|None, \"loo\": f}\n    Returns {\"a1\", \"a2\", \"source\", \"llm_meta\", \"sym_loo\"}.\n    \"\"\"\n    sym = symbolic(trains, test)\n    llm = propose(trains, test)\n    sym_verified = (sym.get(\"loo\") or 0.0) >= 0.5\n\n    if llm is not None and llm[\"agreement\"] >= UNANIMOUS and not sym_verified:\n        loo_llm = certify_understanding(trains, test, llm[\"grid\"])\n        llm[\"loo_llm\"] = loo_llm\n        if loo_llm >= 0.5:\n            a1 = llm[\"grid\"]\n            a2 = sym.get(\"a1\") if (sym.get(\"a1\") is not None\n                                   and tuple(map(tuple, sym[\"a1\"])) !=\n                                   llm[\"key\"]) else sym.get(\"a2\")\n            return {\"a1\": a1, \"a2\": a2, \"source\": \"llm-certified\",\n                    \"llm_meta\": {k: llm[k] for k in\n                                 (\"agreement\", \"n_unique\", \"loo_llm\")},\n                    \"sym_loo\": sym.get(\"loo\")}\n    # symbolic verified (or no usable LLM): symbolic leads\n    a1 = sym.get(\"a1\")\n    a2 = sym.get(\"a2\")\n    if llm is not None and llm[\"key\"] != tuple(map(tuple, a1 or [])):\n        if a2 is None:\n            a2 = llm[\"grid\"]\n    return {\"a1\": a1, \"a2\": a2, \"source\": \"symbolic\",\n            \"llm_meta\": ({k: llm[k] for k in (\"agreement\", \"n_unique\")}\n                         if llm else None),\n            \"sym_loo\": sym.get(\"loo\")}\n"}

for name, src in MODULES.items():
    with open(name, "w") as fp:
        fp.write(src)
print("modules written:", sorted(MODULES))

sys.path.insert(0, os.getcwd())
from solve import solve_all_tests

CANDIDATES = ["/kaggle/input"]
tasks = {}
for root in CANDIDATES:
    for dirpath, _dn, files in os.walk(root):
        if dirpath.count(os.sep) - root.count(os.sep) > 3:
            continue
        for f in files:
            if not f.endswith(".json"):
                continue
            p = os.path.join(dirpath, f)
            try:
                with open(p) as fp:
                    d = json.load(fp)
            except Exception as e:
                print("skip unreadable", p, e)
                continue
            if isinstance(d, dict) and "train" in d and "test" in d:
                tasks[os.path.splitext(f)[0]] = d
            elif isinstance(d, dict):
                for tid, t in d.items():
                    if isinstance(t, dict) and "train" in t and "test" in t:
                        tasks[tid] = t
print("tasks found:", len(tasks))
assert tasks, "no ARC tasks discovered under /kaggle/input"

def valid_grid(g):
    return (isinstance(g, list) and g and all(
        isinstance(r, list) and r and all(
            isinstance(v, int) and 0 <= v <= 9 for v in r) for r in g))

# M1 (2026-10-16): LLM deadline guard + resumable partial. The LLM
# path serves puzzles until the deadline; the rest fall back to the
# symbolic engine (always fast, always completes the schema).
t0 = time.time()
DEADLINE_S = float(os.environ.get("LLM_BUDGET_SECONDS", str(9.5 * 3600)))
try:
    import llm_harness as LH
    LH.set_deadline(t0 + DEADLINE_S)
    print(f"llm budget: {DEADLINE_S/3600:.1f}h, model:",
          "staged" if LH.model_path() else "absent (symbolic only)")
except Exception as e:
    print("llm harness unavailable:", e)

PART = "/kaggle/working/submission.partial.json"
sub = {}
if os.path.exists(PART):
    try:
        sub = json.load(open(PART))
        print("resumed from partial:", len(sub), "tasks")
    except Exception:
        sub = {}

SMOKE_N = int(os.environ.get("SMOKE_N", "0"))  # 12 = GPU smoke; 0 = full
for tid in sorted(tasks)[:SMOKE_N or None]:
    if tid in sub:
        continue
    try:
        preds = solve_all_tests(tasks[tid])
        assert len(preds) == len(tasks[tid]["test"])
        for pr in preds:
            assert valid_grid(pr["attempt_1"]) and valid_grid(pr["attempt_2"])
        sub[tid] = preds
    except Exception as e:
        print("TASK FAIL", tid, repr(e)[:200], "-> input-copy fallback")
        sub[tid] = [{"attempt_1": t["input"], "attempt_2": t["input"]}
                    for t in tasks[tid]["test"]]
    if len(sub) % 20 == 0:
        with open(PART, "w") as fp:
            json.dump(sub, fp)
        print(f"checkpoint {len(sub)}/{len(tasks)} "
              f"({time.time()-t0:.0f}s)")
out = "/kaggle/working/submission.json"
try:
    os.makedirs(os.path.dirname(out), exist_ok=True)
except Exception:
    out = "submission.json"
with open(out, "w") as fp:
    json.dump(sub, fp)
print(f"wrote {out}: {len(sub)} tasks in {time.time()-t0:.1f}s")
try:
    import solve as _S
    print("source tally:", getattr(_S, "SOURCE_TALLY", {}))
except Exception:
    pass
